# Reproducible bilevel optimization of activated sludge operation

This notebook is the sole executable analysis artifact for the article. It validates and runs the
unclipped ASM2d-TSN steady-state model, generates the deterministic mechanistic training data, fits
and freezes the published coupled ICSOR regression head, deploys it through the present study's
unique scaled-$L_2$ physical-projection QP, solves the nominal and robustness optimization cases,
and validates both prediction fidelity and decision quality against independent ASM2d-TSN searches.

`SURROGATE_OPT_PROFILE=full` is the only article-eligible profile. The `smoke` profile exercises the
complete pipeline at reduced scale and is always marked non-final. Every numerical result, timing,
solver diagnostic, coefficient, prediction, optimization evaluation, and plotted datum is written
beneath an immutable checksummed run directory.

In [ ]:
from __future__ import annotations

import contextlib
import copy
import hashlib
import importlib.metadata
import io
import json
import math
import os
import pickle
import platform
import re
import socket
import sys
import tempfile
import time
import warnings
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import osqp
import pandas as pd
import psutil
import scipy
from IPython.display import display
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from scipy import sparse as sp
from scipy.integrate import solve_ivp
from scipy.linalg import null_space
from scipy.optimize import direct, least_squares
from scipy.stats.qmc import LatinHypercube, scale as qmc_scale
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits
from tqdm import tqdm

warnings.filterwarnings("once", category=RuntimeWarning)

ROOT = Path.cwd().resolve()
if not (ROOT / "config").is_dir() or not (ROOT / "data").is_dir():
    raise RuntimeError("Run this notebook from the surrogate-optimization-arch project root.")


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"Expected a JSON object in {path}.")
    return value


def deep_merge(base: Mapping[str, Any], override: Mapping[str, Any]) -> dict[str, Any]:
    merged = copy.deepcopy(dict(base))
    for key, value in override.items():
        if isinstance(value, Mapping) and isinstance(merged.get(key), Mapping):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = copy.deepcopy(value)
    return merged


CONFIG = read_json(ROOT / "config" / "params.json")
PATHS = read_json(ROOT / "config" / "paths.json")
EXPECTED_TOP_LEVEL = {
    "run", "simulation", "icsor", "deployment", "optimization",
    "robustness", "validation", "timing", "reporting",
}
if set(CONFIG) != EXPECTED_TOP_LEVEL:
    raise ValueError(f"Configuration keys must be exactly {sorted(EXPECTED_TOP_LEVEL)}.")
if set(PATHS) != {"workbook", "results_root", "run_root_pattern"}:
    raise ValueError("The path configuration has an unexpected schema.")

profiles = CONFIG["run"]["profiles"]
profile_env = str(CONFIG["run"]["profile_env_var"])
requested_profile = os.environ.get(profile_env, str(CONFIG["run"]["default_profile"])).strip().lower()
if requested_profile not in {"full", "smoke"} or requested_profile not in profiles:
    raise ValueError(f"{profile_env} must select either 'full' or 'smoke'.")

profile_override = copy.deepcopy(dict(profiles[requested_profile]))
profile_run_id = profile_override.pop("run_id", None)
ACTIVE = deep_merge(CONFIG, profile_override)
if profile_run_id is not None:
    ACTIVE["run"]["run_id"] = profile_run_id
ACTIVE["run"]["profile"] = requested_profile
dataset_source_env = str(ACTIVE["run"]["dataset_source_run_env_var"])
dataset_source_run_id = os.environ.get(dataset_source_env, "").strip()
if dataset_source_run_id and not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", dataset_source_run_id):
    raise ValueError(f"{dataset_source_env} contains an invalid run ID.")
ACTIVE["run"]["dataset_source_run_id"] = dataset_source_run_id or None

if ACTIVE["run"]["resume_policy"] != "matching_contract_only":
    raise ValueError("Only contract-matched resume is supported.")
if int(ACTIVE["run"]["threads"]) != int(ACTIVE["simulation"]["sampling"]["parallel_workers"]):
    raise ValueError("run.threads must match simulation.sampling.parallel_workers.")
if int(ACTIVE["run"]["solver_chains"]) != int(ACTIVE["simulation"]["sampling"]["n_chains"]):
    raise ValueError("run.solver_chains must match simulation.sampling.n_chains.")

state_names = list(ACTIVE["simulation"]["workbook"]["state_columns"])
if len(state_names) != 20 or set(ACTIVE["simulation"]["influent_state_ranges"]) != set(state_names):
    raise ValueError("The simulation must define the complete ordered 20-component ASM state.")
if len(ACTIVE["simulation"]["workbook"]["processes"]) != 28:
    raise ValueError("The simulation must define all 28 ASM2d-TSN processes.")
if len(ACTIVE["simulation"]["workbook"]["parameters"]) != 81:
    raise ValueError("The workbook contract must define all 81 parameters.")

full_contract = CONFIG
if (
    int(full_contract["simulation"]["sampling"]["id_samples"]) != 10_000
    or int(full_contract["simulation"]["sampling"]["seed"]) != 42
    or int(full_contract["simulation"]["sampling"]["max_sample_attempts"]) != 25
    or float(full_contract["icsor"]["test_fraction"]) != 0.2
    or int(full_contract["robustness"]["n_cases"]) != 100
    or int(full_contract["robustness"]["seed"]) != 314159
):
    raise ValueError("The locked full-study sampling and validation contract has changed.")

# Fail closed for configuration branches that this article does not implement silently.
for variable in ("HRT", "Aeration"):
    training_bounds = np.asarray(ACTIVE["simulation"]["operational_ranges"][variable], dtype=float)
    optimization_bounds = np.asarray(ACTIVE["optimization"]["bounds"][variable], dtype=float)
    if training_bounds.shape != (2,) or not np.array_equal(training_bounds, optimization_bounds):
        raise ValueError(f"Optimization {variable} bounds must exactly match the trained operating range.")
if ACTIVE["optimization"]["target_map"] != "I_comp":
    raise ValueError("This article requires optimization.target_map='I_comp'.")
if list(ACTIVE["optimization"]["additional_discharge_constraints"]):
    raise ValueError("Additional discharge constraints are not part of the declared case study.")
if not bool(ACTIVE["validation"]["mechanistic_reference_for_every_case"]):
    raise ValueError("Every nominal and robustness case requires a mechanistic reference search.")
if not bool(ACTIVE["validation"]["fixed_solver_initialization"]):
    raise ValueError("Mechanistic validation requires fixed solver initialization.")
if int(ACTIVE["robustness"]["scenario_workers"]) != 1:
    raise ValueError("The deterministic article workflow currently requires scenario_workers=1.")
if list(ACTIVE["reporting"]["quantiles"]) != [0.25, 0.5, 0.75, 0.95]:
    raise ValueError("Reporting quantiles must match the declared article summary contract.")
if set(ACTIVE["reporting"]["figure_formats"]) != {"png", "pdf"}:
    raise ValueError("The article workflow requires both PNG and PDF figures.")


def safe_relative_path(raw_value: str, *, label: str) -> Path:
    raw = Path(str(raw_value))
    if raw.is_absolute() or ".." in raw.parts:
        raise ValueError(f"Configured {label} must be a safe relative path.")
    resolved = (ROOT / raw).resolve()
    if resolved != ROOT and ROOT not in resolved.parents:
        raise ValueError(f"Configured {label} escapes the project root.")
    return resolved


WORKBOOK_PATH = safe_relative_path(PATHS["workbook"], label="workbook")
RESULTS_ROOT = safe_relative_path(PATHS["results_root"], label="results_root")
if PATHS["run_root_pattern"] != "results/{run_id}":
    raise ValueError("The run-root pattern must remain results/{run_id}.")

run_id_env = str(ACTIVE["run"]["run_id_env_var"])
RUN_ID = os.environ.get(run_id_env, str(ACTIVE["run"]["run_id"])).strip()
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", RUN_ID):
    raise ValueError("run_id may contain only letters, numbers, dot, underscore, and hyphen.")
RUN_ROOT = (RESULTS_ROOT / RUN_ID).resolve()
if RUN_ROOT != RESULTS_ROOT and RESULTS_ROOT not in RUN_ROOT.parents:
    raise ValueError("The run root must remain below results/.")
DATASET_SOURCE_RUN_ID = ACTIVE["run"].get("dataset_source_run_id")
if DATASET_SOURCE_RUN_ID == RUN_ID:
    raise ValueError("The dataset source run must differ from the active run ID.")
DATASET_SOURCE_RUN_ROOT = (
    (RESULTS_ROOT / str(DATASET_SOURCE_RUN_ID)).resolve()
    if DATASET_SOURCE_RUN_ID is not None else None
)
if DATASET_SOURCE_RUN_ROOT is not None and RESULTS_ROOT not in DATASET_SOURCE_RUN_ROOT.parents:
    raise ValueError("The dataset source run must remain below results/.")
PREEXISTING_RUN_ENTRIES = sorted(item.name for item in RUN_ROOT.iterdir()) if RUN_ROOT.exists() else []
RUN_ROOT.mkdir(parents=True, exist_ok=True)

DIRS = {
    name: RUN_ROOT / name
    for name in (
        "inputs", "datasets", "matrices", "splits", "models", "predictions",
        "metrics", "timing", "optimization", "validation", "tables", "figures", "logs",
    )
}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def deterministic_array_fingerprint(
    arrays: Mapping[str, np.ndarray],
    *,
    metadata: Mapping[str, Any] | None = None,
) -> str:
    digest = hashlib.sha256()
    for name in sorted(arrays):
        value = np.ascontiguousarray(np.asarray(arrays[name]))
        digest.update(name.encode("utf-8") + b"\0")
        digest.update(value.dtype.str.encode("ascii") + b"\0")
        digest.update(json.dumps(value.shape).encode("ascii") + b"\0")
        digest.update(value.tobytes(order="C"))
    digest.update(sha256_json(metadata or {}).encode("ascii"))
    return digest.hexdigest()


def replace_with_retry(source: Path | str, destination: Path | str, *, attempts: int = 8) -> None:
    """Atomically replace a file, tolerating brief Windows indexer/antivirus locks."""
    source_path = Path(source)
    destination_path = Path(destination)
    for attempt in range(attempts):
        try:
            os.replace(source_path, destination_path)
            return
        except PermissionError:
            if attempt + 1 == attempts:
                raise
            time.sleep(min(1.0, 0.05 * (2**attempt)))


def atomic_bytes(path: Path, payload: bytes) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        replace_with_retry(temporary, path)
    finally:
        with contextlib.suppress(FileNotFoundError):
            os.unlink(temporary)
    return path


def atomic_json(path: Path, value: Any) -> Path:
    return atomic_bytes(path, (json.dumps(value, indent=2, sort_keys=True, default=str) + "\n").encode("utf-8"))


def atomic_csv(path: Path, frame: pd.DataFrame, *, index: bool = False) -> Path:
    return atomic_bytes(path, frame.to_csv(index=index).encode("utf-8"))


def atomic_parquet(path: Path, frame: pd.DataFrame) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.parent / f".{path.name}.{os.getpid()}.tmp"
    try:
        frame.to_parquet(temporary, index=False)
        replace_with_retry(temporary, path)
    finally:
        with contextlib.suppress(FileNotFoundError):
            temporary.unlink()
    return path


def atomic_npz(path: Path, **arrays: np.ndarray) -> Path:
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    return atomic_bytes(path, buffer.getvalue())


def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


if sys.version_info[:2] != (3, 12):
    raise RuntimeError("The authoritative environment requires Python 3.12.")

np.random.seed(int(ACTIVE["icsor"]["split_seed"]))
CONFIG_HASH = sha256_json(ACTIVE)
WORKBOOK_HASH = sha256_file(WORKBOOK_PATH)
NOTEBOOK_HASH = sha256_file(ROOT / "main.ipynb")
LOCK_HASH = sha256_file(ROOT / "uv.lock")
PATHS_HASH = sha256_json(PATHS)
ARTICLE_ELIGIBLE = requested_profile == "full"
NUMERICAL_ENVIRONMENT_CONTRACT = {
    "python": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "threads": int(ACTIVE["run"]["threads"]),
    "packages": {
        name: package_version(name)
        for name in ("numpy", "scipy", "pandas", "pyarrow", "osqp", "scikit-learn")
    },
}

manifest_path = RUN_ROOT / "manifest.json"
RUN_CONTRACT = {
    "configuration_sha256": CONFIG_HASH,
    "workbook_sha256": WORKBOOK_HASH,
    "notebook_sha256": NOTEBOOK_HASH,
    "dependency_lock_sha256": LOCK_HASH,
    "paths_sha256": PATHS_HASH,
    "numerical_environment_sha256": sha256_json(NUMERICAL_ENVIRONMENT_CONTRACT),
}
existing_manifest = read_json(manifest_path) if manifest_path.exists() else None
if existing_manifest is None and PREEXISTING_RUN_ENTRIES:
    raise RuntimeError(f"Refusing an orphaned nonempty run root: {PREEXISTING_RUN_ENTRIES}.")
if existing_manifest and existing_manifest.get("status") == "complete":
    completion_candidate = RUN_ROOT / "COMPLETED.json"
    digest_candidate = RUN_ROOT / "manifest.sha256"
    if not completion_candidate.is_file() or not digest_candidate.is_file():
        raise RuntimeError("The run has a complete manifest but no valid completion seal; use a new run_id.")
    completion_record = read_json(completion_candidate)
    manifest_digest = sha256_file(manifest_path)
    if (
        completion_record.get("run_id") != RUN_ID
        or completion_record.get("profile") != requested_profile
        or completion_record.get("article_eligible") != ARTICLE_ELIGIBLE
        or completion_record.get("contract_sha256") != sha256_json(RUN_CONTRACT)
        or completion_record.get("manifest_sha256") != manifest_digest
        or not digest_candidate.read_text(encoding="ascii").startswith(manifest_digest)
    ):
        raise RuntimeError("The completed run's manifest seal is inconsistent or corrupt.")
    inventory_candidate = RUN_ROOT / "artifact_inventory.csv"
    if (
        not inventory_candidate.is_file()
        or sha256_file(inventory_candidate) != completion_record.get("artifact_inventory_sha256")
    ):
        raise RuntimeError("The completed run's artifact inventory is missing or corrupt.")
    sealed_inventory = pd.read_csv(inventory_candidate)
    if len(sealed_inventory) != int(completion_record.get("artifact_count", -1)):
        raise RuntimeError("The completed run's artifact count does not match its seal.")
    for inventory_row in sealed_inventory.itertuples(index=False):
        sealed_path = (RUN_ROOT / str(inventory_row.path)).resolve()
        if RUN_ROOT not in sealed_path.parents:
            raise RuntimeError("The completed artifact inventory contains an unsafe path.")
        if (
            not sealed_path.is_file()
            or sealed_path.stat().st_size != int(inventory_row.bytes)
            or sha256_file(sealed_path) != str(inventory_row.sha256)
        ):
            raise RuntimeError(f"Completed artifact is missing or corrupt: {inventory_row.path}.")
    raise RuntimeError("A completed run is immutable; choose a new run_id to rerun it.")
if existing_manifest and existing_manifest.get("contract") != RUN_CONTRACT:
    raise RuntimeError("Cannot resume a run created by a different source/input contract.")

MANIFEST: dict[str, Any] = existing_manifest or {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "profile": requested_profile,
    "status": "running",
    "article_eligible": ARTICLE_ELIGIBLE,
    "started_utc": utc_now(),
    "contract": RUN_CONTRACT,
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_cores": psutil.cpu_count(logical=True),
        "ram_bytes": psutil.virtual_memory().total,
        "numerical_environment_contract": NUMERICAL_ENVIRONMENT_CONTRACT,
        "packages": {
            name: package_version(name)
            for name in (
                "numpy", "scipy", "pandas", "pyarrow", "openpyxl", "osqp",
                "scikit-learn", "matplotlib", "psutil", "threadpoolctl", "tqdm",
            )
        },
    },
    "stages": {},
    "artifacts": {},
}


def save_manifest() -> None:
    MANIFEST["updated_utc"] = utc_now()
    atomic_json(manifest_path, MANIFEST)


def register_artifact(name: str, path: Path) -> None:
    resolved = path.resolve()
    if RUN_ROOT not in resolved.parents:
        raise ValueError("Only files below the active run root may be registered.")
    MANIFEST["artifacts"][name] = {
        "path": resolved.relative_to(RUN_ROOT).as_posix(),
        "bytes": resolved.stat().st_size,
        "sha256": sha256_file(resolved),
    }
    save_manifest()


def stage_complete(name: str, **details: Any) -> None:
    MANIFEST["stages"][name] = {"status": "complete", "completed_utc": utc_now(), **details}
    save_manifest()


input_snapshots = {
    DIRS["inputs"] / "params.resolved.json": (json.dumps(ACTIVE, indent=2, sort_keys=True, default=str) + "\n").encode(),
    DIRS["inputs"] / "paths.snapshot.json": (json.dumps(PATHS, indent=2, sort_keys=True, default=str) + "\n").encode(),
    DIRS["inputs"] / "workbook.snapshot.xlsx": WORKBOOK_PATH.read_bytes(),
    DIRS["inputs"] / "main.snapshot.ipynb": (ROOT / "main.ipynb").read_bytes(),
}
for snapshot_path, payload in input_snapshots.items():
    if snapshot_path.exists() and snapshot_path.read_bytes() != payload:
        raise RuntimeError(f"Existing immutable input snapshot changed: {snapshot_path.name}.")
    if not snapshot_path.exists():
        atomic_bytes(snapshot_path, payload)
    register_artifact(f"input_{snapshot_path.stem}", snapshot_path)

print(f"Profile: {requested_profile}")
print(f"Run root: {RUN_ROOT.relative_to(ROOT)}")
print(f"Article eligible: {ARTICLE_ELIGIBLE}")
print(f"Workbook SHA-256: {WORKBOOK_HASH}")

## Mechanistic model and workbook contract

The following cells embed the complete schema-4 ASM2d-TSN implementation used by the preceding
surrogate-projection study. Process rates are finite and non-negative but are not artificially
capped. The workbook formulas, assembled matrices, null-space invariants, fixed kinetic fixture,
and forced BDF fallback are all checked before a dataset may be generated.

In [ ]:
# Compatibility helpers used only by the embedded standalone definitions below.
SIM_PARAMS = copy.deepcopy(ACTIVE["simulation"])
SIM_PARAMS["hyperparameters"] = {
    "n_samples": int(ACTIVE["simulation"]["sampling"]["id_samples"]),
    "seed": int(ACTIVE["simulation"]["sampling"]["seed"]),
    "max_sample_attempts": int(ACTIVE["simulation"]["sampling"]["max_sample_attempts"]),
    "parallel_workers": int(ACTIVE["simulation"]["sampling"]["parallel_workers"]),
    "parallel_chunk_size": int(ACTIVE["simulation"]["sampling"]["parallel_chunk_size"]),
}


def get_repo_root(repo_root: str | Path | None = None) -> Path:
    return Path(repo_root).resolve() if repo_root is not None else ROOT


def load_json_file(path: str | Path) -> dict[str, Any]:
    return read_json(Path(path))


def load_pickle_file(path: str | Path) -> Any:
    with Path(path).open("rb") as handle:
        return pickle.load(handle)


def save_json_file(path: str | Path, data: dict[str, Any], *, indent: int = 2) -> Path:
    return atomic_json(Path(path), data)


def save_pickle_file(path: str | Path, data: Any) -> Path:
    return atomic_bytes(Path(path), pickle.dumps(data, protocol=5))


def compute_file_sha256(path: str | Path, *, chunk_size: int = 1024 * 1024) -> str:
    return sha256_file(Path(path), chunk_size=chunk_size)


def load_model_params(model_name: str, repo_root: str | Path | None = None) -> dict[str, Any]:
    if model_name != "asm2d_tsn_simulation":
        raise KeyError(model_name)
    return copy.deepcopy(SIM_PARAMS)


def load_paths_config(repo_root: str | Path | None = None) -> dict[str, Any]:
    return {
        "asm2d_tsn_reference_workbook": PATHS["workbook"],
        "asm2d_tsn_composition_cache_pattern": str(
            (DIRS["matrices"] / "composition_cache_{workbook_hash}.pkl").relative_to(ROOT)
        ),
        "asm2d_tsn_composition_cache_metadata_pattern": str(
            (DIRS["matrices"] / "composition_cache_{workbook_hash}.json").relative_to(ROOT)
        ),
        "asm2d_tsn_simulation_data_pattern": str((DIRS["datasets"] / "id.parquet").relative_to(ROOT)),
        "asm2d_tsn_simulation_metadata_pattern": str((DIRS["datasets"] / "id.metadata.json").relative_to(ROOT)),
    }


def render_simulation_artifact_paths(*args: Any, **kwargs: Any) -> tuple[Path, Path]:
    return DIRS["datasets"] / "id.parquet", DIRS["datasets"] / "id.metadata.json"


def save_simulation_artifacts(*args: Any, **kwargs: Any) -> tuple[Path, Path]:
    raise RuntimeError("The standalone workflow uses its atomic result-bundle persistence layer.")

MODEL_NAME = "asm2d_tsn_simulation"
WORKBOOK_PATH_KEY = "asm2d_tsn_reference_workbook"
DATA_PATTERN_KEY = "asm2d_tsn_simulation_data_pattern"
METADATA_PATTERN_KEY = "asm2d_tsn_simulation_metadata_pattern"
COMPOSITION_CACHE_PATTERN_KEY = "asm2d_tsn_composition_cache_pattern"
COMPOSITION_CACHE_METADATA_PATTERN_KEY = "asm2d_tsn_composition_cache_metadata_pattern"
STOICHIOMETRIC_SHEET_NAME = "stoichiometric_matrix"
COMPOSITION_SHEET_NAME = "composition_matrix"
PARAMETER_SHEET_NAME = "parameter_table"
PARAMETER_VALUE_COLUMN_INDEX = 5

_EXCEL_REFERENCE_PATTERN = re.compile(
    r"(?<![A-Za-z0-9_])(?:(?:'(?P<sheet_quoted>[^']+)')|(?P<sheet_unquoted>[A-Za-z_][A-Za-z0-9_]*))?!?\$?(?P<column>[A-Za-z]{1,3})\$?(?P<row>\d+)"
)

HEADER_FILL = PatternFill(fill_type="solid", fgColor="D8DEE6")
SECTION_FILL = PatternFill(fill_type="solid", fgColor="EEF2F5")
HEADER_FONT = Font(bold=True, color="22303C")

STOICHIOMETRIC_COEFFICIENTS: list[dict[str, dict[str, str]]] = [
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {"coefficients": {"S_A": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {"coefficients": {"S_A": "1", "S_F": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_H": "-1"}},
    {"coefficients": {"S_A": "-1", "X_PP": "-{Y_PO4}", "X_PHA": "1"}},
    {"coefficients": {"S_O": "-{Y_PHA}", "X_PP": "1", "X_PHA": "-{Y_PHA}"}},
    {
        "coefficients": {
            "S_NO2": "{Y_PHA}/(8/7)",
            "S_NO3": "-({Y_PHA}/(8/7))",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {
        "coefficients": {
            "S_N2": "{Y_PHA}/1.72",
            "S_NO2": "-({Y_PHA}/1.72)",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {"coefficients": {"S_O": "1-1/{Y_PAO}", "X_PAO": "1", "X_PHA": "-1/{Y_PAO}"}},
    {
        "coefficients": {
            "S_NO2": "(1-{Y_PAO})/((8/7)*{Y_PAO})",
            "S_NO3": "-((1-{Y_PAO})/((8/7)*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {
        "coefficients": {
            "S_N2": "(1-{Y_PAO})/(1.72*{Y_PAO})",
            "S_NO2": "-((1-{Y_PAO})/(1.72*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_PAO": "-1"}},
    {"coefficients": {"X_PP": "-1"}},
    {"coefficients": {"S_A": "1", "X_PHA": "-1"}},
    {"coefficients": {"S_NO2": "1/{Y_AOB}", "S_O": "-((3.43-{Y_AOB})/{Y_AOB})", "X_AOB": "1"}},
    {"coefficients": {"S_NO2": "-1/{Y_NOB}", "S_NO3": "1/{Y_NOB}", "S_O": "-((1.14-{Y_NOB})/{Y_NOB})", "X_NOB": "1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_AOB": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_NOB": "-1"}},
    {"coefficients": {"S_PO4": "-1", "X_MeOH": "-3.45", "X_MeP": "4.87"}},
    {"coefficients": {"S_PO4": "1", "X_MeOH": "3.45", "X_MeP": "-4.87"}},
]

COMPOSITION_FORMULAS: dict[str, dict[str, str]] = {
    "S_A": {"COD": "1"},
    "S_F": {"COD": "1", "TN": "{i_NSF}", "TKN": "{i_NSF}", "TP": "{i_PSF}"},
    "S_I": {"COD": "1", "TN": "{i_NSI}", "TKN": "{i_NSI}", "TP": "{i_PSI}"},
    "S_NH4": {"TN": "1", "TKN": "1"},
    "S_NO2": {"TN": "1"},
    "S_NO3": {"TN": "1"},
    "S_PO4": {"TP": "1"},
    "X_I": {"COD": "1", "TN": "{i_NXI}", "TKN": "{i_NXI}", "TP": "{i_PXI}", "TSS": "{i_TSS_XI}"},
    "X_S": {"COD": "1", "TN": "{i_NXS}", "TKN": "{i_NXS}", "TP": "{i_PXS}", "TSS": "{i_TSS_XS}"},
    "X_H": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_PAO": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_PP": {"TP": "1", "TSS": "{i_TSS_PP}"},
    "X_PHA": {"COD": "1", "TSS": "{i_TSS_PHA}"},
    "X_AOB": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_NOB": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_MeOH": {"TSS": "1"},
    "X_MeP": {"TP": "{i_PMeP}", "TSS": "1"},
}

NITROGEN_CONTINUITY_TERMS = {
    "S_F": "{i_NSF}",
    "S_I": "{i_NSI}",
    "S_N2": "1",
    "S_NO2": "1",
    "S_NO3": "1",
    "X_I": "{i_NXI}",
    "X_S": "{i_NXS}",
    "X_H": "{i_NBM}",
    "X_PAO": "{i_NBM}",
    "X_AOB": "{i_NBM}",
    "X_NOB": "{i_NBM}",
}

PHOSPHORUS_CONTINUITY_TERMS = {
    "S_F": "{i_PSF}",
    "S_I": "{i_PSI}",
    "X_I": "{i_PXI}",
    "X_S": "{i_PXS}",
    "X_H": "{i_PBM}",
    "X_PAO": "{i_PBM}",
    "X_PP": "1",
    "X_AOB": "{i_PBM}",
    "X_NOB": "{i_PBM}",
    "X_MeP": "{i_PMeP}",
}

def load_asm2d_tsn_simulation_params(repo_root: str | Path | None = None) -> dict[str, Any]:
    """Load the configured ASM2D-TSN simulation definition."""

    return load_model_params(MODEL_NAME, repo_root)


def resolve_asm2d_tsn_workbook_path(
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> Path:
    """Resolve the configured canonical workbook path."""

    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    return root / Path(config[WORKBOOK_PATH_KEY])


def resolve_asm2d_tsn_simulation_artifact_paths(
    repo_root: str | Path | None = None,
    *,
    timestamp: str | None = None,
    paths_config: Mapping[str, Any] | None = None,
) -> tuple[Path, Path, str]:
    """Resolve the configured ASM2D-TSN dataset and metadata output paths."""

    return render_simulation_artifact_paths(
        MODEL_NAME,
        repo_root=repo_root,
        timestamp=timestamp,
        paths_config=paths_config,
        data_pattern_key=DATA_PATTERN_KEY,
        metadata_pattern_key=METADATA_PATTERN_KEY,
    )


def resolve_asm2d_tsn_composition_cache_paths(
    workbook_hash: str,
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> tuple[Path, Path]:
    """Resolve configured cache paths for one workbook-derived composition artifact."""

    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    matrix_path = root / Path(config[COMPOSITION_CACHE_PATTERN_KEY].format(workbook_hash=str(workbook_hash)))
    metadata_path = root / Path(
        config[COMPOSITION_CACHE_METADATA_PATTERN_KEY].format(workbook_hash=str(workbook_hash))
    )
    return matrix_path, metadata_path


def create_asm2d_tsn_workbook(
    workbook_path: str | Path | None = None,
    *,
    repo_root: str | Path | None = None,
    model_params: Mapping[str, Any] | None = None,
) -> Path:
    """Create the canonical ASM2D-TSN workbook with formula-driven matrices."""

    workbook = build_asm2d_tsn_workbook(model_params=model_params, repo_root=repo_root)
    output_path = Path(workbook_path) if workbook_path is not None else resolve_asm2d_tsn_workbook_path(repo_root)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    workbook.save(output_path)
    return output_path.resolve()


def build_asm2d_tsn_workbook(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
) -> Workbook:
    """Build the workbook object for the configured ASM2D-TSN reference model."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    workbook_config = _validate_workbook_config(params)
    parameter_refs = _build_parameter_reference_map(workbook_config["parameters"])

    workbook = Workbook()
    stoichiometric_sheet = workbook.active
    stoichiometric_sheet.title = STOICHIOMETRIC_SHEET_NAME
    composition_sheet = workbook.create_sheet(COMPOSITION_SHEET_NAME)
    parameter_sheet = workbook.create_sheet(PARAMETER_SHEET_NAME)

    _write_stoichiometric_sheet(stoichiometric_sheet, workbook_config, parameter_refs)
    _write_composition_sheet(composition_sheet, workbook_config, parameter_refs)
    _write_parameter_sheet(parameter_sheet, workbook_config["parameters"])

    for worksheet in workbook.worksheets:
        _auto_size_columns(worksheet)
        worksheet.auto_filter.ref = worksheet.dimensions

    return workbook


def _normalize_excel_reference(sheet_name: str, coordinate: str) -> str:
    return f"{str(sheet_name).strip().lower()}!{str(coordinate).replace('$', '').upper()}"


def _build_workbook_numeric_lookup(workbook) -> dict[str, float]:
    numeric_lookup: dict[str, float] = {}
    for worksheet in workbook.worksheets:
        for row in worksheet.iter_rows(
            min_row=1,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column,
        ):
            for cell in row:
                cell_value = cell.value
                if isinstance(cell_value, bool):
                    continue
                if isinstance(cell_value, (int, float)):
                    numeric_lookup[_normalize_excel_reference(worksheet.title, cell.coordinate)] = float(cell_value)
    return numeric_lookup


def _evaluate_workbook_formula(
    formula: str,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    expression = str(formula).strip()
    if expression.startswith("="):
        expression = expression[1:]

    def _replace_reference(match: re.Match[str]) -> str:
        sheet_name = match.group("sheet_quoted") or match.group("sheet_unquoted") or str(current_sheet_name)
        coordinate = f"{match.group('column').upper()}{match.group('row')}"
        lookup_key = _normalize_excel_reference(sheet_name, coordinate)
        if lookup_key not in numeric_lookup:
            raise KeyError(
                "Workbook formula references a non-numeric or missing cell: "
                f"{sheet_name}!{coordinate}."
            )
        return str(float(numeric_lookup[lookup_key]))

    resolved_expression = _EXCEL_REFERENCE_PATTERN.sub(_replace_reference, expression).replace("^", "**")
    return float(eval(resolved_expression, {"__builtins__": {}}, {}))


def _coerce_workbook_composition_value(
    cell_value: Any,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    if cell_value is None:
        return 0.0
    if isinstance(cell_value, bool):
        return float(cell_value)
    if isinstance(cell_value, (int, float)):
        return float(cell_value)

    text_value = str(cell_value).strip()
    if not text_value:
        return 0.0
    if text_value.startswith("="):
        return _evaluate_workbook_formula(
            text_value,
            numeric_lookup,
            current_sheet_name=current_sheet_name,
        )

    try:
        return float(text_value)
    except ValueError as error:
        raise ValueError(f"Workbook composition_matrix contains a non-numeric value: {text_value!r}") from error


def _read_composition_matrix_from_workbook(
    workbook_path: str | Path,
    *,
    expected_state_columns: list[str],
) -> dict[str, Any]:
    workbook = load_workbook(filename=Path(workbook_path), data_only=False)
    try:
        if COMPOSITION_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{COMPOSITION_SHEET_NAME}'.")
        if PARAMETER_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{PARAMETER_SHEET_NAME}'.")

        worksheet = workbook[COMPOSITION_SHEET_NAME]
        header_values = [str(cell.value).strip() if cell.value is not None else "" for cell in worksheet[1]]
        if "state_variable" not in header_values:
            raise KeyError("Workbook composition_matrix must include a 'state_variable' header column.")

        state_column_number = header_values.index("state_variable") + 1
        reserved_headers = {"state_group", "state_variable", "unit"}
        composite_header_pairs = [
            (column_number, header_name)
            for column_number, header_name in enumerate(header_values, start=1)
            if header_name and header_name not in reserved_headers
        ]
        if not composite_header_pairs:
            raise ValueError("Workbook composition_matrix must define at least one composite output column.")

        measured_output_columns = [header_name for _, header_name in composite_header_pairs]
        _validate_unique_names(measured_output_columns, "composition_matrix output columns")

        numeric_lookup = _build_workbook_numeric_lookup(workbook)
        state_columns: list[str] = []
        coefficients_by_state: list[list[float]] = []
        for row_number in range(2, worksheet.max_row + 1):
            raw_state_name = worksheet.cell(row=row_number, column=state_column_number).value
            if raw_state_name is None:
                continue
            state_name = str(raw_state_name).strip()
            if not state_name:
                continue

            state_columns.append(state_name)
            row_coefficients: list[float] = []
            for column_number, _ in composite_header_pairs:
                raw_value = worksheet.cell(row=row_number, column=column_number).value
                row_coefficients.append(
                    _coerce_workbook_composition_value(
                        raw_value,
                        numeric_lookup,
                        current_sheet_name=worksheet.title,
                    )
                )
            coefficients_by_state.append(row_coefficients)

        if not state_columns:
            raise ValueError("Workbook composition_matrix must define at least one state_variable row.")
        _validate_unique_names(state_columns, "composition_matrix state_variable")

        if state_columns != expected_state_columns:
            raise ValueError(
                "Workbook composition_matrix state_variable rows must match configured workbook state_columns "
                "exactly and in order."
            )

        composition_matrix = np.asarray(coefficients_by_state, dtype=float).T
        expected_shape = (len(measured_output_columns), len(state_columns))
        if composition_matrix.shape != expected_shape:
            raise ValueError(
                "Workbook composition_matrix shape must match measured_output_columns x state_columns."
            )

        return {
            "state_columns": state_columns,
            "measured_output_columns": measured_output_columns,
            "composition_matrix": composition_matrix,
        }
    finally:
        workbook.close()


def _build_workbook_fingerprint(workbook_path: Path) -> dict[str, Any]:
    resolved_path = Path(workbook_path).resolve()
    stat_info = resolved_path.stat()
    return {
        "workbook_path": resolved_path.as_posix(),
        "workbook_sha256": compute_file_sha256(resolved_path),
        "workbook_mtime_ns": int(stat_info.st_mtime_ns),
        "workbook_size_bytes": int(stat_info.st_size),
    }


def _validate_cached_composition_payload(cache_payload: Mapping[str, Any]) -> None:
    required_keys = ("state_columns", "measured_output_columns", "composition_matrix")
    for required_key in required_keys:
        if required_key not in cache_payload:
            raise KeyError(f"Cached composition payload is missing required key '{required_key}'.")


def load_asm2d_tsn_workbook_composition(
    *,
    repo_root: str | Path | None = None,
    workbook_path: str | Path | None = None,
    model_params: Mapping[str, Any] | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_cache: bool = True,
) -> dict[str, Any]:
    """Load workbook-derived composition schema and coefficients, with fingerprinted cache reuse."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    workbook_config = _validate_workbook_config(params)
    expected_state_columns = list(workbook_config["state_columns"])
    resolved_workbook_path = (
        Path(workbook_path)
        if workbook_path is not None
        else resolve_asm2d_tsn_workbook_path(repo_root, paths_config=paths_config)
    )
    if not resolved_workbook_path.exists():
        raise FileNotFoundError(f"ASM2D-TSN workbook not found at {resolved_workbook_path}.")

    fingerprint = _build_workbook_fingerprint(resolved_workbook_path)
    cache_matrix_path, cache_metadata_path = resolve_asm2d_tsn_composition_cache_paths(
        fingerprint["workbook_sha256"],
        repo_root=repo_root,
        paths_config=paths_config,
    )

    if use_cache and cache_matrix_path.exists() and cache_metadata_path.exists():
        cache_metadata = load_json_file(cache_metadata_path)
        if (
            str(cache_metadata.get("workbook_sha256")) == str(fingerprint["workbook_sha256"])
            and int(cache_metadata.get("workbook_mtime_ns", -1)) == int(fingerprint["workbook_mtime_ns"])
            and int(cache_metadata.get("workbook_size_bytes", -1)) == int(fingerprint["workbook_size_bytes"])
        ):
            cache_payload = load_pickle_file(cache_matrix_path)
            _validate_cached_composition_payload(cache_payload)
            state_columns = [str(name) for name in cache_payload["state_columns"]]
            measured_output_columns = [str(name) for name in cache_payload["measured_output_columns"]]
            if state_columns != expected_state_columns:
                raise ValueError(
                    "Cached composition state_columns no longer match configured workbook state_columns."
                )
            composition_matrix = np.asarray(cache_payload["composition_matrix"], dtype=float)
            if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
                raise ValueError("Cached composition_matrix shape is invalid for cached schema.")
            return {
                "state_columns": state_columns,
                "measured_output_columns": measured_output_columns,
                "composition_matrix": composition_matrix,
                **fingerprint,
                "cache_source": "cache",
                "cache_paths": {
                    "composition_matrix": cache_matrix_path,
                    "composition_metadata": cache_metadata_path,
                },
            }

    parsed_composition = _read_composition_matrix_from_workbook(
        resolved_workbook_path,
        expected_state_columns=expected_state_columns,
    )
    composition_payload = {
        "state_columns": list(parsed_composition["state_columns"]),
        "measured_output_columns": list(parsed_composition["measured_output_columns"]),
        "composition_matrix": np.asarray(parsed_composition["composition_matrix"], dtype=float),
    }

    if use_cache:
        save_pickle_file(cache_matrix_path, composition_payload)
        save_json_file(
            cache_metadata_path,
            {
                "cache_schema_version": 1,
                "state_columns": composition_payload["state_columns"],
                "measured_output_columns": composition_payload["measured_output_columns"],
                **fingerprint,
            },
        )

    return {
        **composition_payload,
        **fingerprint,
        "cache_source": "workbook",
        "cache_paths": {
            "composition_matrix": cache_matrix_path,
            "composition_metadata": cache_metadata_path,
        },
    }


def get_asm2d_tsn_matrices(
    model_params: Mapping[str, Any] | None = None,
    *,
    repo_root: str | Path | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_composition_cache: bool = True,
) -> dict[str, Any]:
    """Build numeric Petersen and composition matrices for the configured ASM2D-TSN model."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    composition_bundle = load_asm2d_tsn_workbook_composition(
        repo_root=repo_root,
        model_params=params,
        paths_config=paths_config,
        use_cache=use_composition_cache,
    )
    measured_output_columns = list(composition_bundle["measured_output_columns"])
    runtime = _validate_runtime_structure(params, measured_output_columns=measured_output_columns)
    workbook_config = runtime["workbook_config"]
    parameter_values = _build_parameter_value_map(workbook_config["parameters"])
    state_columns = list(runtime["state_columns"])
    process_names = list(runtime["process_names"])
    process_types = list(runtime["process_types"])
    state_index = _build_state_index(state_columns)

    composition_state_columns = list(composition_bundle["state_columns"])
    if composition_state_columns != state_columns:
        raise ValueError(
            "Workbook composition_matrix state columns do not match the configured ASM2D-TSN state columns."
        )

    petersen_matrix = np.zeros((len(process_names), len(state_columns)), dtype=float)
    composition_matrix = np.asarray(composition_bundle["composition_matrix"], dtype=float)
    if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
        raise ValueError(
            "Workbook composition_matrix shape must match measured_output_columns x state_columns."
        )

    for row_index, process_definition in enumerate(STOICHIOMETRIC_COEFFICIENTS):
        row_values = petersen_matrix[row_index]
        direct_coefficients = process_definition["coefficients"]

        for state_name, expression in direct_coefficients.items():
            row_values[state_index[state_name]] = _evaluate_numeric_expression(expression, parameter_values)

        row_values[state_index["S_NH4"]] = -sum(
            row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
            for state_name, factor_expression in NITROGEN_CONTINUITY_TERMS.items()
        )

        if "S_PO4" not in direct_coefficients:
            row_values[state_index["S_PO4"]] = -sum(
                row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
                for state_name, factor_expression in PHOSPHORUS_CONTINUITY_TERMS.items()
            )

        row_values[state_index["S_ALK"]] = (
            row_values[state_index["S_NH4"]] / 14.0
            - row_values[state_index["S_NO2"]] / 14.0
            - row_values[state_index["S_NO3"]] / 14.0
            + row_values[state_index["S_PO4"]] / 31.0
        )

    return {
        "petersen_matrix": petersen_matrix,
        "composition_matrix": composition_matrix,
        "process_names": process_names,
        "process_types": process_types,
        "state_index": state_index,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "composition_workbook_path": str(composition_bundle["workbook_path"]),
        "composition_workbook_sha256": str(composition_bundle["workbook_sha256"]),
        "composition_workbook_mtime_ns": int(composition_bundle["workbook_mtime_ns"]),
        "composition_workbook_size_bytes": int(composition_bundle["workbook_size_bytes"]),
        "composition_cache_source": str(composition_bundle["cache_source"]),
        "composition_cache_paths": dict(composition_bundle["cache_paths"]),
    }


def build_asm2d_tsn_metadata(
    model_params: Mapping[str, Any],
    *,
    sample_count: int,
    random_seed: int,
    dataset_file: str | None = None,
    measured_output_columns: list[str] | None = None,
    composition_source: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
) -> dict[str, Any]:
    """Create the metadata contract for the ASM2D-TSN mixed-schema dataset."""

    if measured_output_columns is None:
        measured_output_columns = list(
            load_asm2d_tsn_workbook_composition(
                repo_root=repo_root,
                model_params=model_params,
            )["measured_output_columns"]
        )

    runtime = _validate_runtime_structure(model_params, measured_output_columns=measured_output_columns)
    state_columns = list(runtime["state_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])
    process_names = list(runtime["process_names"])
    process_types = list(runtime["process_types"])
    operational_columns = list(runtime["operational_columns"])
    influent_fraction_columns = [f"In_{name}" for name in state_columns]
    influent_composite_columns = [f"In_{name}" for name in measured_output_columns]
    effluent_fraction_columns = [f"Out_{name}" for name in state_columns]
    dependent_columns = [f"Out_{name}" for name in measured_output_columns]

    return {
        "simulation_name": MODEL_NAME,
        "n_samples": sample_count,
        "random_seed": random_seed,
        "sampling_method": "latin_hypercube",
        "dependent_columns": dependent_columns,
        "independent_columns": operational_columns + influent_fraction_columns,
        "identifier_columns": [],
        "ignored_columns": influent_composite_columns + effluent_fraction_columns,
        "dataset_file": dataset_file,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "operational_columns": operational_columns,
        "influent_fraction_columns": influent_fraction_columns,
        "influent_composite_columns": influent_composite_columns,
        "effluent_fraction_columns": effluent_fraction_columns,
        "effluent_composite_columns": dependent_columns,
        "processes": process_names,
        "process_types": process_types,
        "petersen_matrix_shape": [len(process_names), len(state_columns)],
        "composition_matrix_shape": [len(measured_output_columns), len(state_columns)],
        "schema_version": str(model_params["schema_version"]),
        "composition_source": dict(composition_source or {}),
    }


def generate_asm2d_tsn_dataset(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
    n_samples: int | None = None,
    random_seed: int | None = None,
    parallel_workers: int | None = None,
    parallel_chunk_size: int | None = None,
    include_debug_data: bool = False,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    """Generate a mechanistic steady-state ASM2D-TSN dataset with input/output fractions and composites."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    matrix_bundle = get_asm2d_tsn_matrices(params, repo_root=repo_root)
    runtime = _validate_runtime_structure(
        params,
        measured_output_columns=list(matrix_bundle["measured_output_columns"]),
    )
    configured_hyperparameters = params["hyperparameters"]
    sample_count = int(n_samples if n_samples is not None else configured_hyperparameters["n_samples"])
    if sample_count < 0:
        raise ValueError("n_samples must be greater than or equal to 0.")

    seed = int(random_seed if random_seed is not None else configured_hyperparameters["seed"])
    requested_parallel_workers = int(
        parallel_workers if parallel_workers is not None else configured_hyperparameters.get("parallel_workers", 1)
    )
    requested_parallel_chunk_size = int(
        parallel_chunk_size
        if parallel_chunk_size is not None
        else configured_hyperparameters.get("parallel_chunk_size", sample_count or 1)
    )
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])

    influent_states = np.zeros((sample_count, len(state_columns)), dtype=float)
    operational = np.zeros((sample_count, len(operational_columns)), dtype=float)
    measured_outputs = np.zeros((sample_count, len(measured_output_columns)), dtype=float)
    effluent_states = np.zeros((sample_count, len(state_columns)), dtype=float)
    solver_diagnostic_records: list[dict[str, Any]] = []
    chunk_results: list[tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[dict[str, Any]]]] = []
    if sample_count > 0:
        worker_count = _resolve_parallel_workers(requested_parallel_workers, sample_count)
        chunk_size = min(_resolve_parallel_chunk_size(requested_parallel_chunk_size), sample_count)
        chunk_specs = [
            {
                "chunk_start": chunk_start,
                "chunk_size": min(chunk_size, sample_count - chunk_start),
                "chunk_seed": seed + chunk_index,
                "model_params": params,
                "matrix_bundle": matrix_bundle,
                "runtime": runtime,
                "collect_debug_data": include_debug_data,
            }
            for chunk_index, chunk_start in enumerate(range(0, sample_count, chunk_size))
        ]

        if worker_count == 1 or len(chunk_specs) == 1:
            progress_bar = tqdm(
                total=sample_count,
                desc=progress_description or "ASM2D-TSN simulation",
                unit="sample",
                disable=not show_progress,
            )
            try:
                chunk_results = [
                    _generate_asm2d_tsn_dataset_chunk(progress_bar=progress_bar, **chunk_spec)
                    for chunk_spec in chunk_specs
                ]
            finally:
                progress_bar.close()
        else:
            with ProcessPoolExecutor(max_workers=worker_count) as executor:
                future_map = {
                    executor.submit(_generate_asm2d_tsn_dataset_chunk, **chunk_spec): int(chunk_spec["chunk_size"])
                    for chunk_spec in chunk_specs
                }
                progress_bar = tqdm(
                    total=sample_count,
                    desc=progress_description or "ASM2D-TSN simulation",
                    unit="sample",
                    disable=not show_progress,
                )
                try:
                    for future in as_completed(future_map):
                        chunk_results.append(future.result())
                        progress_bar.update(future_map[future])
                finally:
                    progress_bar.close()

    for chunk_start, chunk_influent, chunk_operational, chunk_effluent, chunk_measured, chunk_diagnostics in chunk_results:
        chunk_end = chunk_start + len(chunk_influent)
        influent_states[chunk_start:chunk_end] = chunk_influent
        operational[chunk_start:chunk_end] = chunk_operational
        effluent_states[chunk_start:chunk_end] = chunk_effluent
        measured_outputs[chunk_start:chunk_end] = chunk_measured
        if include_debug_data:
            solver_diagnostic_records.extend(chunk_diagnostics)

    influent_df = pd.DataFrame(influent_states, columns=[f"In_{name}" for name in state_columns])
    influent_composite_df = pd.DataFrame(
        influent_states @ np.asarray(matrix_bundle["composition_matrix"], dtype=float).T,
        columns=[f"In_{name}" for name in measured_output_columns],
    )
    operational_df = pd.DataFrame(operational, columns=operational_columns)
    effluent_df = pd.DataFrame(effluent_states, columns=[f"Out_{name}" for name in state_columns])
    measured_df = pd.DataFrame(measured_outputs, columns=[f"Out_{name}" for name in measured_output_columns])
    dataset = pd.concat([operational_df, influent_df, influent_composite_df, effluent_df, measured_df], axis=1)

    metadata = build_asm2d_tsn_metadata(
        params,
        sample_count=sample_count,
        random_seed=seed,
        measured_output_columns=measured_output_columns,
        composition_source={
            "workbook_path": matrix_bundle["composition_workbook_path"],
            "workbook_sha256": matrix_bundle["composition_workbook_sha256"],
            "workbook_mtime_ns": matrix_bundle["composition_workbook_mtime_ns"],
            "workbook_size_bytes": matrix_bundle["composition_workbook_size_bytes"],
            "cache_source": matrix_bundle["composition_cache_source"],
        },
        repo_root=repo_root,
    )

    simulation_bundle = dict(matrix_bundle)
    if include_debug_data:
        solver_diagnostics = pd.DataFrame(solver_diagnostic_records).sort_values("sample_index").reset_index(drop=True)
        simulation_bundle["effluent_states"] = pd.DataFrame(effluent_states, columns=state_columns)
        simulation_bundle["solver_diagnostics"] = solver_diagnostics
        simulation_bundle["solver_summary"] = _summarize_asm2d_tsn_solver_diagnostics(
            solver_diagnostics,
            float(runtime["solver"]["acceptance_residual_max"]),
        )

    return dataset, metadata, simulation_bundle


def run_asm2d_tsn_simulation(
    *,
    save_artifacts: bool = True,
    repo_root: str | Path | None = None,
    n_samples: int | None = None,
    random_seed: int | None = None,
    parallel_workers: int | None = None,
    parallel_chunk_size: int | None = None,
    timestamp: str | None = None,
    include_debug_data: bool = False,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> dict[str, Any]:
    """Run the ASM2D-TSN steady-state simulation and optionally persist artifacts."""

    params = load_asm2d_tsn_simulation_params(repo_root)
    simulation_bundle: dict[str, Any]
    dataset, metadata, simulation_bundle = generate_asm2d_tsn_dataset(
        model_params=params,
        repo_root=repo_root,
        n_samples=n_samples,
        random_seed=random_seed,
        parallel_workers=parallel_workers,
        parallel_chunk_size=parallel_chunk_size,
        include_debug_data=include_debug_data,
        show_progress=show_progress,
        progress_description=progress_description,
    )

    artifact_paths: dict[str, Path | None] = {
        "dataset_csv": None,
        "metadata_json": None,
    }

    if save_artifacts:
        dataset_path, metadata_path, persisted_metadata = save_simulation_artifacts(
            dataset,
            metadata,
            MODEL_NAME,
            repo_root=repo_root,
            timestamp=timestamp,
            data_pattern_key=DATA_PATTERN_KEY,
            metadata_pattern_key=METADATA_PATTERN_KEY,
        )
        metadata = persisted_metadata
        artifact_paths = {
            "dataset_csv": dataset_path,
            "metadata_json": metadata_path,
        }

    return {
        "dataset": dataset,
        "metadata": metadata,
        "petersen_matrix": simulation_bundle["petersen_matrix"],
        "composition_matrix": simulation_bundle["composition_matrix"],
        "matrix_bundle": simulation_bundle,
        "composite_matrix": simulation_bundle["composition_matrix"],
        "artifact_paths": artifact_paths,
        "effluent_states": simulation_bundle.get("effluent_states"),
        "solver_diagnostics": simulation_bundle.get("solver_diagnostics"),
        "solver_summary": simulation_bundle.get("solver_summary"),
    }


def sweep_asm2d_tsn_operating_space(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
    n_samples: int = 512,
    random_seed: int | None = None,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> dict[str, Any]:
    """Sample the configured operating space and summarize ASM2D-TSN solver behavior."""

    if n_samples < 1:
        raise ValueError("n_samples must be at least 1.")

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    matrix_bundle = get_asm2d_tsn_matrices(params, repo_root=repo_root)
    runtime = _validate_runtime_structure(
        params,
        measured_output_columns=list(matrix_bundle["measured_output_columns"]),
    )
    configured_hyperparameters = params["hyperparameters"]
    seed = int(random_seed if random_seed is not None else configured_hyperparameters["seed"])
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    state_index = dict(matrix_bundle["state_index"])

    # Pre-generate a joint LHS design for all sweep points.
    all_columns = state_columns + operational_columns
    all_ranges = {**runtime["influent_state_ranges"], **runtime["operational_ranges"]}
    candidate_pool = _generate_lhs_candidate_pool(seed, n_samples, all_columns, all_ranges)
    n_state = len(state_columns)

    influent_samples = np.zeros((n_samples, len(state_columns)), dtype=float)
    operational_samples = np.zeros((n_samples, len(operational_columns)), dtype=float)
    effluent_samples = np.zeros((n_samples, len(state_columns)), dtype=float)
    diagnostic_records: list[dict[str, Any]] = []

    progress_bar = tqdm(
        total=n_samples,
        desc=progress_description or "ASM2D-TSN sweep",
        unit="sample",
        disable=not show_progress,
    )
    try:
        for sample_index in range(n_samples):
            sampled_state = candidate_pool[sample_index, :n_state]
            influent_state = _build_influent_state_sample(sampled_state)
            operating_point = candidate_pool[sample_index, n_state:]
            effluent_state, diagnostics = simulate_asm2d_tsn_steady_state(
                influent_state=influent_state,
                hrt_hours=float(operating_point[0]),
                aeration=float(operating_point[1]),
                model_params=params,
                matrix_bundle=matrix_bundle,
                previous_solution=None,
                enforce_acceptance=False,
            )
            influent_samples[sample_index] = influent_state
            operational_samples[sample_index] = operating_point
            effluent_samples[sample_index] = effluent_state
            diagnostic_record = dict(diagnostics)
            diagnostic_record["sample_index"] = sample_index
            diagnostic_record["HRT"] = float(operating_point[0])
            diagnostic_record["Aeration"] = float(operating_point[1])
            diagnostic_records.append(diagnostic_record)
            progress_bar.update(1)
    finally:
        progress_bar.close()

    solver_diagnostics = pd.DataFrame(diagnostic_records).sort_values("sample_index").reset_index(drop=True)
    summary = _summarize_asm2d_tsn_solver_diagnostics(
        solver_diagnostics,
        float(runtime["solver"]["acceptance_residual_max"]),
    )

    return {
        "influent_states": pd.DataFrame(influent_samples, columns=state_columns),
        "operating_conditions": pd.DataFrame(operational_samples, columns=operational_columns),
        "effluent_states": pd.DataFrame(effluent_samples, columns=state_columns),
        "solver_diagnostics": solver_diagnostics,
        "summary": summary,
        "matrix_bundle": matrix_bundle,
    }


def _validate_workbook_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "workbook" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a workbook section.")

    workbook_config = dict(model_params["workbook"])
    expected_sheets = [STOICHIOMETRIC_SHEET_NAME, COMPOSITION_SHEET_NAME, PARAMETER_SHEET_NAME]
    configured_sheets = list(workbook_config["sheets"])
    if configured_sheets != expected_sheets:
        raise ValueError("asm2d_tsn_simulation workbook sheets must match the required three-sheet contract.")

    dissolved_state_columns = list(workbook_config["dissolved_state_columns"])
    particulate_state_columns = list(workbook_config["particulate_state_columns"])
    state_columns = list(workbook_config["state_columns"])
    legacy_composite_variables = workbook_config.get("composite_variables")
    processes = list(workbook_config["processes"])
    parameter_rows = list(workbook_config["parameters"])
    state_units = dict(workbook_config["state_units"])

    _validate_unique_names(dissolved_state_columns, "dissolved_state_columns")
    _validate_unique_names(particulate_state_columns, "particulate_state_columns")
    _validate_unique_names(state_columns, "state_columns")
    _resolve_workbook_composite_variables(
        state_columns,
        legacy_composite_variables=(
            None if legacy_composite_variables is None else list(legacy_composite_variables)
        ),
    )

    if state_columns != dissolved_state_columns + particulate_state_columns:
        raise ValueError("asm2d_tsn_simulation state_columns must concatenate dissolved and particulate state columns.")

    missing_state_units = [state_name for state_name in state_columns if state_name not in state_units]
    if missing_state_units:
        missing_display = ", ".join(missing_state_units)
        raise KeyError(f"asm2d_tsn_simulation missing state units for: {missing_display}")

    if len(processes) != len(STOICHIOMETRIC_COEFFICIENTS):
        raise ValueError("asm2d_tsn_simulation workbook process count does not match the stoichiometric matrix definition.")

    process_indices = [int(process["index"]) for process in processes]
    if process_indices != list(range(1, len(processes) + 1)):
        raise ValueError("asm2d_tsn_simulation workbook processes must be sequentially indexed from 1.")

    parameter_names = [str(parameter_row["excel_name"]) for parameter_row in parameter_rows]
    _validate_unique_names(parameter_names, "parameter excel_name")

    for parameter_row in parameter_rows:
        for required_key in ("category", "symbol", "excel_name", "description", "value", "unit"):
            if required_key not in parameter_row:
                raise KeyError(f"asm2d_tsn_simulation parameter row missing '{required_key}'.")
        float(parameter_row["value"])

    return workbook_config


def _validate_runtime_structure(
    model_params: Mapping[str, Any],
    *,
    measured_output_columns: list[str] | None,
) -> dict[str, Any]:
    workbook_config = _validate_workbook_config(model_params)
    solver = _validate_solver_config(model_params)
    state_columns = list(workbook_config["state_columns"])
    if measured_output_columns is None:
        raise ValueError(
            "asm2d_tsn_simulation requires measured_output_columns derived from workbook composition_matrix."
        )
    measured_output_columns = [str(name) for name in measured_output_columns]
    process_names = [str(process["name"]) for process in workbook_config["processes"]]
    process_types = list(model_params["process_types"])
    operational_columns = list(model_params["operational_columns"])
    influent_state_ranges = dict(model_params["influent_state_ranges"])
    operational_ranges = dict(model_params["operational_ranges"])

    _validate_unique_names(measured_output_columns, "measured_output_columns")
    _validate_unique_names(process_names, "process names")
    _validate_unique_names(operational_columns, "operational_columns")

    if len(process_types) != len(process_names):
        raise ValueError("asm2d_tsn_simulation process_types must align with the configured process list.")

    missing_state_ranges = [state_name for state_name in state_columns if state_name not in influent_state_ranges]
    if missing_state_ranges:
        missing_display = ", ".join(missing_state_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing influent_state_ranges for: {missing_display}")

    missing_operational_ranges = [name for name in operational_columns if name not in operational_ranges]
    if missing_operational_ranges:
        missing_display = ", ".join(missing_operational_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing operational_ranges for: {missing_display}")

    return {
        "workbook_config": workbook_config,
        "solver": solver,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "process_names": process_names,
        "process_types": process_types,
        "operational_columns": operational_columns,
        "influent_state_ranges": influent_state_ranges,
        "operational_ranges": operational_ranges,
    }


def _validate_solver_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "solver" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a solver section.")

    solver = dict(model_params["solver"])
    required_keys = (
        "lower_bound",
        "upper_bound",
        "initial_guess_floor",
        "warm_start_previous_weight",
        "warm_start_influent_weight",
        "initial_s_a_fraction",
        "initial_s_f_fraction",
        "initial_heterotroph_to_xs_ratio",
        "initial_pao_to_pp_ratio",
        "initial_aob_to_nh4_ratio",
        "initial_nob_to_no2_ratio",
        "multistart_s_a_fraction",
        "multistart_s_f_fraction",
        "multistart_heterotroph_to_xs_ratio",
        "multistart_pao_to_pp_ratio",
        "multistart_aob_to_nh4_ratio",
        "multistart_nob_to_no2_ratio",
        "dynamic_relaxation_days",
        "dynamic_absolute_tolerance",
        "dynamic_relative_tolerance",
        "dynamic_max_step",
        "residual_tolerance",
        "variable_tolerance",
        "gradient_tolerance",
        "acceptance_residual_max",
        "max_nfev",
    )

    for key in required_keys:
        if key not in solver:
            raise KeyError(f"asm2d_tsn_simulation solver missing '{key}'.")
        float(solver[key])

    lower_bound = float(solver["lower_bound"])
    upper_bound = float(solver["upper_bound"])
    initial_guess_floor = float(solver["initial_guess_floor"])
    if lower_bound < 0.0:
        raise ValueError("asm2d_tsn_simulation solver lower_bound must be non-negative.")
    if upper_bound <= lower_bound:
        raise ValueError("asm2d_tsn_simulation solver upper_bound must exceed lower_bound.")
    if not (lower_bound <= initial_guess_floor <= upper_bound):
        raise ValueError(
            "asm2d_tsn_simulation solver initial_guess_floor must lie between lower_bound and upper_bound."
        )

    previous_weight = float(solver["warm_start_previous_weight"])
    influent_weight = float(solver["warm_start_influent_weight"])
    if previous_weight < 0.0 or influent_weight < 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must be non-negative.")
    if previous_weight + influent_weight <= 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must sum to a positive value.")

    return solver


def _validate_unique_names(names: list[str], name_type: str) -> None:
    if not names:
        raise ValueError(f"{name_type} must not be empty.")

    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        duplicate_display = ", ".join(duplicates)
        raise ValueError(f"asm2d_tsn_simulation {name_type} contains duplicates: {duplicate_display}")


def _build_parameter_reference_map(parameter_rows: list[Mapping[str, Any]]) -> dict[str, str]:
    value_column_letter = get_column_letter(PARAMETER_VALUE_COLUMN_INDEX)
    parameter_refs: dict[str, str] = {}

    for row_number, parameter_row in enumerate(parameter_rows, start=2):
        excel_name = str(parameter_row["excel_name"])
        parameter_refs[excel_name] = f"'{PARAMETER_SHEET_NAME}'!${value_column_letter}${row_number}"

    return parameter_refs


def _build_parameter_value_map(parameter_rows: list[Mapping[str, Any]]) -> dict[str, float]:
    return {str(parameter_row["excel_name"]): float(parameter_row["value"]) for parameter_row in parameter_rows}


def _evaluate_numeric_expression(expression: str | float | int, parameter_values: Mapping[str, float]) -> float:
    if isinstance(expression, (int, float)):
        return float(expression)

    formatted_expression = str(expression).format_map(parameter_values)
    return float(eval(formatted_expression, {"__builtins__": {}}, {}))


def _build_state_index(state_columns: list[str]) -> dict[str, int]:
    return {name: position for position, name in enumerate(state_columns)}


def _generate_lhs_candidate_pool(
    seed: int,
    n_points: int,
    ordered_names: list[str],
    ranges: Mapping[str, Any],
) -> np.ndarray:
    """Generate n_points Latin Hypercube samples scaled to the configured parameter ranges.

    For degenerate ranges where lower == upper the returned column is the constant
    lower-bound value.  An empty array is returned when n_points or the number of
    dimensions is zero.
    """
    from scipy.stats.qmc import LatinHypercube, scale as qmc_scale

    n_dims = len(ordered_names)
    if n_points == 0 or n_dims == 0:
        return np.zeros((n_points, n_dims), dtype=float)

    lower_bounds = np.array([float(ranges[name][0]) for name in ordered_names], dtype=float)
    upper_bounds = np.array([float(ranges[name][1]) for name in ordered_names], dtype=float)

    degenerate_mask = lower_bounds == upper_bounds
    active_indices = np.where(~degenerate_mask)[0]

    result = np.empty((n_points, n_dims), dtype=float)
    # Fill constant values for degenerate dimensions
    for idx in np.where(degenerate_mask)[0]:
        result[:, idx] = lower_bounds[idx]

    if len(active_indices) == 0:
        return result

    sampler = LatinHypercube(d=len(active_indices), seed=seed)
    unit_samples = sampler.random(n=n_points)
    scaled = qmc_scale(unit_samples, lower_bounds[active_indices], upper_bounds[active_indices])
    result[:, active_indices] = scaled
    return result


def _monod(numerator: float, half_saturation: float) -> float:
    return float(numerator) / max(float(numerator) + float(half_saturation), 1e-9)


def _ratio(numerator: float, denominator: float) -> float:
    return float(numerator) / max(float(denominator), 1e-9)


def _share(numerator: float, denominator_a: float, denominator_b: float) -> float:
    return float(numerator) / max(float(denominator_a) + float(denominator_b), 1e-9)


def _build_influent_state_sample(
    sampled_state: np.ndarray,
) -> np.ndarray:
    return np.maximum(sampled_state.copy(), 0.0)


def _build_initial_guess(
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    state_index: Mapping[str, int],
    model_params: Mapping[str, Any],
    previous_solution: np.ndarray | None = None,
) -> np.ndarray:
    solver = model_params["solver"]
    aeration_model = model_params["aeration_model"]
    lower_floor = float(solver["initial_guess_floor"])
    upper_bound = float(solver["upper_bound"])
    guess = np.clip(influent_state.copy(), lower_floor, upper_bound)

    if previous_solution is not None:
        previous_weight = float(solver["warm_start_previous_weight"])
        influent_weight = float(solver["warm_start_influent_weight"])
        weight_total = max(previous_weight + influent_weight, 1e-9)
        guess = np.clip(
            (previous_weight * previous_solution + influent_weight * guess) / weight_total,
            lower_floor,
            upper_bound,
        )

    guess[state_index["S_A"]] *= float(solver["initial_s_a_fraction"])
    guess[state_index["S_F"]] *= float(solver["initial_s_f_fraction"])
    guess[state_index["X_H"]] = max(
        guess[state_index["X_H"]],
        guess[state_index["X_S"]] * float(solver["initial_heterotroph_to_xs_ratio"]),
    )
    guess[state_index["X_PAO"]] = max(
        guess[state_index["X_PAO"]],
        guess[state_index["X_PP"]] * float(solver["initial_pao_to_pp_ratio"]),
    )
    guess[state_index["X_AOB"]] = max(
        guess[state_index["X_AOB"]],
        guess[state_index["S_NH4"]] * float(solver["initial_aob_to_nh4_ratio"]),
    )
    guess[state_index["X_NOB"]] = max(
        guess[state_index["X_NOB"]],
        guess[state_index["S_NO2"]] * float(solver["initial_nob_to_no2_ratio"]),
    )

    dilution_rate = 24.0 / max(float(hrt_hours), 1e-6)
    kla = float(aeration_model["kla_base"]) + float(aeration_model["kla_per_aeration"]) * max(float(aeration), 0.0)
    do_saturation = float(aeration_model["do_saturation"])
    guess[state_index["S_O"]] = np.clip(
        (dilution_rate * guess[state_index["S_O"]] + kla * do_saturation) / max(dilution_rate + kla, 1e-9),
        lower_floor,
        do_saturation,
    )

    return guess


def _build_multistart_guess(
    initial_guess: np.ndarray,
    influent_state: np.ndarray,
    state_index: Mapping[str, int],
    model_params: Mapping[str, Any],
) -> np.ndarray:
    solver = model_params["solver"]
    multistart_guess = initial_guess.copy()
    multistart_guess[state_index["S_A"]] *= float(solver["multistart_s_a_fraction"])
    multistart_guess[state_index["S_F"]] *= float(solver["multistart_s_f_fraction"])
    multistart_guess[state_index["X_H"]] = max(
        multistart_guess[state_index["X_H"]],
        influent_state[state_index["X_S"]] * float(solver["multistart_heterotroph_to_xs_ratio"]),
    )
    multistart_guess[state_index["X_PAO"]] = max(
        multistart_guess[state_index["X_PAO"]],
        influent_state[state_index["X_PP"]] * float(solver["multistart_pao_to_pp_ratio"]),
    )
    multistart_guess[state_index["X_AOB"]] = max(
        multistart_guess[state_index["X_AOB"]],
        influent_state[state_index["S_NH4"]] * float(solver["multistart_aob_to_nh4_ratio"]),
    )
    multistart_guess[state_index["X_NOB"]] = max(
        multistart_guess[state_index["X_NOB"]],
        influent_state[state_index["S_NO2"]] * float(solver["multistart_nob_to_no2_ratio"]),
    )
    return np.clip(
        multistart_guess,
        float(solver["lower_bound"]),
        float(solver["upper_bound"]),
    )


def _compute_process_rates(
    state: np.ndarray,
    model_params: Mapping[str, Any],
    state_index: Mapping[str, int],
    parameter_values: Mapping[str, float],
) -> np.ndarray:
    s_a = state[state_index["S_A"]]
    s_f = state[state_index["S_F"]]
    s_nh4 = state[state_index["S_NH4"]]
    s_no2 = state[state_index["S_NO2"]]
    s_no3 = state[state_index["S_NO3"]]
    s_po4 = state[state_index["S_PO4"]]
    s_nox = s_no2 + s_no3
    s_alk = state[state_index["S_ALK"]]
    s_o = state[state_index["S_O"]]
    x_s = state[state_index["X_S"]]
    x_h = state[state_index["X_H"]]
    x_pao = state[state_index["X_PAO"]]
    x_pp = state[state_index["X_PP"]]
    x_pha = state[state_index["X_PHA"]]
    x_aob = state[state_index["X_AOB"]]
    x_nob = state[state_index["X_NOB"]]
    x_meoh = state[state_index["X_MeOH"]]
    x_mep = state[state_index["X_MeP"]]

    xs_to_xh = _ratio(x_s, x_h)
    hydrolysis_availability = _monod(xs_to_xh, parameter_values["K_X"])
    nitrate_share = _share(s_no3, s_no3, s_no2)
    nitrite_share = _share(s_no2, s_no3, s_no2)

    oxygen_hyd = _monod(s_o, parameter_values["K_O_hyd"])
    oxygen_hyd_limitation = parameter_values["K_O_hyd"] / max(parameter_values["K_O_hyd"] + s_o, 1e-9)
    oxygen_h = _monod(s_o, parameter_values["K_O_H"])
    oxygen_h_limitation = parameter_values["K_O_H"] / max(parameter_values["K_O_H"] + s_o, 1e-9)
    oxygen_pao = _monod(s_o, parameter_values["K_O_PAO"])
    oxygen_pao_limitation = parameter_values["K_O_PAO"] / max(parameter_values["K_O_PAO"] + s_o, 1e-9)

    alk_h = _monod(s_alk, parameter_values["K_ALK_H"])
    alk_pao = _monod(s_alk, parameter_values["K_ALK_PAO"])
    alk_nit = _monod(s_alk, parameter_values["K_ALK_nit"])
    alk_chem = _monod(s_alk, parameter_values["K_ALK_chem"])

    ammonium_h = _monod(s_nh4, parameter_values["K_NH4_H"])
    phosphate_h = _monod(s_po4, parameter_values["K_PO4_H"])
    ammonium_pao = _monod(s_nh4, parameter_values["K_NH4_PAO"])
    phosphate_pao = _monod(s_po4, parameter_values["K_PO4_PAO"])
    phosphate_storage = _monod(s_po4, parameter_values["K_PS"])
    phosphate_nit = _monod(s_po4, parameter_values["K_PO4_nit"])

    substrate_f = _monod(s_f, parameter_values["K_F"])
    substrate_fe = _monod(s_f, parameter_values["K_fe"])
    substrate_a = _monod(s_a, parameter_values["K_A"])

    pao_ratio_pp = _ratio(x_pp, x_pao)
    pao_ratio_pha = _ratio(x_pha, x_pao)
    pp_capacity = max(parameter_values["K_MAX"] - pao_ratio_pp, 0.0)
    r_pp = _monod(pao_ratio_pp, parameter_values["K_PP"])
    r_pha = _monod(pao_ratio_pha, parameter_values["K_PHA"])
    c_pp = pp_capacity / max(parameter_values["K_IPP"] + pp_capacity, 1e-9)

    process_rates = np.array(
        [
            parameter_values["K_H"] * oxygen_hyd * hydrolysis_availability * x_h,
            parameter_values["eta_hyd_NO2"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * _monod(s_no2, parameter_values["K_NO2_hyd"])
            * nitrite_share
            * hydrolysis_availability
            * x_h,
            parameter_values["eta_hyd_NO3"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * _monod(s_no3, parameter_values["K_NO3_hyd"])
            * nitrate_share
            * hydrolysis_availability
            * x_h,
            parameter_values["eta_hyd_fe"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * (parameter_values["K_NOX_hyd"] / max(parameter_values["K_NOX_hyd"] + s_nox, 1e-9))
            * hydrolysis_availability
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_H"])
            * nitrate_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_H"])
            * nitrite_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_H"])
            * nitrate_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_H"])
            * nitrite_share
            * x_h,
            parameter_values["q_Fe"]
            * parameter_values["mu_H"]
            * oxygen_h_limitation
            * (parameter_values["K_NOX_H"] / max(parameter_values["K_NOX_H"] + s_nox, 1e-9))
            * substrate_fe
            * alk_h
            * x_h,
            parameter_values["b_H"] * x_h,
            parameter_values["q_PHA"] * substrate_a * alk_pao * r_pp * x_pao,
            parameter_values["q_PP"] * oxygen_pao * phosphate_storage * alk_pao * r_pha * c_pp * x_pao,
            parameter_values["q_PP"]
            * oxygen_pao_limitation
            * phosphate_storage
            * alk_pao
            * r_pha
            * c_pp
            * parameter_values["eta_PAO_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_PAO"])
            * nitrate_share
            * x_pao,
            parameter_values["q_PP"]
            * oxygen_pao_limitation
            * phosphate_storage
            * alk_pao
            * r_pha
            * c_pp
            * parameter_values["eta_PAO_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_PAO"])
            * nitrite_share
            * x_pao,
            parameter_values["mu_PAO"] * oxygen_pao * ammonium_pao * phosphate_pao * r_pha * alk_pao * x_pao,
            parameter_values["mu_PAO"]
            * oxygen_pao_limitation
            * ammonium_pao
            * phosphate_pao
            * r_pha
            * alk_pao
            * parameter_values["eta_PAO_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_PAO"])
            * nitrate_share
            * x_pao,
            parameter_values["mu_PAO"]
            * oxygen_pao_limitation
            * ammonium_pao
            * phosphate_pao
            * r_pha
            * alk_pao
            * parameter_values["eta_PAO_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_PAO"])
            * nitrite_share
            * x_pao,
            parameter_values["b_PAO"] * x_pao * alk_pao,
            parameter_values["b_PP"] * x_pp * alk_pao,
            parameter_values["b_PHA"] * x_pha * alk_pao,
            parameter_values["mu_AOB"]
            * _monod(s_o, parameter_values["K_O_AOB"])
            * _monod(s_nh4, parameter_values["K_NH4_AOB"])
            * phosphate_nit
            * alk_nit
            * x_aob,
            parameter_values["mu_NOB"]
            * _monod(s_o, parameter_values["K_O_NOB"])
            * _monod(s_no2, parameter_values["K_NO2_NOB"])
            * phosphate_nit
            * alk_nit
            * x_nob,
            parameter_values["b_AOB"] * x_aob,
            parameter_values["b_NOB"] * x_nob,
            parameter_values["k_PRE"] * s_po4 * x_meoh,
            parameter_values["k_RED"] * x_mep * alk_chem,
        ],
        dtype=float,
    )
    if not np.all(np.isfinite(process_rates)):
        raise FloatingPointError("ASM2d-TSN process rates must remain finite.")
    if np.any(process_rates < 0.0):
        raise FloatingPointError("ASM2d-TSN process rates must remain non-negative.")
    return process_rates


def _compute_aeration_flux(state: np.ndarray, aeration: float, state_index: Mapping[str, int], model_params: Mapping[str, Any]) -> float:
    aeration_model = model_params["aeration_model"]
    kla = float(aeration_model["kla_base"]) + float(aeration_model["kla_per_aeration"]) * max(float(aeration), 0.0)
    do_saturation = float(aeration_model["do_saturation"])
    return kla * (do_saturation - state[state_index["S_O"]])


def _steady_state_residuals(
    state: np.ndarray,
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    matrix_bundle: Mapping[str, Any],
    model_params: Mapping[str, Any],
    parameter_values: Mapping[str, float],
) -> np.ndarray:
    dilution_rate = 24.0 / max(float(hrt_hours), 1e-6)
    state_index = dict(matrix_bundle["state_index"])
    residual = dilution_rate * (influent_state - state)
    process_rates = _compute_process_rates(state, model_params, state_index, parameter_values)
    residual += process_rates @ np.asarray(matrix_bundle["petersen_matrix"], dtype=float)
    residual[state_index["S_O"]] += _compute_aeration_flux(state, aeration, state_index, model_params)
    return residual


def simulate_asm2d_tsn_steady_state(
    *,
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    model_params: Mapping[str, Any],
    matrix_bundle: Mapping[str, Any] | None = None,
    previous_solution: np.ndarray | None = None,
    enforce_acceptance: bool = True,
) -> tuple[np.ndarray, dict[str, float | bool | int]]:
    """Solve a single mechanistic steady-state ASM2D-TSN operating point."""

    matrix_bundle = matrix_bundle if matrix_bundle is not None else get_asm2d_tsn_matrices(model_params)
    resolved_measured_output_columns = (
        list(matrix_bundle["measured_output_columns"])
        if "measured_output_columns" in matrix_bundle
        else None
    )
    runtime = _validate_runtime_structure(
        model_params,
        measured_output_columns=resolved_measured_output_columns,
    )
    parameter_values = _build_parameter_value_map(runtime["workbook_config"]["parameters"])
    state_columns = list(runtime["state_columns"])
    state_index = dict(matrix_bundle["state_index"])
    solver = runtime["solver"]
    lower_bounds = np.full(len(state_columns), float(solver["lower_bound"]), dtype=float)
    upper_bounds = np.full(len(state_columns), float(solver["upper_bound"]), dtype=float)
    initial_guess = _build_initial_guess(
        influent_state,
        hrt_hours,
        aeration,
        state_index,
        model_params,
        previous_solution=previous_solution,
    )
    candidate_guesses = [initial_guess, _build_multistart_guess(initial_guess, influent_state, state_index, model_params)]
    candidate_labels = ["initial", "multistart"]

    best_result = None
    best_residual_max = np.inf
    best_result_label = "initial"
    candidate_residuals: dict[str, float] = {}
    candidate_diagnostics: dict[str, dict[str, float | bool | int]] = {}
    for candidate_label, candidate_guess in zip(candidate_labels, candidate_guesses, strict=True):
        result = least_squares(
            _steady_state_residuals,
            candidate_guess,
            bounds=(lower_bounds, upper_bounds),
            xtol=float(solver["variable_tolerance"]),
            ftol=float(solver["residual_tolerance"]),
            gtol=float(solver["gradient_tolerance"]),
            max_nfev=int(solver["max_nfev"]),
            args=(influent_state, hrt_hours, aeration, matrix_bundle, model_params, parameter_values),
        )
        residual_max = float(np.max(np.abs(result.fun)))
        candidate_residuals[candidate_label] = residual_max
        candidate_diagnostics[candidate_label] = {
            "success": bool(result.success),
            "status": int(result.status),
            "nfev": int(result.nfev),
            "residual_l2": float(np.linalg.norm(result.fun)),
            "residual_max": residual_max,
            "cost": float(result.cost),
            "optimality": float(result.optimality),
        }
        if residual_max < best_residual_max:
            best_result = result
            best_residual_max = residual_max
            best_result_label = candidate_label

    dynamic_relaxation_used = best_residual_max > float(solver["acceptance_residual_max"])
    dynamic_relaxation_improved = False
    dynamic_diagnostics: dict[str, Any] = {
        "attempted": dynamic_relaxation_used,
        "success": False,
        "status": -1,
        "nfev": 0,
        "njev": 0,
        "nlu": 0,
        "message": "not_attempted",
        "refit_success": False,
        "refit_status": -1,
        "refit_nfev": 0,
        "refit_residual_l2": np.nan,
        "refit_residual_max": np.nan,
    }
    if best_residual_max > float(solver["acceptance_residual_max"]):
        dynamic_result = solve_ivp(
            lambda _time, values: _steady_state_residuals(
                values,
                influent_state,
                hrt_hours,
                aeration,
                matrix_bundle,
                model_params,
                parameter_values,
            ),
            (0.0, float(solver["dynamic_relaxation_days"])),
            np.clip(candidate_guesses[-1], lower_bounds, upper_bounds),
            method="BDF",
            atol=float(solver["dynamic_absolute_tolerance"]),
            rtol=float(solver["dynamic_relative_tolerance"]),
            max_step=float(solver["dynamic_max_step"]),
        )
        dynamic_diagnostics.update(
            success=bool(dynamic_result.success),
            status=int(dynamic_result.status),
            nfev=int(dynamic_result.nfev),
            njev=int(getattr(dynamic_result, "njev", 0) or 0),
            nlu=int(getattr(dynamic_result, "nlu", 0) or 0),
            message=str(dynamic_result.message),
        )
        if dynamic_result.success:
            relaxed_guess = np.clip(dynamic_result.y[:, -1], lower_bounds, upper_bounds)
            result = least_squares(
                _steady_state_residuals,
                relaxed_guess,
                bounds=(lower_bounds, upper_bounds),
                xtol=float(solver["variable_tolerance"]),
                ftol=float(solver["residual_tolerance"]),
                gtol=float(solver["gradient_tolerance"]),
                max_nfev=int(solver["max_nfev"]),
                args=(influent_state, hrt_hours, aeration, matrix_bundle, model_params, parameter_values),
            )
            residual_max = float(np.max(np.abs(result.fun)))
            dynamic_diagnostics.update(
                refit_success=bool(result.success),
                refit_status=int(result.status),
                refit_nfev=int(result.nfev),
                refit_residual_l2=float(np.linalg.norm(result.fun)),
                refit_residual_max=residual_max,
            )
            if residual_max < best_residual_max:
                best_result = result
                best_residual_max = residual_max
                best_result_label = "dynamic_relaxation"
                dynamic_relaxation_improved = True

    assert best_result is not None
    result = best_result
    accepted = bool(result.success and best_residual_max <= float(solver["acceptance_residual_max"]))
    diagnostics: dict[str, float | bool | int] = {
        "success": bool(result.success),
        "accepted": accepted,
        "status": int(result.status),
        "nfev": int(result.nfev),
        "residual_l2": float(np.linalg.norm(result.fun)),
        "residual_max": best_residual_max,
        "acceptance_threshold": float(solver["acceptance_residual_max"]),
        **{f"initial_{key}": value for key, value in candidate_diagnostics["initial"].items()},
        **{f"multistart_{key}": value for key, value in candidate_diagnostics["multistart"].items()},
        **{f"bdf_{key}": value for key, value in dynamic_diagnostics.items()},
        "selected_strategy": best_result_label,
        "dynamic_relaxation_used": dynamic_relaxation_used,
        "dynamic_relaxation_improved": dynamic_relaxation_improved,
    }

    if enforce_acceptance and (not accepted):
        raise RuntimeError(
            "asm2d_tsn_simulation steady-state solve failed: "
            f"success={result.success}, status={result.status}, residual_max={diagnostics['residual_max']:.3e}"
        )

    return result.x, diagnostics


def _compute_measured_output_values(state: np.ndarray, matrix_bundle: Mapping[str, Any]) -> np.ndarray:
    return np.asarray(matrix_bundle["composition_matrix"], dtype=float) @ state


def _generate_asm2d_tsn_dataset_chunk(
    *,
    chunk_start: int,
    chunk_size: int,
    chunk_seed: int,
    model_params: Mapping[str, Any],
    matrix_bundle: Mapping[str, Any],
    runtime: Mapping[str, Any],
    collect_debug_data: bool = False,
    progress_bar=None,
) -> tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[dict[str, Any]]]:
    configured_hyperparameters = model_params["hyperparameters"]
    max_sample_attempts = int(configured_hyperparameters["max_sample_attempts"])
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])
    state_index = dict(matrix_bundle["state_index"])
    parameter_values = _build_parameter_value_map(runtime["workbook_config"]["parameters"])

    # Pre-generate a joint LHS candidate pool covering all retry slots.
    # Influent-state and operational columns are combined into a single LHS design
    # so their stratification is jointly controlled.  Sequential consumption means
    # each retry draws the next LHS point rather than an independent uniform draw.
    total_candidates = chunk_size * max_sample_attempts
    all_columns = state_columns + operational_columns
    all_ranges = {**runtime["influent_state_ranges"], **runtime["operational_ranges"]}
    candidate_pool = _generate_lhs_candidate_pool(chunk_seed, total_candidates, all_columns, all_ranges)
    n_state = len(state_columns)

    influent_states = np.zeros((chunk_size, len(state_columns)), dtype=float)
    operational = np.zeros((chunk_size, len(operational_columns)), dtype=float)
    effluent_states = np.zeros((chunk_size, len(state_columns)), dtype=float)
    measured_outputs = np.zeros((chunk_size, len(measured_output_columns)), dtype=float)
    solver_diagnostics: list[dict[str, Any]] = []

    previous_solution: np.ndarray | None = None
    for local_index in range(chunk_size):
        last_error: RuntimeError | None = None
        for attempt_index in range(max_sample_attempts):
            candidate_index = local_index * max_sample_attempts + attempt_index
            sampled_state = candidate_pool[candidate_index, :n_state]
            candidate_influent = _build_influent_state_sample(sampled_state)
            candidate_operational = candidate_pool[candidate_index, n_state:]

            try:
                effluent_state, diagnostics = simulate_asm2d_tsn_steady_state(
                    influent_state=candidate_influent,
                    hrt_hours=float(candidate_operational[0]),
                    aeration=float(candidate_operational[1]),
                    model_params=model_params,
                    matrix_bundle=matrix_bundle,
                    previous_solution=previous_solution,
                )
            except RuntimeError as error:
                last_error = error
                continue

            if not diagnostics["accepted"]:
                last_error = RuntimeError(
                    "asm2d_tsn_simulation steady-state solve did not satisfy the configured acceptance threshold."
                )
                continue

            if not np.all(np.isfinite(effluent_state)):
                last_error = RuntimeError("asm2d_tsn_simulation produced non-finite effluent states.")
                continue

            previous_solution = effluent_state
            influent_states[local_index] = candidate_influent
            operational[local_index] = candidate_operational
            effluent_states[local_index] = effluent_state
            measured_outputs[local_index] = _compute_measured_output_values(effluent_state, matrix_bundle)
            if collect_debug_data:
                diagnostic_record = dict(diagnostics)
                diagnostic_record["sample_index"] = chunk_start + local_index
                diagnostic_record["attempt_count"] = attempt_index + 1
                diagnostic_record["HRT"] = float(candidate_operational[0])
                diagnostic_record["Aeration"] = float(candidate_operational[1])
                solver_diagnostics.append(diagnostic_record)
            if progress_bar is not None:
                progress_bar.update(1)
            break
        else:
            raise RuntimeError(
                "asm2d_tsn_simulation failed to generate a valid sample after "
                f"{max_sample_attempts} attempts at sample index {chunk_start + local_index}."
            ) from last_error

    return chunk_start, influent_states, operational, effluent_states, measured_outputs, solver_diagnostics


def _summarize_asm2d_tsn_solver_diagnostics(
    solver_diagnostics: pd.DataFrame,
    acceptance_threshold: float,
) -> dict[str, Any]:
    if solver_diagnostics.empty:
        return {
            "sample_count": 0,
            "accepted_count": 0,
            "accepted_rate": 0.0,
            "acceptance_threshold": acceptance_threshold,
            "multistart_selected_rate": 0.0,
            "dynamic_relaxation_used_rate": 0.0,
            "dynamic_relaxation_improved_rate": 0.0,
            "residual_max_quantiles": {},
            "nfev_quantiles": {},
            "selected_strategy_counts": {},
        }

    residual_max_quantiles = {
        "q50": float(solver_diagnostics["residual_max"].quantile(0.50)),
        "q90": float(solver_diagnostics["residual_max"].quantile(0.90)),
        "q95": float(solver_diagnostics["residual_max"].quantile(0.95)),
        "q99": float(solver_diagnostics["residual_max"].quantile(0.99)),
        "max": float(solver_diagnostics["residual_max"].max()),
    }
    nfev_quantiles = {
        "q50": float(solver_diagnostics["nfev"].quantile(0.50)),
        "q90": float(solver_diagnostics["nfev"].quantile(0.90)),
        "q95": float(solver_diagnostics["nfev"].quantile(0.95)),
        "max": float(solver_diagnostics["nfev"].max()),
    }
    return {
        "sample_count": int(len(solver_diagnostics)),
        "accepted_count": int(solver_diagnostics["accepted"].sum()),
        "accepted_rate": float(solver_diagnostics["accepted"].mean()),
        "acceptance_threshold": float(acceptance_threshold),
        "multistart_selected_rate": float((solver_diagnostics["selected_strategy"] == "multistart").mean()),
        "dynamic_relaxation_used_rate": float(solver_diagnostics["dynamic_relaxation_used"].mean()),
        "dynamic_relaxation_improved_rate": float(solver_diagnostics["dynamic_relaxation_improved"].mean()),
        "mean_attempt_count": float(solver_diagnostics.get("attempt_count", pd.Series([1])).mean()),
        "max_attempt_count": int(solver_diagnostics.get("attempt_count", pd.Series([1])).max()),
        "residual_max_quantiles": residual_max_quantiles,
        "nfev_quantiles": nfev_quantiles,
        "selected_strategy_counts": {
            str(name): int(count)
            for name, count in solver_diagnostics["selected_strategy"].value_counts().items()
        },
    }


def _resolve_parallel_workers(requested_workers: int, sample_count: int) -> int:
    if requested_workers < 0:
        raise ValueError("parallel_workers must be greater than or equal to 0.")

    if sample_count <= 1:
        return 1

    available_workers = os.cpu_count() or 1
    if requested_workers == 0:
        requested_workers = max(available_workers - 1, 1)

    return min(max(requested_workers, 1), available_workers, sample_count)


def _resolve_parallel_chunk_size(requested_chunk_size: int) -> int:
    if requested_chunk_size < 1:
        raise ValueError("parallel_chunk_size must be at least 1.")

    return requested_chunk_size


def _write_parameter_sheet(worksheet, parameter_rows: list[Mapping[str, Any]]) -> None:
    worksheet.freeze_panes = "A2"
    headers = ["category", "symbol", "excel_name", "description", "value", "unit"]
    _write_header_row(worksheet, headers)

    previous_category = None
    for row_number, parameter_row in enumerate(parameter_rows, start=2):
        current_category = str(parameter_row["category"])
        row_values = [
            current_category,
            str(parameter_row["symbol"]),
            str(parameter_row["excel_name"]),
            str(parameter_row["description"]),
            float(parameter_row["value"]),
            str(parameter_row["unit"]),
        ]

        for column_number, value in enumerate(row_values, start=1):
            cell = worksheet.cell(row=row_number, column=column_number, value=value)
            if column_number == PARAMETER_VALUE_COLUMN_INDEX:
                cell.number_format = "0.###############"

        if current_category != previous_category:
            for column_number in range(1, len(headers) + 1):
                worksheet.cell(row=row_number, column=column_number).fill = SECTION_FILL
        previous_category = current_category


def _write_stoichiometric_sheet(
    worksheet,
    workbook_config: Mapping[str, Any],
    parameter_refs: Mapping[str, str],
) -> None:
    worksheet.freeze_panes = "C2"
    state_columns = list(workbook_config["state_columns"])
    processes = list(workbook_config["processes"])
    headers = ["process_index", "process"] + state_columns
    _write_header_row(worksheet, headers)
    state_column_index = {state_name: position for position, state_name in enumerate(state_columns, start=3)}

    for row_number, process in enumerate(processes, start=2):
        worksheet.cell(row=row_number, column=1, value=int(process["index"]))
        worksheet.cell(row=row_number, column=2, value=str(process["name"]))
        direct_coefficients = STOICHIOMETRIC_COEFFICIENTS[row_number - 2]["coefficients"]

        for state_name in state_columns:
            column_number = state_column_index[state_name]
            cell = worksheet.cell(row=row_number, column=column_number)

            if state_name in direct_coefficients:
                cell.value = _format_formula(direct_coefficients[state_name], parameter_refs)
                continue

            if state_name == "S_NH4":
                cell.value = _build_weighted_formula(
                    row_number,
                    state_column_index,
                    NITROGEN_CONTINUITY_TERMS,
                    parameter_refs,
                    negate=True,
                )
                continue

            if state_name == "S_PO4":
                cell.value = _build_weighted_formula(
                    row_number,
                    state_column_index,
                    PHOSPHORUS_CONTINUITY_TERMS,
                    parameter_refs,
                    negate=True,
                )
                continue

            if state_name == "S_ALK":
                cell.value = _build_alkalinity_formula(row_number, state_column_index)
                continue

def _write_composition_sheet(
    worksheet,
    workbook_config: Mapping[str, Any],
    parameter_refs: Mapping[str, str],
) -> None:
    worksheet.freeze_panes = "D2"
    dissolved_state_columns = list(workbook_config["dissolved_state_columns"])
    particulate_state_columns = list(workbook_config["particulate_state_columns"])
    state_columns = list(workbook_config["state_columns"])
    legacy_composite_variables = workbook_config.get("composite_variables")
    composite_variables = _resolve_workbook_composite_variables(
        state_columns,
        legacy_composite_variables=(
            None if legacy_composite_variables is None else list(legacy_composite_variables)
        ),
    )
    state_units = dict(workbook_config["state_units"])
    headers = ["state_group", "state_variable", "unit"] + composite_variables
    _write_header_row(worksheet, headers)
    composite_column_index = {name: position for position, name in enumerate(composite_variables, start=4)}

    for row_number, state_name in enumerate(state_columns, start=2):
        state_group = "Dissolved" if state_name in dissolved_state_columns else "Particulate"
        worksheet.cell(row=row_number, column=1, value=state_group)
        worksheet.cell(row=row_number, column=2, value=state_name)
        worksheet.cell(row=row_number, column=3, value=state_units[state_name])

        for composite_name, expression in COMPOSITION_FORMULAS.get(state_name, {}).items():
            worksheet.cell(
                row=row_number,
                column=composite_column_index[composite_name],
                value=_format_formula(expression, parameter_refs),
            )


def _resolve_workbook_composite_variables(
    state_columns: list[str],
    *,
    legacy_composite_variables: list[str] | None,
) -> list[str]:
    composite_variables: list[str] = []
    for state_name in state_columns:
        for composite_name in COMPOSITION_FORMULAS.get(state_name, {}):
            normalized_name = str(composite_name)
            if normalized_name not in composite_variables:
                composite_variables.append(normalized_name)

    if not composite_variables:
        raise ValueError(
            "asm2d_tsn_simulation composition formulas do not define any composite output columns."
        )

    if legacy_composite_variables is not None:
        normalized_legacy = [str(name) for name in legacy_composite_variables]
        _validate_unique_names(normalized_legacy, "composite_variables")
        missing_formulas = sorted(name for name in normalized_legacy if name not in composite_variables)
        if missing_formulas:
            missing_display = ", ".join(missing_formulas)
            raise ValueError(
                "asm2d_tsn_simulation legacy workbook composite_variables contains outputs without "
                f"composition formulas: {missing_display}"
            )

    return composite_variables


def _write_header_row(worksheet, headers: list[str]) -> None:
    for column_number, header in enumerate(headers, start=1):
        cell = worksheet.cell(row=1, column=column_number, value=header)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")


def _format_formula(expression: str | float | int, parameter_refs: Mapping[str, str]) -> str:
    if isinstance(expression, (int, float)):
        return f"={expression:g}"

    formatted_expression = str(expression).format_map(parameter_refs)
    if formatted_expression.startswith("="):
        return formatted_expression

    return f"={formatted_expression}"


def _build_weighted_formula(
    row_number: int,
    state_column_index: Mapping[str, int],
    factor_terms: Mapping[str, str],
    parameter_refs: Mapping[str, str],
    *,
    negate: bool,
) -> str:
    terms: list[str] = []
    for state_name, factor_expression in factor_terms.items():
        cell_reference = f"{get_column_letter(state_column_index[state_name])}{row_number}"
        formatted_factor = factor_expression.format_map(parameter_refs)
        if formatted_factor == "1":
            terms.append(cell_reference)
        else:
            terms.append(f"{cell_reference}*({formatted_factor})")

    if not terms:
        return "=0"

    expression = "+".join(terms)
    if negate:
        return f"=-({expression})"

    return f"={expression}"


def _build_alkalinity_formula(row_number: int, state_column_index: Mapping[str, int]) -> str:
    ammonium_ref = f"{get_column_letter(state_column_index['S_NH4'])}{row_number}"
    nitrite_ref = f"{get_column_letter(state_column_index['S_NO2'])}{row_number}"
    nitrate_ref = f"{get_column_letter(state_column_index['S_NO3'])}{row_number}"
    phosphate_ref = f"{get_column_letter(state_column_index['S_PO4'])}{row_number}"
    return f"={ammonium_ref}/14-{nitrite_ref}/14-{nitrate_ref}/14+{phosphate_ref}/31"


def _auto_size_columns(worksheet) -> None:
    for column_cells in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column_cells[0].column)
        for cell in column_cells:
            if cell.value is None:
                continue
            max_length = max(max_length, len(str(cell.value)))
        worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 48)


__all__ = [
    "build_asm2d_tsn_workbook",
    "build_asm2d_tsn_metadata",
    "create_asm2d_tsn_workbook",
    "generate_asm2d_tsn_dataset",
    "get_asm2d_tsn_matrices",
    "load_asm2d_tsn_workbook_composition",
    "load_asm2d_tsn_simulation_params",
    "resolve_asm2d_tsn_composition_cache_paths",
    "resolve_asm2d_tsn_simulation_artifact_paths",
    "resolve_asm2d_tsn_workbook_path",
    "run_asm2d_tsn_simulation",
    "simulate_asm2d_tsn_steady_state",
    "sweep_asm2d_tsn_operating_space",
]

In [ ]:
def evaluate_workbook_cell(workbook, sheet_name: str, coordinate: str, cache: dict[str, float], stack: set[str]) -> float:
    key = _normalize_excel_reference(sheet_name, coordinate)
    if key in cache:
        return cache[key]
    if key in stack:
        raise ValueError(f"Circular workbook formula reference at {key}.")
    stack.add(key)
    cell = workbook[sheet_name][coordinate]
    value = cell.value
    if value is None or value == "":
        result = 0.0
    elif isinstance(value, bool):
        result = float(value)
    elif isinstance(value, (int, float)):
        result = float(value)
    else:
        expression = str(value).strip()
        if not expression.startswith("="):
            result = float(expression)
        else:
            expression = expression[1:]

            def replace_reference(match: re.Match[str]) -> str:
                referenced_sheet = match.group("sheet_quoted") or match.group("sheet_unquoted") or sheet_name
                referenced_coordinate = f"{match.group('column').upper()}{match.group('row')}"
                resolved = evaluate_workbook_cell(
                    workbook, referenced_sheet, referenced_coordinate, cache, stack
                )
                return repr(float(resolved))

            resolved_expression = _EXCEL_REFERENCE_PATTERN.sub(replace_reference, expression).replace("^", "**")
            result = float(eval(resolved_expression, {"__builtins__": {}}, {}))
    stack.remove(key)
    cache[key] = result
    return result


def reevaluate_workbook_matrices(path: Path, state_columns: Sequence[str]) -> tuple[np.ndarray, np.ndarray, list[str], list[str], list[dict[str, Any]], dict[str, str]]:
    workbook = load_workbook(path, data_only=False, read_only=False)
    try:
        stoich_sheet = workbook["stoichiometric_matrix"]
        composition_sheet = workbook["composition_matrix"]
        parameter_sheet = workbook["parameter_table"]
        if parameter_sheet.max_row - 1 != 81 or parameter_sheet.max_column != 6:
            raise ValueError("The parameter table must contain exactly 81 rows and six declared fields.")
        parameter_headers = [str(parameter_sheet.cell(1, col).value).strip() for col in range(1, 7)]
        if parameter_headers != ["category", "symbol", "excel_name", "description", "value", "unit"]:
            raise ValueError("The parameter-table schema is invalid.")
        workbook_parameters = [
            {
                "category": str(parameter_sheet.cell(row, 1).value).strip(),
                "symbol": str(parameter_sheet.cell(row, 2).value).strip(),
                "excel_name": str(parameter_sheet.cell(row, 3).value).strip(),
                "description": str(parameter_sheet.cell(row, 4).value).strip(),
                "value": float(parameter_sheet.cell(row, 5).value),
                "unit": str(parameter_sheet.cell(row, 6).value).strip(),
            }
            for row in range(2, 83)
        ]
        if any(not np.isfinite(parameter["value"]) or not all(str(parameter[field]).strip() for field in ("category", "symbol", "excel_name", "description", "unit")) for parameter in workbook_parameters):
            raise ValueError("Workbook parameters must have finite values and complete metadata.")
        cache: dict[str, float] = {}
        stoich_headers = [str(stoich_sheet.cell(1, col).value).strip() for col in range(3, 23)]
        if stoich_headers != list(state_columns):
            raise ValueError("Workbook stoichiometric state order does not match the declared state order.")
        process_names = [str(stoich_sheet.cell(row, 2).value).strip() for row in range(2, 30)]
        stoich = np.asarray(
            [
                [evaluate_workbook_cell(workbook, stoich_sheet.title, stoich_sheet.cell(row, col).coordinate, cache, set())
                 for col in range(3, 23)]
                for row in range(2, 30)
            ],
            dtype=np.float64,
        )
        composition_headers = [str(composition_sheet.cell(1, col).value).strip() for col in range(4, 8)]
        composition_states = [str(composition_sheet.cell(row, 2).value).strip() for row in range(2, 22)]
        state_units = {
            str(composition_sheet.cell(row, 2).value).strip(): str(composition_sheet.cell(row, 3).value).strip()
            for row in range(2, 22)
        }
        if composition_states != list(state_columns):
            raise ValueError("Workbook composition state order does not match the declared state order.")
        composition = np.asarray(
            [
                [evaluate_workbook_cell(workbook, composition_sheet.title, composition_sheet.cell(row, col).coordinate, cache, set())
                 for row in range(2, 22)]
                for col in range(4, 8)
            ],
            dtype=np.float64,
        )
    finally:
        workbook.close()
    return stoich, composition, process_names, composition_headers, workbook_parameters, state_units


def read_cached_workbook_matrices(path: Path) -> tuple[np.ndarray, np.ndarray]:
    workbook = load_workbook(path, data_only=True, read_only=True)
    formula_workbook = load_workbook(path, data_only=False, read_only=True)
    try:
        stoich_sheet = workbook["stoichiometric_matrix"]
        composition_sheet = workbook["composition_matrix"]
        stoich_formula_sheet = formula_workbook["stoichiometric_matrix"]
        composition_formula_sheet = formula_workbook["composition_matrix"]
        for row in range(2, 30):
            for col in range(3, 23):
                formula = stoich_formula_sheet.cell(row, col).value
                cached = stoich_sheet.cell(row, col).value
                if isinstance(formula, str) and formula.startswith("=") and cached is None:
                    raise ValueError(f"Missing cached result for stoichiometric formula {stoich_sheet.cell(row, col).coordinate}.")
        for row in range(2, 22):
            for col in range(4, 8):
                formula = composition_formula_sheet.cell(row, col).value
                cached = composition_sheet.cell(row, col).value
                if isinstance(formula, str) and formula.startswith("=") and cached is None:
                    raise ValueError(f"Missing cached result for composition formula {composition_sheet.cell(row, col).coordinate}.")
        stoich = np.asarray(
            [[stoich_sheet.cell(row, col).value or 0.0 for col in range(3, 23)] for row in range(2, 30)],
            dtype=np.float64,
        )
        composition = np.asarray(
            [[composition_sheet.cell(row, col).value or 0.0 for row in range(2, 22)] for col in range(4, 8)],
            dtype=np.float64,
        )
    finally:
        workbook.close()
        formula_workbook.close()
    if not np.all(np.isfinite(stoich)) or not np.all(np.isfinite(composition)):
        raise ValueError("The workbook must contain finite cached results for every supported formula.")
    return stoich, composition


MATRIX_BUNDLE = get_asm2d_tsn_matrices(
    SIM_PARAMS,
    repo_root=ROOT,
    paths_config=load_paths_config(ROOT),
    use_composition_cache=False,
)

In [ ]:
STATE_COLUMNS = list(MATRIX_BUNDLE["state_columns"])
OPERATIONAL_COLUMNS = list(SIM_PARAMS["operational_columns"])
MEASURED_COLUMNS = list(MATRIX_BUNDLE["measured_output_columns"])
PETERSEN = np.asarray(MATRIX_BUNDLE["petersen_matrix"], dtype=np.float64)
COMPOSITION = np.asarray(MATRIX_BUNDLE["composition_matrix"], dtype=np.float64)
workbook_petersen, workbook_composition, workbook_processes, workbook_composites, workbook_parameters, workbook_state_units = reevaluate_workbook_matrices(
    WORKBOOK_PATH, STATE_COLUMNS
)
cached_petersen, cached_composition = read_cached_workbook_matrices(WORKBOOK_PATH)
if PETERSEN.shape != (28, 20) or COMPOSITION.shape != (4, 20):
    raise ValueError("The mechanistic matrices must have shapes 28x20 and 4x20.")
if not np.allclose(PETERSEN, workbook_petersen, atol=1e-12, rtol=1e-12):
    raise ValueError("Re-evaluated workbook stoichiometry differs from the assembled Petersen matrix.")
if not np.allclose(COMPOSITION, workbook_composition, atol=1e-12, rtol=1e-12):
    raise ValueError("Re-evaluated workbook composition differs from the assembled composition matrix.")
if not np.allclose(cached_petersen, workbook_petersen, atol=1e-12, rtol=1e-12):
    raise ValueError("Cached workbook stoichiometry differs from the independently re-evaluated formulas.")
if not np.allclose(cached_composition, workbook_composition, atol=1e-12, rtol=1e-12):
    raise ValueError("Cached workbook composition differs from the independently re-evaluated formulas.")
if workbook_processes != list(MATRIX_BUNDLE["process_names"]) or workbook_composites != MEASURED_COLUMNS:
    raise ValueError("Workbook process or composite ordering differs from the declared contract.")
configured_parameters = ACTIVE["simulation"]["workbook"]["parameters"]
if len(configured_parameters) != len(workbook_parameters):
    raise ValueError("Workbook and configuration parameter counts differ.")
for index, (workbook_parameter, configured_parameter) in enumerate(zip(workbook_parameters, configured_parameters, strict=True)):
    for field in ("category", "symbol", "excel_name", "description", "unit"):
        if str(workbook_parameter[field]) != str(configured_parameter[field]):
            raise ValueError(f"Workbook parameter {index + 1} field {field} differs from the configuration.")
    if not np.isclose(float(workbook_parameter["value"]), float(configured_parameter["value"]), rtol=0.0, atol=0.0):
        raise ValueError(f"Workbook parameter {index + 1} value differs from the configuration.")
if workbook_state_units != ACTIVE["simulation"]["workbook"]["state_units"]:
    raise ValueError("Workbook state units differ from the configuration.")

singular_values = np.linalg.svd(PETERSEN, compute_uv=False)
rank_tolerance = PETERSEN.shape[0] * np.finfo(np.float64).eps * singular_values[0]
PETERSEN_RANK = int(np.sum(singular_values > rank_tolerance))
A_MATRIX = null_space(PETERSEN, rcond=rank_tolerance / singular_values[0]).T
Z_MATRIX = null_space(A_MATRIX)
if PETERSEN_RANK != 15 or A_MATRIX.shape != (5, 20) or Z_MATRIX.shape != (20, 15):
    raise ValueError("Expected rank 15 with five invariants and a 15-dimensional feasible direction space.")
invariant_error = float(np.max(np.abs(A_MATRIX @ PETERSEN.T)))
aeration_error = float(np.max(np.abs(A_MATRIX[:, STATE_COLUMNS.index("S_O")])))
if invariant_error > 1e-12:
    raise ValueError(f"Invariant construction failed: max |A nu^T|={invariant_error:.3e}.")
if aeration_error > 1e-12:
    raise ValueError(f"The invariant operator does not annihilate the aeration basis: {aeration_error:.3e}.")

matrix_path = atomic_npz(
    DIRS["matrices"] / "operators.npz",
    petersen=PETERSEN,
    composition=COMPOSITION,
    invariant=A_MATRIX,
    null_basis=Z_MATRIX,
    singular_values=singular_values,
)
matrix_summary_path = atomic_json(
    DIRS["matrices"] / "validation.json",
    {
        "petersen_shape": list(PETERSEN.shape),
        "composition_shape": list(COMPOSITION.shape),
        "rank": PETERSEN_RANK,
        "rank_tolerance": rank_tolerance,
        "max_abs_A_nu_transpose": invariant_error,
        "max_abs_A_aeration_basis": aeration_error,
        "cached_formula_crosscheck": True,
        "state_columns": STATE_COLUMNS,
        "process_names": workbook_processes,
        "composite_columns": MEASURED_COLUMNS,
        "workbook_sha256": WORKBOOK_HASH,
        "parameter_rows_validated": len(workbook_parameters),
        "state_units": workbook_state_units,
    },
)
register_artifact("operators", matrix_path)
register_artifact("matrix_validation", matrix_summary_path)
stage_complete("workbook_and_matrices", rank=PETERSEN_RANK, invariant_error=invariant_error)
print(f"Validated workbook; rank={PETERSEN_RANK}, max |A nu^T|={invariant_error:.3e}")


# Fixed midpoint kinetics fixture and an exercised 20-day BDF fallback.
fixture_state = np.asarray(
    [
        np.mean(ACTIVE["simulation"]["influent_state_ranges"][name])
        for name in STATE_COLUMNS
    ],
    dtype=np.float64,
)
fixture_runtime = _validate_runtime_structure(
    SIM_PARAMS,
    measured_output_columns=MEASURED_COLUMNS,
)
fixture_parameters = _build_parameter_value_map(fixture_runtime["workbook_config"]["parameters"])
fixture_rates = _compute_process_rates(
    fixture_state,
    SIM_PARAMS,
    MATRIX_BUNDLE["state_index"],
    fixture_parameters,
)
expected_fixture_rates = np.asarray(
    [
        92.69796111901375, 9.101254364412256, 28.764458238142446, 2.471945629840367,
        128.93214239104688, 52.085812791737965, 60.01205173110546, 18.988188243045084,
        24.243578317608947, 7.6708197020559545, 36.83730535582387, 23.0,
        144.17449029658616, 0.7426197345146636, 0.026884334631318736, 0.10936763362852317,
        9.873012234382074, 0.3574229883638925, 1.4540254381544506, 6.496178718400941,
        2.198706643151088, 3.098177542621987, 1.9064642665015905, 0.603245727879159,
        0.8500000000000001, 0.7225, 60.0, 3.5894428152492663,
    ],
    dtype=np.float64,
)
if not np.allclose(fixture_rates, expected_fixture_rates, rtol=1e-12, atol=1e-12):
    raise AssertionError("The fixed mechanistic rate fixture changed.")
fallback_params = copy.deepcopy(SIM_PARAMS)
fallback_params["solver"]["max_nfev"] = 1
fallback_params["solver"]["acceptance_residual_max"] = 0.0
_, fallback_diagnostics = simulate_asm2d_tsn_steady_state(
    influent_state=fixture_state,
    hrt_hours=21.0,
    aeration=1.5,
    model_params=fallback_params,
    matrix_bundle=MATRIX_BUNDLE,
    enforce_acceptance=False,
)
if not (
    fallback_diagnostics["dynamic_relaxation_used"]
    and fallback_diagnostics["bdf_attempted"]
    and fallback_diagnostics["bdf_success"]
    and int(fallback_diagnostics["bdf_nfev"]) > 0
):
    raise AssertionError("The declared BDF fallback path was not exercised successfully.")
stage_complete("mechanistic_tests", known_rate_fixture=True, bdf_fallback=True)

## Deterministic mechanistic datasets

Candidates are generated in indexed Latin-hypercube batches and assigned round-robin to fixed warm-start chains. Thread completion order cannot change accepted-row order. All attempted candidates, including rejected solves, are retained under the run root.


In [ ]:
def stable_seed(label: str) -> int:
    base = int(ACTIVE["simulation"]["sampling"]["seed"])
    digest = hashlib.sha256(f"{base}:{label}".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def make_candidate_batch(n_points: int, *, label: str, batch_index: int, regime: str | None = None) -> pd.DataFrame:
    all_columns = STATE_COLUMNS + OPERATIONAL_COLUMNS
    all_ranges = {
        **ACTIVE["simulation"]["influent_state_ranges"],
        **ACTIVE["simulation"]["operational_ranges"],
    }
    lower = np.asarray([all_ranges[name][0] for name in all_columns], dtype=np.float64)
    upper = np.asarray([all_ranges[name][1] for name in all_columns], dtype=np.float64)
    seed = int(ACTIVE["simulation"]["sampling"]["seed"]) if label == "id" and batch_index == 0 else stable_seed(f"{label}:batch:{batch_index}")
    samples = qmc_scale(LatinHypercube(d=len(all_columns), seed=seed).random(n_points), lower, upper)
    frame = pd.DataFrame(samples, columns=all_columns)
    frame["ood_regime"] = regime if regime is not None else "id"
    frame["perturbed_variables"] = ""
    if regime is not None:
        regime_config = ACTIVE["simulation"]["ood"]["regimes"][regime]
        ood_ranges = regime_config["ranges"]
        rng = np.random.default_rng(stable_seed(f"{label}:selection:{batch_index}"))
        one_fraction = float(ACTIVE["simulation"]["ood"]["candidate_mix"]["one_variable"])
        one_count = int(round(n_points * one_fraction))
        perturbation_counts = np.asarray([1] * one_count + [2] * (n_points - one_count), dtype=int)
        rng.shuffle(perturbation_counts)
        eligible = list(ood_ranges)
        for row_index, count in enumerate(perturbation_counts):
            selected = sorted(rng.choice(eligible, size=int(count), replace=False).tolist())
            frame.at[row_index, "perturbed_variables"] = ";".join(selected)
            for variable in selected:
                lo, hi = map(float, ood_ranges[variable])
                frame.at[row_index, variable] = rng.uniform(lo, hi)
    return frame


def solve_candidate_chain(
    chain_id: int,
    candidate_rows: list[tuple[int, dict[str, Any]]],
    previous_solution: np.ndarray | None,
) -> tuple[list[dict[str, Any]], np.ndarray | None]:
    records: list[dict[str, Any]] = []
    for candidate_id, row in candidate_rows:
        influent = np.asarray([row[name] for name in STATE_COLUMNS], dtype=np.float64)
        started = time.perf_counter()
        failure_reason = ""
        try:
            effluent, diagnostics = simulate_asm2d_tsn_steady_state(
                influent_state=influent,
                hrt_hours=float(row["HRT"]),
                aeration=float(row["Aeration"]),
                model_params=SIM_PARAMS,
                matrix_bundle=MATRIX_BUNDLE,
                previous_solution=previous_solution,
                enforce_acceptance=False,
            )
            finite = bool(np.all(np.isfinite(effluent)))
            accepted = bool(diagnostics["accepted"] and finite)
            if accepted:
                previous_solution = np.asarray(effluent, dtype=np.float64)
            elif not finite:
                failure_reason = "non_finite_effluent"
            else:
                failure_reason = (
                    f"solver_rejected: success={diagnostics['success']}, "
                    f"residual_max={float(diagnostics['residual_max']):.6e}"
                )
        except Exception as error:
            effluent = np.full(len(STATE_COLUMNS), np.nan, dtype=np.float64)
            diagnostics = {
                "success": False,
                "accepted": False,
                "status": -1,
                "nfev": -1,
                "residual_l2": np.nan,
                "residual_max": np.nan,
                "selected_strategy": "exception",
                "dynamic_relaxation_used": False,
                "dynamic_relaxation_improved": False,
            }
            accepted = False
            failure_reason = f"{type(error).__name__}: {error}"
        record = {
            "candidate_id": int(candidate_id),
            "chain_id": int(chain_id),
            "accepted": accepted,
            "failure_reason": failure_reason,
            "solve_seconds": time.perf_counter() - started,
            "ood_regime": row["ood_regime"],
            "perturbed_variables": row["perturbed_variables"],
            **{name: float(row[name]) for name in OPERATIONAL_COLUMNS},
            **{f"In_{name}": float(row[name]) for name in STATE_COLUMNS},
            **{f"Out_{name}": float(value) for name, value in zip(STATE_COLUMNS, effluent, strict=True)},
            **{f"solver_{key}": value for key, value in diagnostics.items()},
        }
        records.append(record)
    return records, previous_solution


def generate_dataset(target_count: int, *, label: str, regime: str | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    sampling = ACTIVE["simulation"]["sampling"]
    max_candidates = target_count * int(sampling["max_sample_attempts"])
    n_chains = int(sampling["n_chains"])
    workers = max(1, min(int(sampling["parallel_workers"]), n_chains))
    partial_attempts_path = DIRS["datasets"] / f"{label}.attempts.partial.parquet"
    partial_accepted_path = DIRS["datasets"] / f"{label}.accepted.partial.parquet"
    partial_state_path = DIRS["datasets"] / f"{label}.generation_state.json"

    accepted_records: list[dict[str, Any]] = []
    attempt_records: list[dict[str, Any]] = []
    next_candidate_id = 0
    batch_index = 0
    if bool(ACTIVE["run"].get("resume", True)) and partial_state_path.exists():
        state = read_json(partial_state_path)
        expected_state = {
            "label": label,
            "regime": regime,
            "target_count": target_count,
            "config_sha256": CONFIG_HASH,
        }
        if any(state.get(key) != value for key, value in expected_state.items()):
            raise RuntimeError(f"Partial {label} generation state does not match the active contract.")
        if partial_attempts_path.exists():
            partial_attempts = pd.read_parquet(partial_attempts_path).iloc[: int(state["attempt_count"])]
            attempt_records = partial_attempts.to_dict(orient="records")
        if partial_accepted_path.exists():
            partial_accepted = pd.read_parquet(partial_accepted_path).iloc[: int(state["accepted_count"])]
            accepted_records = partial_accepted.to_dict(orient="records")
        if len(attempt_records) != int(state["attempt_count"]) or len(accepted_records) != int(state["accepted_count"]):
            raise RuntimeError(f"Partial {label} generation files are incomplete.")
        next_candidate_id = int(state["next_candidate_id"])
        batch_index = int(state["batch_index"])

    chain_warm_starts: list[np.ndarray | None] = [None] * n_chains
    for record in sorted(accepted_records, key=lambda value: int(value["candidate_id"])):
        chain_id = int(record["chain_id"])
        chain_warm_starts[chain_id] = np.asarray(
            [record[f"Out_{name}"] for name in STATE_COLUMNS], dtype=np.float64
        )

    while len(accepted_records) < target_count and next_candidate_id < max_candidates:
        remaining = target_count - len(accepted_records)
        pool_size = target_count
        pool_index = next_candidate_id // pool_size
        pool_offset = next_candidate_id % pool_size
        batch_capacity = n_chains * int(sampling["parallel_chunk_size"])
        batch_count = min(
            max(remaining, n_chains),
            batch_capacity,
            pool_size - pool_offset,
            max_candidates - next_candidate_id,
        )
        candidate_pool = make_candidate_batch(pool_size, label=label, batch_index=pool_index, regime=regime)
        candidates = candidate_pool.iloc[pool_offset : pool_offset + batch_count].reset_index(drop=True)
        indexed = [(next_candidate_id + i, row.to_dict()) for i, (_, row) in enumerate(candidates.iterrows())]
        chains = [
            [item for item in indexed if int(item[0]) % n_chains == chain_id]
            for chain_id in range(n_chains)
        ]
        chain_results: list[dict[str, Any]] = []
        with ThreadPoolExecutor(max_workers=workers) as executor:
            future_map = {
                executor.submit(
                    solve_candidate_chain,
                    chain_id,
                    rows,
                    chain_warm_starts[chain_id],
                ): chain_id
                for chain_id, rows in enumerate(chains)
                if rows
            }
            for future in as_completed(future_map):
                records, final_warm_start = future.result()
                chain_id = future_map[future]
                chain_warm_starts[chain_id] = final_warm_start
                chain_results.extend(records)
        chain_results.sort(key=lambda record: int(record["candidate_id"]))
        attempt_records.extend(chain_results)
        for record in chain_results:
            if record["accepted"] and len(accepted_records) < target_count:
                accepted_records.append(record)
        next_candidate_id += batch_count
        batch_index += 1

        atomic_parquet(partial_attempts_path, pd.DataFrame(attempt_records))
        if accepted_records:
            atomic_parquet(partial_accepted_path, pd.DataFrame(accepted_records))
        atomic_json(
            partial_state_path,
            {
                "label": label,
                "regime": regime,
                "target_count": target_count,
                "config_sha256": CONFIG_HASH,
                "attempt_count": len(attempt_records),
                "accepted_count": len(accepted_records),
                "next_candidate_id": next_candidate_id,
                "batch_index": batch_index,
            },
        )

    if len(accepted_records) != target_count:
        raise RuntimeError(f"Generated only {len(accepted_records)} accepted {label} states from {next_candidate_id} candidates.")
    accepted = pd.DataFrame(accepted_records).copy()
    accepted.insert(0, "sample_id", [f"{label}_{index:06d}" for index in range(target_count)])
    accepted = accepted.drop(columns=["solve_seconds"], errors="ignore")
    attempts = pd.DataFrame(attempt_records).copy()
    accepted_identifier = dict(zip(accepted["candidate_id"], accepted["sample_id"], strict=True))
    attempts.insert(1, "accepted_sample_id", attempts["candidate_id"].map(accepted_identifier).fillna(""))
    attempts.insert(2, "selected_for_dataset", attempts["candidate_id"].isin(accepted_identifier))
    input_values = accepted[[f"In_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
    output_values = accepted[[f"Out_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
    input_composites = input_values @ COMPOSITION.T
    output_composites = output_values @ COMPOSITION.T
    for column_index, name in enumerate(MEASURED_COLUMNS):
        accepted[f"In_{name}"] = input_composites[:, column_index]
        accepted[f"Out_{name}"] = output_composites[:, column_index]
    for partial_path in (partial_attempts_path, partial_accepted_path, partial_state_path):
        with contextlib.suppress(FileNotFoundError):
            partial_path.unlink()
    return accepted, attempts


def validate_completed_dataset(
    dataset: pd.DataFrame,
    attempts: pd.DataFrame,
    *,
    label: str,
    target_count: int,
) -> None:
    expected_ids = [f"{label}_{index:06d}" for index in range(target_count)]
    if dataset.get("sample_id", pd.Series(dtype=str)).astype(str).tolist() != expected_ids:
        raise RuntimeError(f"Completed {label} dataset has invalid or reordered sample identifiers.")
    if dataset["candidate_id"].duplicated().any() or attempts["candidate_id"].duplicated().any():
        raise RuntimeError(f"Completed {label} dataset has duplicate candidate identifiers.")
    selected = attempts.loc[attempts["selected_for_dataset"].astype(bool)].copy()
    if len(dataset) != target_count or len(selected) != target_count:
        raise RuntimeError(f"Completed {label} dataset has the wrong accepted-row cardinality.")
    expected_link = dict(zip(dataset["candidate_id"], dataset["sample_id"], strict=True))
    observed_link = dict(zip(selected["candidate_id"], selected["accepted_sample_id"], strict=True))
    if expected_link != observed_link or not selected["accepted"].astype(bool).all():
        raise RuntimeError(f"Completed {label} dataset and attempt ledger are not linked consistently.")


def load_or_generate_dataset(
    label: str,
    target_count: int,
    regime: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, float, bool]:
    dataset_path = DIRS["datasets"] / f"{label}.parquet"
    attempts_path = DIRS["datasets"] / f"{label}.attempts.parquet"
    completion_path = DIRS["datasets"] / f"{label}.complete.json"
    if bool(ACTIVE["run"].get("resume", True)) and completion_path.exists():
        completion = read_json(completion_path)
        expected = {"label": label, "regime": regime, "target_count": target_count, "config_sha256": CONFIG_HASH}
        if any(completion.get(key) != value for key, value in expected.items()):
            raise RuntimeError(f"Completed {label} dataset belongs to another contract.")
        if not dataset_path.is_file() or not attempts_path.is_file():
            raise RuntimeError(f"Completed {label} dataset is missing a data file.")
        if sha256_file(dataset_path) != completion.get("dataset_sha256"):
            raise RuntimeError(f"Completed {label} dataset hash is inconsistent.")
        if sha256_file(attempts_path) != completion.get("attempts_sha256"):
            raise RuntimeError(f"Completed {label} attempt-ledger hash is inconsistent.")
        dataset = pd.read_parquet(dataset_path)
        attempts = pd.read_parquet(attempts_path)
        validate_completed_dataset(dataset, attempts, label=label, target_count=target_count)
        for artifact_name, artifact_path in (
            (f"dataset_{label}", dataset_path), (f"attempts_{label}", attempts_path),
            (f"dataset_{label}_complete", completion_path),
        ):
            register_artifact(artifact_name, artifact_path)
        return dataset, attempts, float(completion["generation_seconds"]), True
    if dataset_path.exists() or attempts_path.exists():
        raise RuntimeError(f"Found unsealed completed {label} dataset files; use a new run_id.")
    if DATASET_SOURCE_RUN_ROOT is not None:
        source_dataset_path = DATASET_SOURCE_RUN_ROOT / "datasets" / f"{label}.parquet"
        source_attempts_path = DATASET_SOURCE_RUN_ROOT / "datasets" / f"{label}.attempts.parquet"
        source_completion_path = DATASET_SOURCE_RUN_ROOT / "datasets" / f"{label}.complete.json"
        source_params_path = DATASET_SOURCE_RUN_ROOT / "inputs" / "params.resolved.json"
        source_manifest_path = DATASET_SOURCE_RUN_ROOT / "manifest.json"
        required_source_files = (
            source_dataset_path, source_attempts_path, source_completion_path,
            source_params_path, source_manifest_path,
        )
        if not all(path.is_file() for path in required_source_files):
            raise RuntimeError(f"Dataset source run {DATASET_SOURCE_RUN_ID!r} is incomplete.")
        source_completion = read_json(source_completion_path)
        source_params = read_json(source_params_path)
        source_manifest = read_json(source_manifest_path)
        if (
            source_completion.get("label") != label
            or source_completion.get("regime") != regime
            or int(source_completion.get("target_count", -1)) != target_count
            or source_completion.get("dataset_sha256") != sha256_file(source_dataset_path)
            or source_completion.get("attempts_sha256") != sha256_file(source_attempts_path)
            or source_manifest.get("contract", {}).get("workbook_sha256") != WORKBOOK_HASH
            or sha256_json(source_params.get("simulation")) != sha256_json(ACTIVE["simulation"])
        ):
            raise RuntimeError(f"Dataset source run {DATASET_SOURCE_RUN_ID!r} failed provenance checks.")
        dataset = pd.read_parquet(source_dataset_path)
        attempts = pd.read_parquet(source_attempts_path)
        validate_completed_dataset(dataset, attempts, label=label, target_count=target_count)
        atomic_bytes(dataset_path, source_dataset_path.read_bytes())
        atomic_bytes(attempts_path, source_attempts_path.read_bytes())
        generation_seconds = float(source_completion["generation_seconds"])
        atomic_json(
            completion_path,
            {
                "label": label,
                "regime": regime,
                "target_count": target_count,
                "config_sha256": CONFIG_HASH,
                "dataset_sha256": sha256_file(dataset_path),
                "attempts_sha256": sha256_file(attempts_path),
                "generation_seconds": generation_seconds,
                "source_run_id": DATASET_SOURCE_RUN_ID,
                "source_completion_sha256": sha256_file(source_completion_path),
                "imported_utc": utc_now(),
                "completed_utc": utc_now(),
            },
        )
        register_artifact(f"dataset_{label}", dataset_path)
        register_artifact(f"attempts_{label}", attempts_path)
        register_artifact(f"dataset_{label}_complete", completion_path)
        return dataset, attempts, generation_seconds, True
    generation_started = time.perf_counter()
    dataset, attempts = generate_dataset(target_count, label=label, regime=regime)
    atomic_parquet(dataset_path, dataset)
    atomic_parquet(attempts_path, attempts)
    generation_seconds = time.perf_counter() - generation_started
    validate_completed_dataset(dataset, attempts, label=label, target_count=target_count)
    atomic_json(
        completion_path,
        {
            "label": label,
            "regime": regime,
            "target_count": target_count,
            "config_sha256": CONFIG_HASH,
            "dataset_sha256": sha256_file(dataset_path),
            "attempts_sha256": sha256_file(attempts_path),
            "generation_seconds": generation_seconds,
            "completed_utc": utc_now(),
        },
    )
    register_artifact(f"dataset_{label}", dataset_path)
    register_artifact(f"attempts_{label}", attempts_path)
    register_artifact(f"dataset_{label}_complete", completion_path)
    return dataset, attempts, generation_seconds, False

In [ ]:
ID_TARGET = int(ACTIVE["simulation"]["sampling"]["id_samples"])
dataset_started = time.perf_counter()
ID_DATA, ID_ATTEMPTS, dataset_generation_seconds, dataset_resumed = load_or_generate_dataset("id", ID_TARGET)
dataset_load_or_generation_seconds = time.perf_counter() - dataset_started
dataset_path = DIRS["datasets"] / "id.parquet"
attempts_path = DIRS["datasets"] / "id.attempts.parquet"
register_artifact("dataset_id", dataset_path)
register_artifact("attempts_id", attempts_path)

input_state_columns = [f"In_{name}" for name in STATE_COLUMNS]
output_state_columns = [f"Out_{name}" for name in STATE_COLUMNS]
input_composite_columns = [f"In_{name}" for name in MEASURED_COLUMNS]
output_composite_columns = [f"Out_{name}" for name in MEASURED_COLUMNS]
if len(ID_DATA) != ID_TARGET or ID_DATA[output_state_columns].isna().any().any():
    raise ValueError("The accepted mechanistic dataset is incomplete or non-finite.")
id_in = ID_DATA[input_state_columns].to_numpy(np.float64)
id_out = ID_DATA[output_state_columns].to_numpy(np.float64)
ground_residual = (A_MATRIX @ (id_out - id_in).T).T
ground_linf = np.max(np.abs(ground_residual), axis=1)
if float(ground_linf.max()) > 1e-8 or float(id_out.min()) < -1e-10:
    raise ValueError("Accepted mechanistic states violate the physical data contract.")

dataset_validation = {
    "accepted": int(len(ID_DATA)),
    "attempted": int(len(ID_ATTEMPTS)),
    "acceptance_rate": float(len(ID_DATA) / len(ID_ATTEMPTS)),
    "maximum_ground_truth_conservation_linf": float(ground_linf.max()),
    "minimum_ground_truth_component": float(id_out.min()),
    "generation_seconds": float(dataset_generation_seconds),
    "load_or_generation_seconds_this_execution": float(dataset_load_or_generation_seconds),
    "resumed_completed_dataset": bool(dataset_resumed),
    "dataset_source_run_id": DATASET_SOURCE_RUN_ID,
    "median_attempt_solve_seconds": float(ID_ATTEMPTS["solve_seconds"].median()),
    "p95_attempt_solve_seconds": float(ID_ATTEMPTS["solve_seconds"].quantile(0.95)),
    "solver_strategy_counts": ID_DATA["solver_selected_strategy"].value_counts().to_dict(),
}
validation_path = atomic_json(DIRS["datasets"] / "id.validation.json", dataset_validation)
register_artifact("dataset_validation", validation_path)
stage_complete("mechanistic_dataset", **dataset_validation)
display(pd.Series(dataset_validation, name="value").to_frame())

## Frozen assessment split and ICSOR feature contract

A seed-42 permutation fixes 80% of the mechanistic states for coefficient assessment and leaves 20%
untouched until prediction. Component scales used by the assessment deployment QP are population
standard deviations of the 80% training targets only. The ordered 467-column feature map is exactly
$[1,u,x,u\otimes u,x\otimes x,u\otimes x]$; both ordered cross-products are retained.

In [ ]:
rng = np.random.default_rng(int(ACTIVE["icsor"]["split_seed"]))
permutation = rng.permutation(ID_TARGET)
n_test = int(round(float(ACTIVE["icsor"]["test_fraction"]) * ID_TARGET))
n_train = ID_TARGET - n_test
train_indices = np.sort(permutation[:n_train])
test_indices = np.sort(permutation[n_train:])
if set(train_indices).intersection(test_indices) or len(train_indices) + len(test_indices) != ID_TARGET:
    raise AssertionError("The assessment split is not a disjoint partition.")
if requested_profile == "full" and (len(train_indices), len(test_indices)) != (8000, 2000):
    raise AssertionError("The article assessment split must contain 8,000/2,000 rows.")

split_frame = pd.DataFrame(
    {
        "row_index": np.arange(ID_TARGET, dtype=int),
        "sample_id": ID_DATA["sample_id"].astype(str),
        "partition": np.where(np.isin(np.arange(ID_TARGET), train_indices), "train", "test"),
    }
)
split_path = atomic_csv(DIRS["splits"] / "assessment_split.csv", split_frame)
split_hash_path = atomic_json(
    DIRS["splits"] / "assessment_split.json",
    {
        "seed": int(ACTIVE["icsor"]["split_seed"]),
        "train_count": int(len(train_indices)),
        "test_count": int(len(test_indices)),
        "train_index_sha256": hashlib.sha256(train_indices.tobytes()).hexdigest(),
        "test_index_sha256": hashlib.sha256(test_indices.tobytes()).hexdigest(),
    },
)
register_artifact("assessment_split", split_path)
register_artifact("assessment_split_contract", split_hash_path)
stage_complete("assessment_split", train_count=len(train_indices), test_count=len(test_indices))

U_ALL = ID_DATA[OPERATIONAL_COLUMNS].to_numpy(np.float64)
X_ALL = ID_DATA[input_state_columns].to_numpy(np.float64)
Y_ALL = ID_DATA[output_state_columns].to_numpy(np.float64)


@dataclass(frozen=True)
class DesignSchema:
    labels: tuple[str, ...]
    blocks: tuple[str, ...]
    ranges: Mapping[str, tuple[int, int]]
    n_operational: int
    n_influent: int


def build_design_schema() -> DesignSchema:
    if not bool(ACTIVE["icsor"]["include_bias_term"]):
        raise ValueError(
            "This article's frozen 467-term ICSOR contract requires include_bias_term=true."
        )
    u_names = tuple(OPERATIONAL_COLUMNS)
    x_names = tuple(STATE_COLUMNS)
    labels: list[str] = []
    blocks: list[str] = []
    ranges: dict[str, tuple[int, int]] = {}

    def add(block: str, block_labels: Iterable[str]) -> None:
        start = len(labels)
        values = list(block_labels)
        labels.extend(values)
        blocks.extend([block] * len(values))
        ranges[block] = (start, len(labels))

    # Keep the tested icsor-model storage order and labels exactly:
    # [u, x, bias, u (x) u, x (x) x, u (x) x].  This is an exact column
    # permutation of the conceptual feature vector used in the manuscript.
    add("linear_operational", u_names)
    add("linear_influent", x_names)
    add("bias", ["Bias"])
    add("quadratic_operational", [f"{left}*{right}" for left in u_names for right in u_names])
    add("quadratic_influent", [f"{left}*{right}" for left in x_names for right in x_names])
    add(
        "interaction_operational_influent",
        [f"{left}*{right}" for left in u_names for right in x_names],
    )
    return DesignSchema(tuple(labels), tuple(blocks), ranges, len(u_names), len(x_names))


DESIGN_SCHEMA = build_design_schema()
if len(DESIGN_SCHEMA.labels) != 467 or len(set(DESIGN_SCHEMA.labels)) != 467:
    raise AssertionError("The ICSOR design must contain 467 uniquely labeled ordered terms.")

PUBLISHED_HELPER_INDEX = {label: index for index, label in enumerate(DESIGN_SCHEMA.labels)}


def build_design_matrix(u: np.ndarray, x: np.ndarray) -> np.ndarray:
    u_array = np.atleast_2d(np.asarray(u, dtype=np.float64))
    x_array = np.atleast_2d(np.asarray(x, dtype=np.float64))
    if u_array.shape[0] != x_array.shape[0] or u_array.shape[1] != 2 or x_array.shape[1] != 20:
        raise ValueError("ICSOR inputs must have aligned shapes N x 2 and N x 20.")
    uu = np.einsum("ni,nj->nij", u_array, u_array).reshape(len(u_array), -1)
    xx = np.einsum("ni,nj->nij", x_array, x_array).reshape(len(x_array), -1)
    ux = np.einsum("ni,nj->nij", u_array, x_array).reshape(len(u_array), -1)
    design = np.column_stack([u_array, x_array, np.ones(len(u_array)), uu, xx, ux])
    if design.shape[1] != len(DESIGN_SCHEMA.labels) or not np.all(np.isfinite(design)):
        raise AssertionError("The ICSOR feature map is incomplete or non-finite.")
    return design


PHI_ALL = build_design_matrix(U_ALL, X_ALL)
feature_contract_path = atomic_csv(
    DIRS["models"] / "feature_contract.csv",
    pd.DataFrame(
        {
            "column_index": np.arange(len(DESIGN_SCHEMA.labels)),
            "published_helper_index": [PUBLISHED_HELPER_INDEX[label] for label in DESIGN_SCHEMA.labels],
            "block": DESIGN_SCHEMA.blocks,
            "feature": DESIGN_SCHEMA.labels,
        }
    ),
)
register_artifact("feature_contract", feature_contract_path)


def symmetrize_quadratic_blocks(B: np.ndarray) -> np.ndarray:
    result = np.asarray(B, dtype=np.float64).copy()
    for block, dimension in (
        ("quadratic_operational", DESIGN_SCHEMA.n_operational),
        ("quadratic_influent", DESIGN_SCHEMA.n_influent),
    ):
        start, stop = DESIGN_SCHEMA.ranges[block]
        matrices = result[:, start:stop].reshape(result.shape[0], dimension, dimension)
        matrices[...] = 0.5 * (matrices + matrices.transpose(0, 2, 1))
    return result


def symmetry_error(B: np.ndarray) -> float:
    errors = []
    for block, dimension in (
        ("quadratic_operational", DESIGN_SCHEMA.n_operational),
        ("quadratic_influent", DESIGN_SCHEMA.n_influent),
    ):
        start, stop = DESIGN_SCHEMA.ranges[block]
        matrices = np.asarray(B)[:, start:stop].reshape(B.shape[0], dimension, dimension)
        errors.append(float(np.max(np.abs(matrices - matrices.transpose(0, 2, 1)))))
    return max(errors)

## Deterministic recursive coupled-QP estimation

Each sweep follows the published Gauss--Seidel order $B\rightarrow\Gamma\rightarrow\widehat C$.
The recursive-QP code path below is a self-contained adoption of the tested implementation in
`icsor-model/src/models/ml/icsor_coupled_qp.py` at the recorded commit and file hash.  The only
deliberate algorithmic change in this study is the scaled-$L_2$ deployment projection below.

In [ ]:
ICSOR_SETTINGS = copy.deepcopy(ACTIVE["icsor"])
ICSOR_TRAINING_SOURCE_COMMIT = str(ICSOR_SETTINGS["training_source_commit"])
ICSOR_TRAINING_SOURCE_SHA256 = str(ICSOR_SETTINGS["training_source_sha256"])
ICSOR_FEATURE_SOURCE_SHA256 = str(ICSOR_SETTINGS["feature_source_sha256"])
if int(ICSOR_SETTINGS["n_restarts"]) != 1:
    raise ValueError("This study adopts the tested single-restart recursive-QP ICSOR protocol.")


def osqp_solved(result: Any) -> bool:
    return (
        str(result.info.status or "").strip().lower() in {"solved", "solved inaccurate"}
        and result.x is not None
    )


def info_value(info: Any, *names: str) -> float:
    for name in names:
        if hasattr(info, name):
            return float(getattr(info, name))
    return float("nan")


def solve_B_update(phi: np.ndarray, C_hat: np.ndarray, Gamma: np.ndarray) -> np.ndarray:
    lambda_sys = float(ICSOR_SETTINGS["lambda_sys"])
    lambda_B = float(ICSOR_SETTINGS["lambda_B"])
    response = C_hat @ (np.eye(20) - Gamma).T
    lhs = lambda_sys * (phi.T @ phi) + lambda_B * np.eye(phi.shape[1])
    rhs = lambda_sys * (phi.T @ response)
    try:
        solved = np.linalg.solve(lhs, rhs)
    except np.linalg.LinAlgError:
        solved = np.linalg.pinv(lhs, rcond=1e-10) @ rhs
    unsymmetrized = solved.T
    driver_before = phi @ unsymmetrized.T
    B = symmetrize_quadratic_blocks(unsymmetrized)
    driver_after = phi @ B.T
    change = float(np.max(np.abs(driver_before - driver_after)))
    scale = max(1.0, float(np.max(np.abs(driver_before))))
    if change > float(ICSOR_SETTINGS["symmetry_tolerance"]) * scale:
        raise AssertionError(f"Symmetry projection changed fitted drivers by {change:.3e}.")
    return B


def contract_Gamma(Gamma: np.ndarray) -> tuple[np.ndarray, float, float]:
    limit = float(ICSOR_SETTINGS["conditioning_max"])
    identity = np.eye(Gamma.shape[0])
    for exponent in range(25):
        factor = 0.5**exponent
        candidate = factor * np.asarray(Gamma, dtype=np.float64)
        condition = float(np.linalg.cond(identity - candidate))
        if np.isfinite(condition) and condition <= limit:
            return candidate, condition, factor
    return np.zeros_like(Gamma, dtype=np.float64), 1.0, 0.0


def sanitize_warm_start_vector(
    vector: np.ndarray,
    *,
    lower_bounds: np.ndarray,
    upper_bounds: np.ndarray,
    clip_tolerance: float,
) -> np.ndarray | None:
    """Adopt the tested icsor-model warm-start sanitizer exactly."""
    candidate = np.asarray(vector, dtype=np.float64).reshape(-1)
    lower = np.asarray(lower_bounds, dtype=np.float64).reshape(-1)
    upper = np.asarray(upper_bounds, dtype=np.float64).reshape(-1)
    if candidate.shape != lower.shape or candidate.shape != upper.shape:
        return None
    if not np.all(np.isfinite(candidate)):
        return None
    clipped = candidate.copy()
    near_lower = (clipped < lower) & (clipped >= lower - clip_tolerance)
    clipped[near_lower] = lower[near_lower]
    clipped = np.maximum(clipped, lower)
    finite_upper = np.isfinite(upper)
    near_upper = finite_upper & (clipped > upper) & (clipped <= upper + clip_tolerance)
    clipped[near_upper] = upper[near_upper]
    clipped[finite_upper] = np.minimum(clipped[finite_upper], upper[finite_upper])
    return clipped if np.all(np.isfinite(clipped)) else None


def solve_Gamma_update(
    C_hat: np.ndarray,
    driver: np.ndarray,
    previous: np.ndarray,
    *,
    sweep: int,
) -> tuple[np.ndarray, pd.DataFrame]:
    n_outputs = C_hat.shape[1]
    lam_sys = float(ICSOR_SETTINGS["lambda_sys"])
    lam_gamma = float(ICSOR_SETTINGS["lambda_gamma"])
    bound = float(ICSOR_SETTINGS["gamma_abs_bound"])
    P = 2.0 * (lam_sys * (C_hat.T @ C_hat) + lam_gamma * np.eye(n_outputs))
    constraints = sp.eye(n_outputs, format="csc")
    lower_base = np.full(n_outputs, -bound)
    upper_base = np.full(n_outputs, bound)
    P_matrix = sp.csc_matrix(0.5 * (P + P.T))
    solver = osqp.OSQP()
    lower = lower_base.copy(); upper = upper_base.copy(); lower[0] = upper[0] = 0.0
    solver.setup(
        P=P_matrix, q=np.zeros(n_outputs), A=constraints,
        l=lower, u=upper, eps_abs=float(ICSOR_SETTINGS["osqp_eps_abs"]),
        eps_rel=float(ICSOR_SETTINGS["osqp_eps_rel"]),
        max_iter=int(ICSOR_SETTINGS["osqp_max_iter"]),
        polishing=bool(ICSOR_SETTINGS["osqp_polish"]),
        verbose=bool(ICSOR_SETTINGS["osqp_verbose"]),
        warm_starting=(
            bool(ICSOR_SETTINGS["enable_training_warm_start"])
            and bool(ICSOR_SETTINGS["enable_gamma_warm_start"])
        ),
    )
    residual_target = C_hat - driver
    cross = C_hat.T @ residual_target
    Gamma = np.zeros((n_outputs, n_outputs), dtype=np.float64)
    rows: list[dict[str, Any]] = []

    for target in range(n_outputs):
        q = -2.0 * lam_sys * cross[:, target]
        lower = lower_base.copy(); upper = upper_base.copy()
        lower[target] = upper[target] = 0.0
        solver.update(q=q, l=lower, u=upper)
        warm_start_used = False
        warm_start_skipped_invalid = False
        if (
            bool(ICSOR_SETTINGS["enable_training_warm_start"])
            and bool(ICSOR_SETTINGS["enable_gamma_warm_start"])
        ):
            warm = sanitize_warm_start_vector(
                previous[target],
                lower_bounds=lower,
                upper_bounds=upper,
                clip_tolerance=float(ICSOR_SETTINGS["warm_start_clip_tolerance"]),
            )
            if warm is not None:
                solver.warm_start(x=warm)
                warm_start_used = True
            else:
                warm_start_skipped_invalid = True
        result = solver.solve()
        solved = osqp_solved(result)
        fallback_used = not solved
        candidate = np.asarray(result.x, dtype=np.float64) if solved else np.zeros(n_outputs)
        candidate[target] = 0.0
        candidate = np.clip(candidate, -bound, bound)
        bound_violation = float(max(np.max(lower - candidate), np.max(candidate - upper), 0.0))
        Gamma[target] = candidate
        Gamma[target, target] = 0.0
        rows.append(
            {
                "sweep": sweep,
                "target_index": target,
                "target_component": STATE_COLUMNS[target],
                "status": str(result.info.status),
                "iterations": int(result.info.iter),
                "primal_residual": info_value(result.info, "prim_res", "pri_res"),
                "dual_residual": info_value(result.info, "dual_res", "dua_res"),
                "maximum_constraint_violation": bound_violation,
                "warm_start_used": warm_start_used,
                "warm_start_skipped_invalid": warm_start_skipped_invalid,
                "fallback_used": fallback_used,
            }
        )
    np.fill_diagonal(Gamma, 0.0)
    return Gamma, pd.DataFrame(rows)


def solve_Chat_update(
    targets: np.ndarray,
    influent: np.ndarray,
    A: np.ndarray,
    R: np.ndarray,
    driver: np.ndarray,
    previous: np.ndarray,
    *,
    sweep: int,
) -> tuple[np.ndarray, pd.DataFrame]:
    n_samples, n_outputs = targets.shape
    lam_inv = float(ICSOR_SETTINGS["lambda_inv"])
    lam_sys = float(ICSOR_SETTINGS["lambda_sys"])
    ata = A.T @ A
    P = 2.0 * (np.eye(n_outputs) + lam_inv * ata + lam_sys * (R.T @ R))
    linear = -2.0 * (targets + lam_inv * (influent @ ata.T) + lam_sys * (driver @ R))
    P_matrix = sp.csc_matrix(0.5 * (P + P.T))
    constraints = sp.eye(n_outputs, format="csc")
    lower = np.zeros(n_outputs)
    upper = np.full(n_outputs, np.inf)
    solver = osqp.OSQP()
    solver.setup(
        P=P_matrix, q=np.zeros(n_outputs), A=constraints, l=lower, u=upper,
        eps_abs=float(ICSOR_SETTINGS["osqp_eps_abs"]),
        eps_rel=float(ICSOR_SETTINGS["osqp_eps_rel"]),
        max_iter=int(ICSOR_SETTINGS["osqp_max_iter"]),
        polishing=bool(ICSOR_SETTINGS["osqp_polish"]),
        verbose=bool(ICSOR_SETTINGS["osqp_verbose"]),
        warm_starting=(
            bool(ICSOR_SETTINGS["enable_training_warm_start"])
            and bool(ICSOR_SETTINGS["enable_c_hat_warm_start"])
        ),
    )
    C_hat = np.empty_like(targets)
    rows: list[dict[str, Any]] = []

    for sample in range(n_samples):
        solver.update(q=linear[sample])
        warm_start_used = False
        warm_start_skipped_invalid = False
        if (
            bool(ICSOR_SETTINGS["enable_training_warm_start"])
            and bool(ICSOR_SETTINGS["enable_c_hat_warm_start"])
        ):
            warm = sanitize_warm_start_vector(
                previous[sample],
                lower_bounds=lower,
                upper_bounds=upper,
                clip_tolerance=float(ICSOR_SETTINGS["warm_start_clip_tolerance"]),
            )
            if warm is not None:
                solver.warm_start(x=warm)
                warm_start_used = True
            else:
                warm_start_skipped_invalid = True
        result = solver.solve()
        solved = osqp_solved(result)
        fallback_used = not solved
        if solved:
            candidate = np.maximum(np.asarray(result.x, dtype=np.float64), 0.0)
        else:
            candidate = np.maximum(np.asarray(targets[sample], dtype=np.float64), 0.0)
        C_hat[sample] = candidate
        rows.append(
            {
                "sweep": sweep,
                "sample_index": sample,
                "status": str(result.info.status),
                "iterations": int(result.info.iter),
                "primal_residual": info_value(result.info, "prim_res", "pri_res"),
                "dual_residual": info_value(result.info, "dual_res", "dua_res"),
                "minimum_component": float(np.min(candidate)),
                "maximum_constraint_violation": float(max(0.0, -np.min(candidate))),
                "warm_start_used": warm_start_used,
                "warm_start_skipped_invalid": warm_start_skipped_invalid,
                "fallback_used": fallback_used,
            }
        )
    return C_hat, pd.DataFrame(rows)


def objective_terms(
    targets: np.ndarray,
    influent: np.ndarray,
    A: np.ndarray,
    phi: np.ndarray,
    B: np.ndarray,
    Gamma: np.ndarray,
    C_hat: np.ndarray,
) -> dict[str, float]:
    R = np.eye(Gamma.shape[0]) - Gamma
    driver = phi @ B.T
    values = {
        "fit": float(np.sum((targets - C_hat) ** 2)),
        "invariant": float(ICSOR_SETTINGS["lambda_inv"] * np.sum(((C_hat - influent) @ A.T) ** 2)),
        "system": float(ICSOR_SETTINGS["lambda_sys"] * np.sum((C_hat @ R.T - driver) ** 2)),
        "B_ridge": float(ICSOR_SETTINGS["lambda_B"] * np.sum(B**2)),
        "Gamma_ridge": float(ICSOR_SETTINGS["lambda_gamma"] * np.sum(Gamma**2)),
    }
    values["objective"] = float(sum(values.values()))
    return values


def regression_indicator(running_best: Sequence[float], window: int) -> tuple[bool, float]:
    if len(running_best) < window:
        return False, float("nan")
    values = np.asarray(running_best[-window:], dtype=np.float64)
    values = values / (1.0 + abs(values[-1]))
    x = np.arange(window, dtype=np.float64)
    slope = float(np.polyfit(x, values, 1)[0])
    return True, abs(slope)


def fit_icsor(
    u: np.ndarray,
    x: np.ndarray,
    targets: np.ndarray,
    *,
    fit_label: str,
) -> dict[str, Any]:
    started = time.perf_counter()
    phi = build_design_matrix(u, x)
    diagnostic_dir = DIRS["models"] / f"{fit_label}_training_qp"
    diagnostic_dir.mkdir(parents=True, exist_ok=True)
    for pattern in ("gamma_sweep_*.parquet", "chat_sweep_*.parquet"):
        for stale_path in diagnostic_dir.glob(pattern):
            stale_path.unlink()
    Gamma, condition, contraction = contract_Gamma(np.zeros((20, 20)))
    C_hat = np.maximum(np.asarray(targets, dtype=np.float64), 0.0)
    B = solve_B_update(phi, C_hat, Gamma)
    initial_terms = objective_terms(targets, x, A_MATRIX, phi, B, Gamma, C_hat)
    history: list[dict[str, Any]] = [{"sweep": 0, **initial_terms, "R_condition": condition, "Gamma_contraction": contraction}]
    running_best = [initial_terms["objective"]]
    best_objective = initial_terms["objective"]
    best_iteration = 0
    stop_reason = "maximum_sweeps"
    max_sweeps = int(ICSOR_SETTINGS["max_outer_iterations"])
    for sweep in tqdm(range(1, max_sweeps + 1), desc=f"Fitting ICSOR ({fit_label})", unit="sweep"):
        B = solve_B_update(phi, C_hat, Gamma)
        driver = phi @ B.T
        Gamma_candidate, gamma_diagnostics = solve_Gamma_update(C_hat, driver, Gamma, sweep=sweep)
        Gamma, condition, contraction = contract_Gamma(Gamma_candidate)
        R = np.eye(20) - Gamma
        C_hat, chat_diagnostics = solve_Chat_update(
            targets, x, A_MATRIX, R, driver, C_hat, sweep=sweep
        )
        gamma_path = atomic_parquet(diagnostic_dir / f"gamma_sweep_{sweep:03d}.parquet", gamma_diagnostics)
        chat_path = atomic_parquet(diagnostic_dir / f"chat_sweep_{sweep:03d}.parquet", chat_diagnostics)
        terms = objective_terms(targets, x, A_MATRIX, phi, B, Gamma, C_hat)
        if terms["objective"] < best_objective:
            best_objective = terms["objective"]
            best_iteration = sweep
        running_best.append(min(running_best[-1], terms["objective"]))
        active, indicator = regression_indicator(
            running_best, int(ICSOR_SETTINGS["objective_regression_window"])
        )
        history.append(
            {
                "sweep": sweep,
                **terms,
                "running_best_objective": running_best[-1],
                "regression_indicator": indicator,
                "R_condition": condition,
                "Gamma_contraction": contraction,
                "gamma_qp_file": gamma_path.name,
                "chat_qp_file": chat_path.name,
                "gamma_mean_iterations": float(gamma_diagnostics["iterations"].mean()),
                "chat_mean_iterations": float(chat_diagnostics["iterations"].mean()),
                "gamma_fallback_count": int(gamma_diagnostics["fallback_used"].sum()),
                "chat_fallback_count": int(chat_diagnostics["fallback_used"].sum()),
                "gamma_warm_start_used_count": int(gamma_diagnostics["warm_start_used"].sum()),
                "chat_warm_start_used_count": int(chat_diagnostics["warm_start_used"].sum()),
                "chat_minimum_component": float(chat_diagnostics["minimum_component"].min()),
            }
        )
        if active and indicator <= float(ICSOR_SETTINGS["objective_regression_slope_tolerance"]):
            stop_reason = "running_best_regression_slope"
            break
    elapsed = time.perf_counter() - started
    final_terms = objective_terms(targets, x, A_MATRIX, phi, B, Gamma, C_hat)
    if not np.isclose(
        final_terms["objective"], history[-1]["objective"],
        rtol=float(ICSOR_SETTINGS["objective_replay_rtol"]),
        atol=float(ICSOR_SETTINGS["objective_replay_atol"]),
    ):
        raise AssertionError("The final ICSOR objective does not replay from frozen blocks.")
    if symmetry_error(B) > float(ICSOR_SETTINGS["symmetry_tolerance"]):
        raise AssertionError("The frozen coefficient matrix is not symmetric in ordered square blocks.")
    if B.shape != (20, 467) or not np.all(np.isfinite(B)):
        raise AssertionError("The frozen B matrix has an invalid shape or non-finite entries.")
    gamma_tolerance = float(ICSOR_SETTINGS["constraint_tolerance"])
    gamma_bound = float(ICSOR_SETTINGS["gamma_abs_bound"])
    if Gamma.shape != (20, 20) or not np.all(np.isfinite(Gamma)):
        raise AssertionError("The frozen Gamma matrix has an invalid shape or non-finite entries.")
    if float(np.max(np.abs(np.diag(Gamma)))) > gamma_tolerance:
        raise AssertionError("The frozen Gamma diagonal is not zero to tolerance.")
    off_diagonal = Gamma.copy()
    np.fill_diagonal(off_diagonal, 0.0)
    if float(np.max(np.abs(off_diagonal))) > gamma_bound + gamma_tolerance:
        raise AssertionError("The frozen Gamma matrix violates its entrywise bound.")
    if C_hat.shape != targets.shape or not np.all(np.isfinite(C_hat)):
        raise AssertionError("The final fitted-state matrix is incomplete or non-finite.")
    if float(np.min(C_hat)) < -float(ICSOR_SETTINGS["nonnegativity_tolerance"]):
        raise AssertionError("The final fitted-state matrix violates non-negativity tolerance.")
    R = np.eye(20) - Gamma
    condition = float(np.linalg.cond(R))
    if not np.isfinite(condition) or condition > float(ICSOR_SETTINGS["conditioning_max"]):
        raise AssertionError("The frozen coupled system is inadmissibly conditioned.")
    history_frame = pd.DataFrame(history)
    history_frame["selected_for_freezing"] = history_frame["sweep"].eq(
        int(history_frame["sweep"].max())
    )
    history_path = atomic_csv(DIRS["models"] / f"{fit_label}_training_history.csv", history_frame)
    return {
        "fit_label": fit_label,
        "A": A_MATRIX.copy(),
        "B": B,
        "Gamma": Gamma,
        "R": R,
        "C_hat": C_hat,
        "history": history_frame,
        "history_path": history_path,
        "training_seconds": elapsed,
        "training_rows": int(len(targets)),
        "sweeps": int(history_frame["sweep"].max()),
        "stop_reason": stop_reason,
        "best_objective": float(best_objective),
        "best_iteration": int(best_iteration),
        "final_objective": float(final_terms["objective"]),
        "last_iteration_objective": float(final_terms["objective"]),
        "R_condition": condition,
        "training_source_commit": ICSOR_TRAINING_SOURCE_COMMIT,
        "training_source_sha256": ICSOR_TRAINING_SOURCE_SHA256,
        "feature_source_sha256": ICSOR_FEATURE_SOURCE_SHA256,
    }


def raw_predict(model: Mapping[str, Any], u: np.ndarray, x: np.ndarray) -> np.ndarray:
    phi = build_design_matrix(u, x)
    driver = phi @ np.asarray(model["B"]).T
    return scipy.linalg.solve(
        np.asarray(model["R"]), driver.T, assume_a="gen", check_finite=False
    ).T


def affine_predict(raw: np.ndarray, x: np.ndarray) -> np.ndarray:
    raw_array = np.atleast_2d(np.asarray(raw, dtype=np.float64))
    x_array = np.atleast_2d(np.asarray(x, dtype=np.float64))
    projector = A_MATRIX.T @ A_MATRIX
    return raw_array - (raw_array - x_array) @ projector.T

## Unique scaled-$L_2$ physical deployment and held-out assessment

The present study replaces the published weighted-$L_1$ deployment branch by one strictly convex
QP. Its Hessian is fixed after the component scales are frozen. OSQP's result is independently
checked for equality feasibility, non-negativity, stationarity, dual feasibility, and
complementarity. A failed check raises an error; the influent is never substituted as a prediction.

In [ ]:
class DeploymentError(RuntimeError):
    pass


class L2Projector:
    def __init__(self, component_scales: np.ndarray, invariant_operator: np.ndarray):
        self.scales = np.asarray(component_scales, dtype=np.float64).reshape(-1)
        if self.scales.shape != (20,) or np.any(~np.isfinite(self.scales)) or np.any(self.scales <= 0):
            raise ValueError("All 20 deployment scales must be finite and strictly positive.")
        self.A = np.asarray(invariant_operator, dtype=np.float64)
        if self.A.ndim != 2 or self.A.shape[1] != 20 or not np.all(np.isfinite(self.A)):
            raise ValueError("The frozen invariant operator must be a finite K x 20 matrix.")
        if np.linalg.matrix_rank(self.A) != self.A.shape[0]:
            raise ValueError("The frozen invariant operator must have full row rank.")
        self.H = np.diag(1.0 / self.scales**2)
        self.constraint_matrix = sp.csc_matrix(np.vstack([self.A, np.eye(20)]))
        self._solver: osqp.OSQP | None = None
        self._last_x: np.ndarray | None = None

    def _new_solver(self, q: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> osqp.OSQP:
        settings = ACTIVE["deployment"]
        solver = osqp.OSQP()
        solver.setup(
            P=sp.csc_matrix(self.H), q=q, A=self.constraint_matrix, l=lower, u=upper,
            eps_abs=float(settings["osqp_eps_abs"]), eps_rel=float(settings["osqp_eps_rel"]),
            max_iter=int(settings["osqp_max_iter"]), polishing=bool(settings["osqp_polish"]),
            verbose=False, warm_starting=True,
        )
        return solver

    def _diagnostics(
        self,
        result: Any,
        c_aff: np.ndarray,
        influent: np.ndarray,
        *,
        attempt: str,
        elapsed_seconds: float,
    ) -> dict[str, Any]:
        if result.x is None or result.y is None:
            raise DeploymentError(f"OSQP returned no primal/dual solution ({result.info.status}).")
        c = np.asarray(result.x, dtype=np.float64)
        dual = np.asarray(result.y, dtype=np.float64)
        equality_dual = dual[: self.A.shape[0]]
        mu = -dual[self.A.shape[0] :]
        stationarity = self.H @ (c - c_aff) + self.A.T @ equality_dual - mu
        invariant_residual = self.A @ (c - influent)
        return {
            "status": str(result.info.status),
            "attempt": attempt,
            "iterations": int(result.info.iter),
            "solve_seconds": float(elapsed_seconds),
            "osqp_primal_residual": info_value(result.info, "prim_res", "pri_res"),
            "osqp_dual_residual": info_value(result.info, "dual_res", "dua_res"),
            "invariant_linf": float(np.max(np.abs(invariant_residual))),
            "minimum_component": float(np.min(c)),
            "stationarity_linf": float(np.max(np.abs(stationarity))),
            "minimum_nonnegativity_dual": float(np.min(mu)),
            "complementarity_linf": float(np.max(np.abs(c * mu))),
            "scaled_l2_displacement": float(np.linalg.norm((c - c_aff) / self.scales)),
            "qp_objective": float(0.5 * np.sum(((c - c_aff) / self.scales) ** 2)),
            "active_component_count": int(np.sum(c <= float(ACTIVE["deployment"]["active_component_tolerance"]))),
            "equality_dual": equality_dual,
            "nonnegativity_dual": mu,
        }

    @staticmethod
    def _accepted(diagnostics: Mapping[str, Any]) -> bool:
        settings = ACTIVE["deployment"]
        return bool(
            str(diagnostics.get("status", "")).strip().lower() == "solved"
            and float(diagnostics.get("invariant_linf", np.inf)) <= float(settings["invariant_tolerance"])
            and float(diagnostics.get("minimum_component", -np.inf)) >= -float(settings["nonnegativity_tolerance"])
            and float(diagnostics.get("stationarity_linf", np.inf)) <= float(settings["stationarity_tolerance"])
            and float(diagnostics.get("minimum_nonnegativity_dual", -np.inf)) >= -float(settings["dual_tolerance"])
            and float(diagnostics.get("complementarity_linf", np.inf)) <= float(settings["complementarity_tolerance"])
        )

    def solve(self, c_aff: np.ndarray, influent: np.ndarray) -> tuple[np.ndarray, dict[str, Any]]:
        c_aff_array = np.asarray(c_aff, dtype=np.float64).reshape(20)
        influent_array = np.asarray(influent, dtype=np.float64).reshape(20)
        q = -(self.H @ c_aff_array)
        equality_rhs = self.A @ influent_array
        lower = np.concatenate([equality_rhs, np.zeros(20)])
        upper = np.concatenate([equality_rhs, np.full(20, np.inf)])

        attempts = ["warm"]
        if bool(ACTIVE["deployment"]["cold_retry"]):
            attempts.append("cold_retry")
        failures: list[dict[str, Any]] = []
        for attempt in attempts:
            if attempt == "cold_retry" or self._solver is None:
                self._solver = self._new_solver(q, lower, upper)
            else:
                self._solver.update(q=q, l=lower, u=upper)
            if attempt == "warm" and self._last_x is not None:
                self._solver.warm_start(x=self._last_x)
            started = time.perf_counter()
            result = self._solver.solve()
            elapsed = time.perf_counter() - started
            try:
                diagnostics = self._diagnostics(
                    result, c_aff_array, influent_array, attempt=attempt, elapsed_seconds=elapsed
                )
            except DeploymentError:
                diagnostics = {"status": str(result.info.status), "attempt": attempt}
            if self._accepted(diagnostics):
                c = np.asarray(result.x, dtype=np.float64)
                self._last_x = c.copy()
                return c, diagnostics
            failures.append(diagnostics)
            self._solver = None
        raise DeploymentError(f"The lower physical-projection QP failed strict acceptance: {failures}")


def deploy_batch(
    model: Mapping[str, Any],
    u: np.ndarray,
    x: np.ndarray,
    component_scales: np.ndarray,
) -> dict[str, Any]:
    u_array = np.atleast_2d(np.asarray(u, dtype=np.float64))
    x_array = np.atleast_2d(np.asarray(x, dtype=np.float64))
    raw = raw_predict(model, u_array, x_array)
    affine = affine_predict(raw, x_array)
    projector = L2Projector(component_scales, np.asarray(model["A"], dtype=np.float64))
    deployed = np.empty_like(raw)
    rows: list[dict[str, Any]] = []
    equality_duals = np.empty((len(raw), np.asarray(model["A"]).shape[0]))
    nonnegativity_duals = np.empty_like(raw)
    for index in range(len(raw)):
        deployed[index], diagnostics = projector.solve(affine[index], x_array[index])
        equality_duals[index] = diagnostics.pop("equality_dual")
        nonnegativity_duals[index] = diagnostics.pop("nonnegativity_dual")
        rows.append({"row_index": index, **diagnostics})
    return {
        "raw": raw,
        "affine": affine,
        "deployed": deployed,
        "diagnostics": pd.DataFrame(rows),
        "equality_duals": equality_duals,
        "nonnegativity_duals": nonnegativity_duals,
    }


def physical_stage_summary(stages: Mapping[str, np.ndarray], influent: np.ndarray) -> pd.DataFrame:
    rows = []
    tolerance = float(ACTIVE["deployment"]["nonnegativity_tolerance"])
    for stage, values in stages.items():
        array = np.asarray(values, dtype=np.float64)
        residual = (A_MATRIX @ (array - influent).T).T
        rows.append(
            {
                "stage": stage,
                "maximum_invariant_linf": float(np.max(np.abs(residual))),
                "minimum_component": float(array.min()),
                "numerically_negative_component_count": int(np.sum(array < 0.0)),
                "below_tolerance_component_count": int(np.sum(array < -tolerance)),
                "sample_below_tolerance_rate": float(np.mean(np.min(array, axis=1) < -tolerance)),
            }
        )
    return pd.DataFrame(rows)


def accuracy_rows(
    actual: np.ndarray,
    predicted: np.ndarray,
    scales: np.ndarray,
    labels: Sequence[str],
    *,
    space: str,
    partition: str,
) -> pd.DataFrame:
    rows = []
    for index, label in enumerate(labels):
        residual = predicted[:, index] - actual[:, index]
        rmse = float(np.sqrt(np.mean(residual**2)))
        mae = float(np.mean(np.abs(residual)))
        rows.append(
            {
                "partition": partition,
                "space": space,
                "quantity": label,
                "scale": float(scales[index]),
                "rmse": rmse,
                "mae": mae,
                "bias": float(np.mean(residual)),
                "r2": float(r2_score(actual[:, index], predicted[:, index])),
                "nrmse": float(rmse / scales[index]),
                "nmae": float(mae / scales[index]),
                "maximum_absolute_error": float(np.max(np.abs(residual))),
            }
        )
    return pd.DataFrame(rows)


def assemble_prediction_rows(
    indices: np.ndarray,
    payload: Mapping[str, Any],
    *,
    partition: str,
) -> pd.DataFrame:
    data: dict[str, Any] = {
        "row_index": indices,
        "sample_id": ID_DATA.iloc[indices]["sample_id"].to_numpy(),
        "partition": np.repeat(partition, len(indices)),
        "HRT": U_ALL[indices, 0],
        "Aeration": U_ALL[indices, 1],
    }
    for stage in ("raw", "affine", "deployed"):
        values = np.asarray(payload[stage])
        composites = values @ COMPOSITION.T
        for j, name in enumerate(STATE_COLUMNS):
            data[f"{stage}_{name}"] = values[:, j]
        for j, name in enumerate(MEASURED_COLUMNS):
            data[f"{stage}_{name}"] = composites[:, j]
    for j, name in enumerate(STATE_COLUMNS):
        data[f"actual_{name}"] = Y_ALL[indices, j]
        data[f"error_{name}"] = payload["deployed"][:, j] - Y_ALL[indices, j]
    actual_composites = Y_ALL[indices] @ COMPOSITION.T
    deployed_composites = payload["deployed"] @ COMPOSITION.T
    for j, name in enumerate(MEASURED_COLUMNS):
        data[f"actual_{name}"] = actual_composites[:, j]
        data[f"error_{name}"] = deployed_composites[:, j] - actual_composites[:, j]
    frame = pd.DataFrame(data)
    diagnostics = payload["diagnostics"].drop(columns=["row_index"]).reset_index(drop=True)
    return pd.concat([frame.reset_index(drop=True), diagnostics], axis=1)


def evaluate_partition(
    model: Mapping[str, Any],
    indices: np.ndarray,
    component_scales: np.ndarray,
    composite_scales: np.ndarray,
    *,
    partition: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    payload = deploy_batch(model, U_ALL[indices], X_ALL[indices], component_scales)
    actual = Y_ALL[indices]
    component_metrics = accuracy_rows(
        actual, payload["deployed"], component_scales, STATE_COLUMNS,
        space="component", partition=partition,
    )
    actual_composites = actual @ COMPOSITION.T
    predicted_composites = payload["deployed"] @ COMPOSITION.T
    composite_metrics = accuracy_rows(
        actual_composites, predicted_composites, composite_scales, MEASURED_COLUMNS,
        space="composite", partition=partition,
    )
    physics = physical_stage_summary(
        {name: payload[name] for name in ("raw", "affine", "deployed")}, X_ALL[indices]
    )
    physics.insert(0, "partition", partition)
    rows = assemble_prediction_rows(indices, payload, partition=partition)
    return rows, pd.concat([component_metrics, composite_metrics], ignore_index=True), physics


def timed_call(label: str, function: Callable[[], Any], *, warmups: int, repeats: int, items: int) -> pd.DataFrame:
    for _ in range(warmups):
        function()
    rows = []
    for repetition in range(repeats):
        started = time.perf_counter_ns()
        function()
        seconds = (time.perf_counter_ns() - started) / 1e9
        rows.append(
            {
                "operation": label,
                "repetition": repetition,
                "items": items,
                "seconds": seconds,
                "microseconds_per_item": seconds * 1e6 / items,
            }
        )
    return pd.DataFrame(rows)


with threadpool_limits(limits=int(ACTIVE["run"]["threads"])):
    ASSESSMENT_MODEL = fit_icsor(
        U_ALL[train_indices], X_ALL[train_indices], Y_ALL[train_indices], fit_label="assessment"
    )

D_C_ASSESSMENT = np.std(Y_ALL[train_indices], axis=0, ddof=int(ACTIVE["deployment"]["scale_ddof"]))
D_T_ASSESSMENT = np.std(
    Y_ALL[train_indices] @ COMPOSITION.T, axis=0, ddof=int(ACTIVE["deployment"]["scale_ddof"])
)
if np.any(D_C_ASSESSMENT <= 0.0) or np.any(D_T_ASSESSMENT <= 0.0):
    raise ValueError("Assessment target scales must be strictly positive.")

# Property test: project an invariant-consistent point that has crossed the non-negative orthant,
# then replay it to verify feasibility and uniqueness without componentwise clipping.
property_influent = np.asarray(ACTIVE["optimization"]["nominal_influent"], dtype=np.float64)
test_direction = None
test_step = None
for column in range(Z_MATRIX.shape[1]):
    for sign in (-1.0, 1.0):
        direction = sign * Z_MATRIX[:, column]
        negative = direction < -1e-12
        if np.any(negative):
            crossing = float(np.min(property_influent[negative] / (-direction[negative])))
            if np.isfinite(crossing) and crossing > 0.0:
                test_direction = direction
                test_step = 1.25 * crossing
                break
    if test_direction is not None:
        break
if test_direction is None or test_step is None:
    raise AssertionError("Could not construct an invariant-consistent negative deployment fixture.")
test_affine = property_influent + test_step * test_direction
if float(test_affine.min()) >= 0.0 or float(np.max(np.abs(A_MATRIX @ (test_affine - property_influent)))) > 1e-10:
    raise AssertionError("The lower-QP property fixture is invalid.")
property_projector = L2Projector(D_C_ASSESSMENT, ASSESSMENT_MODEL["A"])
test_projected_1, test_qp_1 = property_projector.solve(test_affine, property_influent)
test_projected_2, test_qp_2 = property_projector.solve(test_affine, property_influent)
uniqueness_replay_error = float(np.max(np.abs(test_projected_1 - test_projected_2)))
if uniqueness_replay_error > 1e-8:
    raise AssertionError("The strictly convex lower QP did not replay its unique response.")
deployment_test_path = atomic_json(
    DIRS["models"] / "lower_qp_property_test.json",
    {
        "affine_minimum_component": float(test_affine.min()),
        "projected_minimum_component": float(test_projected_1.min()),
        "projected_invariant_linf": float(np.max(np.abs(A_MATRIX @ (test_projected_1 - property_influent)))),
        "uniqueness_replay_linf": uniqueness_replay_error,
        "first_solve": {key: value for key, value in test_qp_1.items() if np.isscalar(value)},
        "second_solve": {key: value for key, value in test_qp_2.items() if np.isscalar(value)},
    },
)
register_artifact("lower_qp_property_test", deployment_test_path)
stage_complete("deployment_tests", uniqueness_replay_linf=uniqueness_replay_error)

assessment_train_rows, assessment_train_metrics, assessment_train_physics = evaluate_partition(
    ASSESSMENT_MODEL, train_indices, D_C_ASSESSMENT, D_T_ASSESSMENT, partition="train"
)
assessment_test_rows, assessment_test_metrics, assessment_test_physics = evaluate_partition(
    ASSESSMENT_MODEL, test_indices, D_C_ASSESSMENT, D_T_ASSESSMENT, partition="test"
)
assessment_predictions = pd.concat([assessment_train_rows, assessment_test_rows], ignore_index=True)
assessment_metrics = pd.concat([assessment_train_metrics, assessment_test_metrics], ignore_index=True)
assessment_physics = pd.concat([assessment_train_physics, assessment_test_physics], ignore_index=True)
prediction_path = atomic_parquet(DIRS["predictions"] / "assessment_predictions.parquet", assessment_predictions)
metric_path = atomic_csv(DIRS["metrics"] / "assessment_accuracy.csv", assessment_metrics)
physics_path = atomic_csv(DIRS["metrics"] / "assessment_physical_diagnostics.csv", assessment_physics)
for name, path in (
    ("assessment_predictions", prediction_path),
    ("assessment_accuracy", metric_path),
    ("assessment_physics", physics_path),
    ("assessment_training_history", ASSESSMENT_MODEL["history_path"]),
):
    register_artifact(name, path)

timing_config = ACTIVE["timing"]
timing_batch_size = min(int(timing_config["batch_size"]), len(test_indices))
timing_indices = test_indices[:timing_batch_size]
timing_raw = raw_predict(ASSESSMENT_MODEL, U_ALL[timing_indices], X_ALL[timing_indices])
timing_affine = affine_predict(timing_raw, X_ALL[timing_indices])
assessment_timing = pd.concat(
    [
        pd.DataFrame(
            [{
                "operation": "assessment_training",
                "repetition": 0,
                "items": len(train_indices),
                "seconds": ASSESSMENT_MODEL["training_seconds"],
                "microseconds_per_item": ASSESSMENT_MODEL["training_seconds"] * 1e6 / len(train_indices),
            }]
        ),
        timed_call(
            "raw_dense_solve_batch",
            lambda: raw_predict(ASSESSMENT_MODEL, U_ALL[timing_indices], X_ALL[timing_indices]),
            warmups=int(timing_config["warmup_runs"]), repeats=int(timing_config["measured_runs"]),
            items=timing_batch_size,
        ),
        timed_call(
            "affine_projection_batch",
            lambda: affine_predict(timing_raw, X_ALL[timing_indices]),
            warmups=int(timing_config["warmup_runs"]), repeats=int(timing_config["measured_runs"]),
            items=timing_batch_size,
        ),
        timed_call(
            "lower_qp_batch",
            lambda: [L2Projector(D_C_ASSESSMENT, ASSESSMENT_MODEL["A"]).solve(ca, x) for ca, x in zip(timing_affine, X_ALL[timing_indices], strict=True)],
            warmups=int(timing_config["warmup_runs"]), repeats=int(timing_config["measured_runs"]),
            items=timing_batch_size,
        ),
        timed_call(
            "end_to_end_inference_batch",
            lambda: deploy_batch(ASSESSMENT_MODEL, U_ALL[timing_indices], X_ALL[timing_indices], D_C_ASSESSMENT),
            warmups=int(timing_config["warmup_runs"]), repeats=int(timing_config["measured_runs"]),
            items=timing_batch_size,
        ),
        timed_call(
            "end_to_end_inference_single",
            lambda: deploy_batch(ASSESSMENT_MODEL, U_ALL[timing_indices[:1]], X_ALL[timing_indices[:1]], D_C_ASSESSMENT),
            warmups=int(timing_config["warmup_runs"]), repeats=int(timing_config["single_prediction_repeats"]),
            items=1,
        ),
    ],
    ignore_index=True,
)
assessment_timing_path = atomic_csv(DIRS["timing"] / "assessment_fit_and_inference.csv", assessment_timing)
register_artifact("assessment_timing", assessment_timing_path)

test_summary = assessment_metrics.query("partition == 'test'").groupby("space")[["nrmse", "nmae"]].mean()
stage_complete(
    "icsor_assessment",
    training_seconds=ASSESSMENT_MODEL["training_seconds"],
    sweeps=ASSESSMENT_MODEL["sweeps"],
    stop_reason=ASSESSMENT_MODEL["stop_reason"],
    component_nrmse=float(test_summary.loc["component", "nrmse"]),
    component_nmae=float(test_summary.loc["component", "nmae"]),
    composite_nrmse=float(test_summary.loc["composite", "nrmse"]),
    composite_nmae=float(test_summary.loc["composite", "nmae"]),
)
display(test_summary)
display(assessment_physics)

## Production refit, frozen coefficients, and scales

Only after the untouched assessment has closed is ICSOR refitted on all accepted mechanistic states.
The complete labeled $20\times467$ coefficient matrix, $20\times20$ coupling matrix, physical
operators, scales, convergence history, and both raw and standardized coefficient forms are frozen.

In [ ]:
with threadpool_limits(limits=int(ACTIVE["run"]["threads"])):
    PRODUCTION_MODEL = fit_icsor(U_ALL, X_ALL, Y_ALL, fit_label="production")

D_C_PRODUCTION = np.std(Y_ALL, axis=0, ddof=int(ACTIVE["deployment"]["scale_ddof"]))
D_T_PRODUCTION = np.std(Y_ALL @ COMPOSITION.T, axis=0, ddof=int(ACTIVE["deployment"]["scale_ddof"]))
if np.any(D_C_PRODUCTION <= 0.0) or np.any(D_T_PRODUCTION <= 0.0):
    raise ValueError("Production target scales must be strictly positive.")

if requested_profile == "full":
    manuscript_dc = np.asarray([
        1.37206, 3.52010, 19.57521, 12.41127, 4.21792, 11.92429, 5.90257, 6.37422,
        23.09403, 51.98979, 29.14448, 42.47296, 45.60359, 17.42100, 5.27816, 14.90540,
        2.58557, 2.25104, 5.59170, 2.36730,
    ])
    manuscript_dt = np.asarray([79.4252, 12.1549, 7.07121, 54.5020])
    if not np.allclose(D_C_PRODUCTION, manuscript_dc, atol=5.1e-6, rtol=0.0):
        raise AssertionError("Regenerated component scales disagree with the manuscript's rounded values.")
    if not np.allclose(D_T_PRODUCTION, manuscript_dt, atol=5.1e-5, rtol=0.0):
        raise AssertionError("Regenerated composite scales disagree with the manuscript's rounded values.")

B = np.asarray(PRODUCTION_MODEL["B"])
Gamma = np.asarray(PRODUCTION_MODEL["Gamma"])
driver_all = PHI_ALL @ B.T
feature_sd = np.std(PHI_ALL, axis=0, ddof=0)
driver_sd = np.std(driver_all, axis=0, ddof=0)
standardized_B = B * feature_sd[None, :] / np.maximum(driver_sd[:, None], np.finfo(float).tiny)
coefficient_frame = pd.DataFrame(
    {
        "component": np.repeat(STATE_COLUMNS, B.shape[1]),
        "feature_index": np.tile(np.arange(B.shape[1]), B.shape[0]),
        "published_helper_index": np.tile(
            [PUBLISHED_HELPER_INDEX[label] for label in DESIGN_SCHEMA.labels], B.shape[0]
        ),
        "block": np.tile(DESIGN_SCHEMA.blocks, B.shape[0]),
        "feature": np.tile(DESIGN_SCHEMA.labels, B.shape[0]),
        "coefficient": B.ravel(),
        "standardized_coefficient": standardized_B.ravel(),
    }
)
gamma_frame = pd.DataFrame(
    {
        "target_component": np.repeat(STATE_COLUMNS, 20),
        "source_component": np.tile(STATE_COLUMNS, 20),
        "target_index": np.repeat(np.arange(20), 20),
        "source_index": np.tile(np.arange(20), 20),
        "coefficient": Gamma.ravel(),
    }
)
scale_frame = pd.DataFrame(
    {
        "space": ["component"] * 20 + ["composite"] * 4,
        "quantity": STATE_COLUMNS + MEASURED_COLUMNS,
        "assessment_scale": np.concatenate([D_C_ASSESSMENT, D_T_ASSESSMENT]),
        "production_scale": np.concatenate([D_C_PRODUCTION, D_T_PRODUCTION]),
    }
)
coefficient_path = atomic_csv(DIRS["models"] / "production_B_long.csv", coefficient_frame)
gamma_path = atomic_csv(DIRS["models"] / "production_Gamma_long.csv", gamma_frame)
scale_path = atomic_csv(DIRS["models"] / "target_scales.csv", scale_frame)
array_path = atomic_npz(
    DIRS["models"] / "production_icsor_arrays.npz",
    B=B, Gamma=Gamma, R=PRODUCTION_MODEL["R"], A=A_MATRIX,
    composition=COMPOSITION, D_c=D_C_PRODUCTION, D_T=D_T_PRODUCTION,
)
PRODUCTION_MODEL_FINGERPRINT = deterministic_array_fingerprint(
    {
        "B": B,
        "Gamma": Gamma,
        "R": PRODUCTION_MODEL["R"],
        "A": A_MATRIX,
        "composition": COMPOSITION,
        "D_c": D_C_PRODUCTION,
        "D_T": D_T_PRODUCTION,
    },
    metadata={
        "feature_labels": DESIGN_SCHEMA.labels,
        "optimization": ACTIVE["optimization"],
        "deployment": ACTIVE["deployment"],
    },
)
model_metadata_path = atomic_json(
    DIRS["models"] / "production_icsor_metadata.json",
    {
        "feature_labels": DESIGN_SCHEMA.labels,
        "feature_blocks": DESIGN_SCHEMA.blocks,
        "feature_block_ranges": DESIGN_SCHEMA.ranges,
        "feature_order": "icsor-model [u,x,1,u_otimes_u,x_otimes_x,u_otimes_x]",
        "published_helper_index": [PUBLISHED_HELPER_INDEX[label] for label in DESIGN_SCHEMA.labels],
        "state_columns": STATE_COLUMNS,
        "composite_columns": MEASURED_COLUMNS,
        "training_seconds": PRODUCTION_MODEL["training_seconds"],
        "sweeps": PRODUCTION_MODEL["sweeps"],
        "stop_reason": PRODUCTION_MODEL["stop_reason"],
        "best_objective": PRODUCTION_MODEL["best_objective"],
        "best_iteration": PRODUCTION_MODEL["best_iteration"],
        "final_objective": PRODUCTION_MODEL["final_objective"],
        "last_iteration_objective": PRODUCTION_MODEL["last_iteration_objective"],
        "R_condition": PRODUCTION_MODEL["R_condition"],
        "settings": ICSOR_SETTINGS,
        "training_source_commit": ICSOR_TRAINING_SOURCE_COMMIT,
        "training_source_sha256": ICSOR_TRAINING_SOURCE_SHA256,
        "feature_source_sha256": ICSOR_FEATURE_SOURCE_SHA256,
        "deployment_settings": ACTIVE["deployment"],
        "production_model_fingerprint": PRODUCTION_MODEL_FINGERPRINT,
        "array_sha256": sha256_file(array_path),
    },
)
for name, path in (
    ("production_B", coefficient_path), ("production_Gamma", gamma_path),
    ("production_scales", scale_path), ("production_arrays", array_path),
    ("production_metadata", model_metadata_path),
    ("production_training_history", PRODUCTION_MODEL["history_path"]),
):
    register_artifact(name, path)

coefficient_summary = (
    coefficient_frame.groupby(["component", "block"])
    .agg(nonzero=("coefficient", lambda s: int(np.count_nonzero(s))),
         mean_absolute=("coefficient", lambda s: float(np.mean(np.abs(s)))),
         maximum_absolute=("coefficient", lambda s: float(np.max(np.abs(s)))))
    .reset_index()
)
coefficient_summary_path = atomic_csv(DIRS["tables"] / "coefficient_block_summary.csv", coefficient_summary)
register_artifact("coefficient_summary", coefficient_summary_path)
stage_complete(
    "production_icsor",
    training_seconds=PRODUCTION_MODEL["training_seconds"],
    sweeps=PRODUCTION_MODEL["sweeps"],
    stop_reason=PRODUCTION_MODEL["stop_reason"],
    R_condition=PRODUCTION_MODEL["R_condition"],
    B_entries=int(B.size),
    Gamma_entries=int(Gamma.size),
)
display(scale_frame)
display(coefficient_summary.head(12))

## Fixed-influent quadratic response and deterministic bounded search

For one influent, all loading-only feature terms become an offset and all operation--loading terms
become operating slopes. The frozen 22-input head is therefore reduced exactly to a two-variable
quadratic before search. The search combines normalized two-dimensional DIRECT, explicit corners,
four one-dimensional boundary searches, an independent grid, and a halved local mesh. Every unique
verified candidate and lower-QP diagnostic is archived.

In [ ]:
OPERATING_LOWER = np.asarray([
    ACTIVE["optimization"]["bounds"]["HRT"][0],
    ACTIVE["optimization"]["bounds"]["Aeration"][0],
], dtype=np.float64)
OPERATING_UPPER = np.asarray([
    ACTIVE["optimization"]["bounds"]["HRT"][1],
    ACTIVE["optimization"]["bounds"]["Aeration"][1],
], dtype=np.float64)
OPERATING_RANGE = OPERATING_UPPER - OPERATING_LOWER
TARGET_WEIGHTS = np.asarray(ACTIVE["optimization"]["target_weights"], dtype=np.float64)
OPERATING_PENALTIES = np.asarray(ACTIVE["optimization"]["operating_penalties"], dtype=np.float64)
if ACTIVE["deployment"]["metric"] != "scaled_squared_l2":
    raise ValueError("The operational study requires the scaled-squared-L2 deployment metric.")
if (
    TARGET_WEIGHTS.shape != (4,)
    or np.any(~np.isfinite(TARGET_WEIGHTS))
    or not np.isclose(TARGET_WEIGHTS.sum(), 1.0)
    or np.any(TARGET_WEIGHTS < 0.0)
):
    raise ValueError("Target weights must be non-negative and sum to one.")
if OPERATING_PENALTIES.shape != (2,) or np.any(~np.isfinite(OPERATING_PENALTIES)) or np.any(OPERATING_PENALTIES < 0.0):
    raise ValueError("The HRT and aeration penalties must be finite and non-negative.")
if OPERATING_RANGE.shape != (2,) or np.any(~np.isfinite(OPERATING_RANGE)) or np.any(OPERATING_RANGE <= 0.0):
    raise ValueError("Each operating range must have finite, strictly increasing bounds.")


def objective_decomposition(u: np.ndarray, c: np.ndarray, D_T: np.ndarray) -> dict[str, float]:
    u_array = np.asarray(u, dtype=np.float64).reshape(2)
    c_array = np.asarray(c, dtype=np.float64).reshape(20)
    composites = COMPOSITION @ c_array
    scaled_targets = composites / np.asarray(D_T, dtype=np.float64)
    target_terms = TARGET_WEIGHTS * scaled_targets
    operating_terms = OPERATING_PENALTIES * ((u_array - OPERATING_LOWER) / OPERATING_RANGE)
    return {
        **{f"composite_{name}": float(composites[i]) for i, name in enumerate(MEASURED_COLUMNS)},
        **{f"scaled_{name}_term": float(target_terms[i]) for i, name in enumerate(MEASURED_COLUMNS)},
        "quality_objective": float(target_terms.sum()),
        "HRT_penalty": float(operating_terms[0]),
        "Aeration_penalty": float(operating_terms[1]),
        "operating_objective": float(operating_terms.sum()),
        "objective": float(target_terms.sum() + operating_terms.sum()),
    }


def derive_fixed_influent_quadratic(model: Mapping[str, Any], influent: np.ndarray) -> dict[str, np.ndarray]:
    x = np.asarray(influent, dtype=np.float64).reshape(20)
    B = np.asarray(model["B"], dtype=np.float64)
    ranges = DESIGN_SCHEMA.ranges
    b0, b1 = ranges["bias"]
    u0, u1 = ranges["linear_operational"]
    x0, x1 = ranges["linear_influent"]
    uu0, uu1 = ranges["quadratic_operational"]
    xx0, xx1 = ranges["quadratic_influent"]
    ux0, ux1 = ranges["interaction_operational_influent"]
    intercept_driver = (
        B[:, b0:b1].reshape(20)
        + B[:, x0:x1] @ x
        + B[:, xx0:xx1] @ np.kron(x, x)
    )
    interaction = B[:, ux0:ux1].reshape(20, 2, 20)
    linear_driver = B[:, u0:u1] + np.einsum("fux,x->fu", interaction, x)
    quadratic_driver = B[:, uu0:uu1]
    R = np.asarray(model["R"], dtype=np.float64)
    return {
        "intercept": scipy.linalg.solve(R, intercept_driver, assume_a="gen", check_finite=False),
        "linear": scipy.linalg.solve(R, linear_driver, assume_a="gen", check_finite=False),
        "quadratic_ordered": scipy.linalg.solve(R, quadratic_driver, assume_a="gen", check_finite=False),
    }


def evaluate_fixed_quadratic(surface: Mapping[str, np.ndarray], u: np.ndarray) -> np.ndarray:
    u_array = np.asarray(u, dtype=np.float64).reshape(2)
    return (
        np.asarray(surface["intercept"])
        + np.asarray(surface["linear"]) @ u_array
        + np.asarray(surface["quadratic_ordered"]) @ np.kron(u_array, u_array)
    )


def point_key(u: np.ndarray) -> tuple[str, str]:
    value = np.asarray(u, dtype=np.float64).reshape(2)
    return float(value[0]).hex(), float(value[1]).hex()


def flatten_state(prefix: str, values: np.ndarray) -> dict[str, float]:
    return {f"{prefix}_{name}": float(value) for name, value in zip(STATE_COLUMNS, values, strict=True)}


def flatten_composites(prefix: str, values: np.ndarray) -> dict[str, float]:
    return {f"{prefix}_{name}": float(value) for name, value in zip(MEASURED_COLUMNS, values, strict=True)}


class SurrogateCaseEvaluator:
    def __init__(
        self,
        model: Mapping[str, Any],
        influent: np.ndarray,
        D_c: np.ndarray,
        D_T: np.ndarray,
        *,
        budget: int,
        case_id: str,
    ):
        self.model = model
        self.influent = np.asarray(influent, dtype=np.float64).reshape(20)
        self.D_c = np.asarray(D_c, dtype=np.float64)
        self.D_T = np.asarray(D_T, dtype=np.float64)
        self.budget = int(budget)
        self.case_id = case_id
        self.surface = derive_fixed_influent_quadratic(model, self.influent)
        self.projector = L2Projector(self.D_c, np.asarray(model["A"], dtype=np.float64))
        self.cache: dict[tuple[str, str], dict[str, Any]] = {}
        self.budget_hit = False
        self.phase_visits: defaultdict[tuple[str, str], set[str]] = defaultdict(set)

    @property
    def evaluations(self) -> int:
        return len(self.cache)

    def _compute(self, u: np.ndarray, *, phase: str) -> dict[str, Any]:
        u_array = np.asarray(u, dtype=np.float64).reshape(2)
        if np.any(u_array < OPERATING_LOWER) or np.any(u_array > OPERATING_UPPER):
            raise ValueError("An optimization candidate lies outside the trained operating domain.")
        started = time.perf_counter()
        raw = evaluate_fixed_quadratic(self.surface, u_array)
        affine = affine_predict(raw[None, :], self.influent[None, :])[0]
        deployed, qp = self.projector.solve(affine, self.influent)
        elapsed = time.perf_counter() - started
        decomposition = objective_decomposition(u_array, deployed, self.D_T)
        record = {
            "case_id": self.case_id,
            "evaluation_index": self.evaluations,
            "phase_first": phase,
            "HRT": float(u_array[0]),
            "Aeration": float(u_array[1]),
            "normalized_HRT": float((u_array[0] - OPERATING_LOWER[0]) / OPERATING_RANGE[0]),
            "normalized_Aeration": float((u_array[1] - OPERATING_LOWER[1]) / OPERATING_RANGE[1]),
            "evaluation_seconds": float(elapsed),
            **decomposition,
            **{f"lower_{key}": value for key, value in qp.items() if np.isscalar(value)},
            **flatten_state("raw", raw),
            **flatten_state("affine", affine),
            **flatten_state("predicted", deployed),
            **flatten_composites("predicted", COMPOSITION @ deployed),
        }
        return record

    def evaluate(self, u: np.ndarray, *, phase: str) -> float:
        key = point_key(u)
        self.phase_visits[key].add(phase)
        if key in self.cache:
            return float(self.cache[key]["objective"])
        if self.evaluations >= self.budget:
            self.budget_hit = True
            return float("inf")
        record = self._compute(u, phase=phase)
        self.cache[key] = record
        return float(record["objective"])

    def verify_uncached(self, u: np.ndarray) -> dict[str, Any]:
        record = self._compute(u, phase="uncached_final_verification")
        cached = self.cache[point_key(u)]
        if not np.isclose(record["objective"], cached["objective"], rtol=0.0, atol=1e-10):
            raise AssertionError("The final surrogate objective did not replay deterministically.")
        return record

    def best_record(self) -> dict[str, Any]:
        finite = [row for row in self.cache.values() if np.isfinite(row["objective"])]
        if not finite:
            raise RuntimeError("The surrogate search produced no finite verified candidate.")
        return min(finite, key=lambda row: (row["objective"], row["HRT"], row["Aeration"]))

    def frame(self) -> pd.DataFrame:
        rows = []
        for key, record in self.cache.items():
            rows.append({**record, "phase_visits": ";".join(sorted(self.phase_visits[key]))})
        return pd.DataFrame(rows).sort_values("evaluation_index").reset_index(drop=True)


class MechanisticCaseEvaluator:
    def __init__(self, influent: np.ndarray, D_T: np.ndarray, *, budget: int, case_id: str):
        self.influent = np.asarray(influent, dtype=np.float64).reshape(20)
        self.D_T = np.asarray(D_T, dtype=np.float64)
        self.budget = int(budget)
        self.case_id = case_id
        self.cache: dict[tuple[str, str], dict[str, Any]] = {}
        self.budget_hit = False
        self.phase_visits: defaultdict[tuple[str, str], set[str]] = defaultdict(set)

    @property
    def evaluations(self) -> int:
        return len(self.cache)

    def _compute(self, u: np.ndarray, *, phase: str) -> dict[str, Any]:
        u_array = np.asarray(u, dtype=np.float64).reshape(2)
        started = time.perf_counter()
        failure = ""
        try:
            effluent, diagnostics = simulate_asm2d_tsn_steady_state(
                influent_state=self.influent,
                hrt_hours=float(u_array[0]), aeration=float(u_array[1]),
                model_params=SIM_PARAMS, matrix_bundle=MATRIX_BUNDLE,
                previous_solution=None, enforce_acceptance=False,
            )
            accepted = bool(diagnostics["accepted"] and np.all(np.isfinite(effluent)))
            if not accepted:
                failure = f"solver_rejected: residual_max={diagnostics.get('residual_max')}"
        except Exception as error:
            effluent = np.full(20, np.nan)
            diagnostics = {"accepted": False, "success": False, "selected_strategy": "exception"}
            accepted = False
            failure = f"{type(error).__name__}: {error}"
        elapsed = time.perf_counter() - started
        decomposition = (
            objective_decomposition(u_array, effluent, self.D_T)
            if accepted else {"objective": float("inf"), "quality_objective": np.nan,
                              "operating_objective": np.nan, "HRT_penalty": np.nan,
                              "Aeration_penalty": np.nan,
                              **{f"composite_{name}": np.nan for name in MEASURED_COLUMNS}}
        )
        return {
            "case_id": self.case_id,
            "evaluation_index": self.evaluations,
            "phase_first": phase,
            "HRT": float(u_array[0]),
            "Aeration": float(u_array[1]),
            "accepted": accepted,
            "failure_reason": failure,
            "evaluation_seconds": float(elapsed),
            **decomposition,
            **{f"solver_{key}": value for key, value in diagnostics.items() if np.isscalar(value)},
            **flatten_state("mechanistic", effluent),
            **(flatten_composites("mechanistic", COMPOSITION @ effluent) if accepted else {}),
        }

    def evaluate(self, u: np.ndarray, *, phase: str) -> float:
        key = point_key(u)
        self.phase_visits[key].add(phase)
        if key in self.cache:
            return float(self.cache[key]["objective"])
        if self.evaluations >= self.budget:
            self.budget_hit = True
            return float("inf")
        record = self._compute(u, phase=phase)
        self.cache[key] = record
        return float(record["objective"])

    def verify_uncached(self, u: np.ndarray) -> dict[str, Any]:
        record = self._compute(u, phase="uncached_final_verification")
        cached = self.cache[point_key(u)]
        if bool(record["accepted"]) != bool(cached["accepted"]):
            raise AssertionError("The final mechanistic acceptance status did not replay.")
        if record["accepted"] and not np.isclose(record["objective"], cached["objective"], rtol=1e-10, atol=1e-10):
            raise AssertionError("The final mechanistic objective did not replay deterministically.")
        return record

    def best_record(self) -> dict[str, Any]:
        finite = [row for row in self.cache.values() if row["accepted"] and np.isfinite(row["objective"])]
        if not finite:
            raise RuntimeError("The mechanistic search produced no accepted finite candidate.")
        return min(finite, key=lambda row: (row["objective"], row["HRT"], row["Aeration"]))

    def frame(self) -> pd.DataFrame:
        rows = []
        for key, record in self.cache.items():
            rows.append({**record, "phase_visits": ";".join(sorted(self.phase_visits[key]))})
        return pd.DataFrame(rows).sort_values("evaluation_index").reset_index(drop=True)


def normalized_to_physical(z: np.ndarray) -> np.ndarray:
    return OPERATING_LOWER + np.asarray(z, dtype=np.float64).reshape(2) * OPERATING_RANGE


def run_direct_phase(
    evaluator: Any,
    settings: Mapping[str, Any],
    *,
    phase: str,
    fixed_dimension: int | None = None,
    fixed_value: float | None = None,
) -> dict[str, Any]:
    if evaluator.budget_hit:
        return {"phase": phase, "skipped": True, "reason": "total_budget"}
    if fixed_dimension is None:
        bounds = [(0.0, 1.0), (0.0, 1.0)]
        maxfun = int(settings["interior_direct_maxfun"])

        def function(z: np.ndarray) -> float:
            return evaluator.evaluate(normalized_to_physical(z), phase=phase)
    else:
        bounds = [(0.0, 1.0)]
        maxfun = int(settings["edge_direct_maxfun"])

        def function(value: np.ndarray) -> float:
            z = np.empty(2)
            z[fixed_dimension] = float(fixed_value)
            z[1 - fixed_dimension] = float(np.asarray(value).reshape(-1)[0])
            return evaluator.evaluate(normalized_to_physical(z), phase=phase)
    started = time.perf_counter()
    result = direct(
        function, bounds, maxfun=maxfun, maxiter=max(1000, maxfun),
        locally_biased=bool(settings["locally_biased"]),
        len_tol=float(settings["direct_len_tol"]), vol_tol=float(settings["direct_vol_tol"]),
    )
    return {
        "phase": phase,
        "success": bool(result.success),
        "message": str(result.message),
        "nfev_reported": int(result.nfev),
        "fun_reported": float(result.fun),
        "seconds": time.perf_counter() - started,
    }


def run_bounded_search(evaluator: Any, settings: Mapping[str, Any]) -> dict[str, Any]:
    started = time.perf_counter()
    phase_status: list[dict[str, Any]] = []
    evaluator.evaluate(np.asarray(ACTIVE["optimization"]["baseline"]), phase="center")
    phase_status.append(run_direct_phase(evaluator, settings, phase="interior_direct"))

    for z0 in (0.0, 1.0):
        for z1 in (0.0, 1.0):
            evaluator.evaluate(normalized_to_physical(np.asarray([z0, z1])), phase="corner")
    for fixed_dimension, fixed_name in ((0, "HRT"), (1, "Aeration")):
        for fixed_value, side in ((0.0, "lower"), (1.0, "upper")):
            phase_status.append(
                run_direct_phase(
                    evaluator, settings,
                    phase=f"edge_{fixed_name}_{side}",
                    fixed_dimension=fixed_dimension, fixed_value=fixed_value,
                )
            )

    grid_h, grid_a = map(int, settings["independent_grid_points"])
    grid_completed = True
    for h in np.linspace(OPERATING_LOWER[0], OPERATING_UPPER[0], grid_h):
        for a in np.linspace(OPERATING_LOWER[1], OPERATING_UPPER[1], grid_a):
            evaluator.evaluate(np.asarray([h, a]), phase="independent_grid")
            if evaluator.budget_hit:
                grid_completed = False
                break
        if evaluator.budget_hit:
            break

    step = np.asarray(settings["mesh_initial_step"], dtype=np.float64)
    minimum_step = np.asarray(settings["mesh_min_step"], dtype=np.float64)
    relative_tolerance = float(settings["objective_relative_tolerance"])
    mesh_completed = False
    mesh_iterations = 0
    while not evaluator.budget_hit:
        mesh_iterations += 1
        incumbent_before = evaluator.best_record()
        u0 = np.asarray([incumbent_before["HRT"], incumbent_before["Aeration"]])
        for dh in (-step[0], 0.0, step[0]):
            for da in (-step[1], 0.0, step[1]):
                if dh == 0.0 and da == 0.0:
                    continue
                candidate = np.clip(u0 + np.asarray([dh, da]), OPERATING_LOWER, OPERATING_UPPER)
                evaluator.evaluate(candidate, phase=f"local_mesh_{mesh_iterations:02d}")
                if evaluator.budget_hit:
                    break
            if evaluator.budget_hit:
                break
        incumbent_after = evaluator.best_record()
        improvement = incumbent_before["objective"] - incumbent_after["objective"]
        threshold = relative_tolerance * max(1.0, abs(incumbent_before["objective"]))
        if improvement <= threshold:
            if np.all(step <= minimum_step + np.finfo(float).eps):
                mesh_completed = True
                break
            step = np.maximum(step / 2.0, minimum_step)
        if mesh_iterations > 1000:
            raise RuntimeError("Local mesh refinement exceeded its deterministic safety limit.")

    best = evaluator.best_record()
    evaluator.verify_uncached(np.asarray([best["HRT"], best["Aeration"]]))
    resolution_qualified = bool(not evaluator.budget_hit and grid_completed and mesh_completed)
    return {
        "best": best,
        "termination": "resolution_qualified" if resolution_qualified else "budget_limited_incumbent",
        "resolution_qualified": resolution_qualified,
        "budget_hit": bool(evaluator.budget_hit),
        "verified_evaluations": int(evaluator.evaluations),
        "achieved_mesh_step_HRT": float(step[0]),
        "achieved_mesh_step_Aeration": float(step[1]),
        "grid_completed": bool(grid_completed),
        "mesh_completed": bool(mesh_completed),
        "mesh_iterations": int(mesh_iterations),
        "phase_status": phase_status,
        "search_seconds": float(time.perf_counter() - started),
    }


class ToyBoundaryEvaluator:
    """Minimal deterministic evaluator used to test the bounded-search orchestration."""
    def __init__(self, budget: int):
        self.budget = int(budget)
        self.cache: dict[tuple[str, str], dict[str, Any]] = {}
        self.budget_hit = False

    @property
    def evaluations(self) -> int:
        return len(self.cache)

    def evaluate(self, u: np.ndarray, *, phase: str) -> float:
        key = point_key(u)
        if key in self.cache:
            return float(self.cache[key]["objective"])
        if self.evaluations >= self.budget:
            self.budget_hit = True
            return float("inf")
        value = np.asarray(u, dtype=np.float64)
        objective = float(((value[0] - OPERATING_UPPER[0]) / OPERATING_RANGE[0]) ** 2
                          + ((value[1] - OPERATING_LOWER[1]) / OPERATING_RANGE[1]) ** 2)
        self.cache[key] = {
            "objective": objective, "HRT": float(value[0]), "Aeration": float(value[1]),
            "phase_first": phase,
        }
        return objective

    def best_record(self) -> dict[str, Any]:
        return min(self.cache.values(), key=lambda row: (row["objective"], row["HRT"], row["Aeration"]))

    def verify_uncached(self, u: np.ndarray) -> dict[str, Any]:
        value = np.asarray(u, dtype=np.float64)
        objective = float(((value[0] - OPERATING_UPPER[0]) / OPERATING_RANGE[0]) ** 2
                          + ((value[1] - OPERATING_LOWER[1]) / OPERATING_RANGE[1]) ** 2)
        if not np.isclose(objective, self.cache[point_key(value)]["objective"], atol=0.0, rtol=0.0):
            raise AssertionError("The toy objective replay failed.")
        return self.cache[point_key(value)]


toy_settings = copy.deepcopy(ACTIVE["optimization"]["surrogate_search"])
toy_settings.update(
    {
        "total_verified_budget": 180,
        "interior_direct_maxfun": 40,
        "edge_direct_maxfun": 10,
        "independent_grid_points": [7, 5],
        "direct_len_tol": 0.05,
        "mesh_initial_step": [1.0, 0.1],
        "mesh_min_step": [0.25, 0.025],
    }
)
toy_evaluator = ToyBoundaryEvaluator(int(toy_settings["total_verified_budget"]))
toy_search = run_bounded_search(toy_evaluator, toy_settings)
toy_best = toy_search["best"]
if not (
    np.isclose(toy_best["HRT"], OPERATING_UPPER[0], atol=0.0, rtol=0.0)
    and np.isclose(toy_best["Aeration"], OPERATING_LOWER[1], atol=0.0, rtol=0.0)
    and np.isclose(toy_best["objective"], 0.0, atol=0.0, rtol=0.0)
):
    raise AssertionError("The search orchestration failed its exact boundary-optimum test.")
optimization_test_path = atomic_json(
    DIRS["optimization"] / "search_property_test.json",
    {
        "known_optimum": [float(OPERATING_UPPER[0]), float(OPERATING_LOWER[1])],
        "returned_optimum": [toy_best["HRT"], toy_best["Aeration"]],
        "objective": toy_best["objective"],
        "evaluations": toy_search["verified_evaluations"],
        "termination": toy_search["termination"],
    },
)
register_artifact("search_property_test", optimization_test_path)
stage_complete("optimization_tests", exact_boundary_optimum=True)


def boundary_flags(u: np.ndarray, tolerance: Sequence[float]) -> dict[str, bool]:
    value = np.asarray(u, dtype=np.float64)
    tol = np.asarray(tolerance, dtype=np.float64)
    return {
        "HRT_lower_active": bool(value[0] <= OPERATING_LOWER[0] + tol[0]),
        "HRT_upper_active": bool(value[0] >= OPERATING_UPPER[0] - tol[0]),
        "Aeration_lower_active": bool(value[1] <= OPERATING_LOWER[1] + tol[1]),
        "Aeration_upper_active": bool(value[1] >= OPERATING_UPPER[1] - tol[1]),
    }


# Exact parameterization replay test at deterministic operating points.
NOMINAL_X = np.asarray(ACTIVE["optimization"]["nominal_influent"], dtype=np.float64)
if NOMINAL_X.shape != (20,):
    raise ValueError("The nominal influent must contain exactly 20 ordered components.")
for component_index, name in enumerate(STATE_COLUMNS):
    lo, hi = ACTIVE["simulation"]["influent_state_ranges"][name]
    if not float(lo) <= NOMINAL_X[component_index] <= float(hi):
        raise ValueError(f"Nominal {name} lies outside the training domain.")
surface_test_u = qmc_scale(
    LatinHypercube(d=2, seed=271828).random(10), OPERATING_LOWER, OPERATING_UPPER
)
replay_influents = [NOMINAL_X, X_ALL[0], X_ALL[len(X_ALL) // 2], X_ALL[-1]]
replay_rows: list[dict[str, Any]] = []
for replay_index, replay_influent in enumerate(replay_influents):
    replay_surface = derive_fixed_influent_quadratic(PRODUCTION_MODEL, replay_influent)
    surface_direct = raw_predict(
        PRODUCTION_MODEL, surface_test_u, np.repeat(replay_influent[None, :], len(surface_test_u), axis=0)
    )
    surface_reduced = np.vstack([evaluate_fixed_quadratic(replay_surface, u) for u in surface_test_u])
    surface_error = float(np.max(np.abs(surface_direct - surface_reduced)))
    replay_scale = max(1.0, float(np.max(np.abs(surface_direct))))
    replay_rows.append(
        {
            "influent_replay_index": replay_index,
            "operating_points": len(surface_test_u),
            "maximum_absolute_error": surface_error,
            "maximum_relative_scaled_error": surface_error / replay_scale,
        }
    )
    if surface_error > 1e-8 * replay_scale:
        raise AssertionError("The fixed-influent quadratic reduction does not replay the full ICSOR head.")
fixed_quadratic_replay_path = atomic_json(
    DIRS["models"] / "fixed_influent_quadratic_replay.json",
    {"cases": replay_rows, "maximum_absolute_error": max(row["maximum_absolute_error"] for row in replay_rows)},
)
register_artifact("fixed_influent_quadratic_replay", fixed_quadratic_replay_path)
stage_complete("fixed_influent_quadratic_replay", cases=len(replay_rows), operating_points=len(surface_test_u))


def state_from_record(record: Mapping[str, Any], prefix: str) -> np.ndarray:
    return np.asarray([record[f"{prefix}_{name}"] for name in STATE_COLUMNS], dtype=np.float64)


def composite_from_record(record: Mapping[str, Any], prefix: str) -> np.ndarray:
    return np.asarray([record[f"{prefix}_{name}"] for name in MEASURED_COLUMNS], dtype=np.float64)


def json_scalar(value: Any) -> Any:
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    return value


def run_local_mechanistic_neighborhood(
    evaluator: MechanisticCaseEvaluator,
    center: np.ndarray,
) -> pd.DataFrame:
    delta = np.asarray(ACTIVE["validation"]["surrogate_neighborhood_step"], dtype=np.float64)
    keys: set[tuple[str, str]] = set()
    points: list[np.ndarray] = []
    for dh in (-delta[0], 0.0, delta[0]):
        for da in (-delta[1], 0.0, delta[1]):
            point = np.clip(np.asarray(center) + np.asarray([dh, da]), OPERATING_LOWER, OPERATING_UPPER)
            key = point_key(point)
            if key not in keys:
                keys.add(key)
                points.append(point)
    for point in points:
        evaluator.evaluate(point, phase="surrogate_local_neighborhood")
    missing = [point for point in points if point_key(point) not in evaluator.cache]
    if missing:
        raise RuntimeError("The mechanistic evaluation budget could not accommodate the complete local neighborhood.")
    frame = evaluator.frame()
    neighborhood_keys = {point_key(point) for point in points}
    mask = [point_key(np.asarray([row.HRT, row.Aeration])) in neighborhood_keys for row in frame.itertuples()]
    return frame.loc[mask].copy().reset_index(drop=True)


def save_case_search(
    case_dir: Path,
    surrogate_evaluator: SurrogateCaseEvaluator,
    mechanistic_evaluator: MechanisticCaseEvaluator,
    surrogate_search: Mapping[str, Any],
    mechanistic_search: Mapping[str, Any],
    neighborhood: pd.DataFrame,
) -> dict[str, Path]:
    paths = {
        "surrogate_archive": atomic_parquet(case_dir / "surrogate_search.parquet", surrogate_evaluator.frame()),
        "mechanistic_archive": atomic_parquet(case_dir / "mechanistic_search.parquet", mechanistic_evaluator.frame()),
        "mechanistic_neighborhood": atomic_parquet(case_dir / "mechanistic_neighborhood.parquet", neighborhood),
        "surrogate_search_summary": atomic_json(
            case_dir / "surrogate_search_summary.json",
            {key: value for key, value in surrogate_search.items() if key != "best"},
        ),
        "mechanistic_search_summary": atomic_json(
            case_dir / "mechanistic_search_summary.json",
            {key: value for key, value in mechanistic_search.items() if key != "best"},
        ),
    }
    return paths


def run_validation_case(case_id: str, influent: np.ndarray, *, case_kind: str) -> dict[str, Any]:
    case_dir = DIRS["optimization"] / case_id
    case_dir.mkdir(parents=True, exist_ok=True)
    complete_path = case_dir / "case_complete.json"
    x = np.asarray(influent, dtype=np.float64).reshape(20)
    influent_sha256 = sha256_json(x.tolist())
    if bool(ACTIVE["run"]["resume"]) and complete_path.exists():
        existing = read_json(complete_path)
        if (
            existing.get("contract_sha256") != CONFIG_HASH
            or existing.get("case_id") != case_id
            or existing.get("case_kind") != case_kind
            or existing.get("influent_sha256") != influent_sha256
            or existing.get("production_model_fingerprint") != PRODUCTION_MODEL_FINGERPRINT
        ):
            raise RuntimeError(f"Completed case {case_id} belongs to another contract.")
        for artifact_name, metadata in existing.get("artifacts", {}).items():
            artifact_path = case_dir / str(metadata.get("path", ""))
            if not artifact_path.is_file() or sha256_file(artifact_path) != metadata.get("sha256"):
                raise RuntimeError(f"Completed case {case_id} has a missing or corrupt {artifact_name} artifact.")
        if set(existing.get("artifacts", {})) != {
            "surrogate_archive", "mechanistic_archive", "mechanistic_neighborhood",
            "surrogate_search_summary", "mechanistic_search_summary",
        }:
            raise RuntimeError(f"Completed case {case_id} has an incomplete artifact set.")
        for artifact_name, metadata in existing["artifacts"].items():
            register_artifact(f"case_{case_id}_{artifact_name}", case_dir / metadata["path"])
        register_artifact(f"case_{case_id}_complete", complete_path)
        return dict(existing["summary"])

    for j, name in enumerate(STATE_COLUMNS):
        lo, hi = ACTIVE["simulation"]["influent_state_ranges"][name]
        if not float(lo) <= x[j] <= float(hi):
            raise ValueError(f"{case_id}: {name} lies outside the trained range.")
    case_started = time.perf_counter()

    surrogate_settings = ACTIVE["optimization"]["surrogate_search"]
    surrogate_evaluator = SurrogateCaseEvaluator(
        PRODUCTION_MODEL, x, D_C_PRODUCTION, D_T_PRODUCTION,
        budget=int(surrogate_settings["total_verified_budget"]), case_id=case_id,
    )
    surrogate_search = run_bounded_search(surrogate_evaluator, surrogate_settings)
    surrogate_best = surrogate_search["best"]
    u_surrogate = np.asarray([surrogate_best["HRT"], surrogate_best["Aeration"]])
    predicted_state = state_from_record(surrogate_best, "predicted")
    predicted_composites = COMPOSITION @ predicted_state

    mechanistic_settings = ACTIVE["optimization"]["mechanistic_search"]
    mechanistic_evaluator = MechanisticCaseEvaluator(
        x, D_T_PRODUCTION,
        budget=int(mechanistic_settings["total_verified_budget"]), case_id=case_id,
    )
    # Reserve and complete the selected-point neighborhood before the bounded reference search.
    # These points consume the common budget, enter the candidate archive, and cannot be truncated
    # by a later DIRECT/grid phase.
    mechanistic_evaluator.evaluate(u_surrogate, phase="surrogate_selected_validation")
    neighborhood = run_local_mechanistic_neighborhood(mechanistic_evaluator, u_surrogate)
    mechanistic_search = run_bounded_search(mechanistic_evaluator, mechanistic_settings)
    mechanistic_selected = mechanistic_evaluator.cache[point_key(u_surrogate)]
    if not mechanistic_selected["accepted"]:
        raise RuntimeError(f"ASM2d-TSN rejected the surrogate-selected point for {case_id}.")
    mechanistic_best = mechanistic_evaluator.best_record()
    u_mechanistic = np.asarray([mechanistic_best["HRT"], mechanistic_best["Aeration"]])
    actual_state = state_from_record(mechanistic_selected, "mechanistic")
    actual_composites = COMPOSITION @ actual_state
    component_error = predicted_state - actual_state
    composite_error = predicted_composites - actual_composites
    regret = float(mechanistic_selected["objective"] - mechanistic_best["objective"])
    regret_tolerance = float(ACTIVE["validation"]["regret_tolerance"])
    if regret < -regret_tolerance:
        raise AssertionError(f"Negative decision regret for {case_id}: {regret:.3e}.")
    decision_displacement = float(np.linalg.norm((u_surrogate - u_mechanistic) / OPERATING_RANGE))

    baseline_u = np.asarray(ACTIVE["optimization"]["baseline"], dtype=np.float64)
    surrogate_evaluator.evaluate(baseline_u, phase="baseline_reporting")
    mechanistic_evaluator.evaluate(baseline_u, phase="baseline_reporting")
    surrogate_baseline = surrogate_evaluator.cache[point_key(baseline_u)]
    mechanistic_baseline = mechanistic_evaluator.cache[point_key(baseline_u)]
    if not mechanistic_baseline["accepted"]:
        raise RuntimeError(f"ASM2d-TSN rejected the declared baseline for {case_id}.")

    surrogate_boundary = boundary_flags(u_surrogate, surrogate_settings["boundary_tolerance"])
    mechanistic_boundary = boundary_flags(u_mechanistic, mechanistic_settings["boundary_tolerance"])
    summary: dict[str, Any] = {
        "case_id": case_id,
        "case_kind": case_kind,
        "surrogate_HRT": float(u_surrogate[0]),
        "surrogate_Aeration": float(u_surrogate[1]),
        "surrogate_dilution_rate_d-1": float(24.0 / u_surrogate[0]),
        "surrogate_kLa_d-1": float(
            ACTIVE["simulation"]["aeration_model"]["kla_base"]
            + ACTIVE["simulation"]["aeration_model"]["kla_per_aeration"] * u_surrogate[1]
        ),
        "surrogate_predicted_objective": float(surrogate_best["objective"]),
        "surrogate_ASM_objective": float(mechanistic_selected["objective"]),
        "objective_prediction_error": float(surrogate_best["objective"] - mechanistic_selected["objective"]),
        "quality_objective_predicted": float(surrogate_best["quality_objective"]),
        "operating_objective": float(surrogate_best["operating_objective"]),
        "HRT_penalty": float(surrogate_best["HRT_penalty"]),
        "Aeration_penalty": float(surrogate_best["Aeration_penalty"]),
        "mechanistic_reference_HRT": float(u_mechanistic[0]),
        "mechanistic_reference_Aeration": float(u_mechanistic[1]),
        "mechanistic_reference_objective": float(mechanistic_best["objective"]),
        "decision_regret": regret,
        "normalized_decision_displacement": decision_displacement,
        "component_standardized_prediction_error": float(
            np.sqrt(np.mean((component_error / D_C_PRODUCTION) ** 2))
        ),
        "composite_standardized_prediction_error": float(
            np.sqrt(np.mean((composite_error / D_T_PRODUCTION) ** 2))
        ),
        "surrogate_search_seconds": float(surrogate_search["search_seconds"]),
        "surrogate_verified_evaluations": int(surrogate_search["verified_evaluations"]),
        "surrogate_termination": str(surrogate_search["termination"]),
        "surrogate_resolution_qualified": bool(surrogate_search["resolution_qualified"]),
        "mechanistic_search_seconds": float(mechanistic_search["search_seconds"]),
        "mechanistic_verified_evaluations": int(mechanistic_evaluator.evaluations),
        "mechanistic_neighborhood_points": int(len(neighborhood)),
        "mechanistic_termination": str(mechanistic_search["termination"]),
        "mechanistic_resolution_qualified": bool(mechanistic_search["resolution_qualified"]),
        "baseline_surrogate_objective": float(surrogate_baseline["objective"]),
        "baseline_ASM_objective": float(mechanistic_baseline["objective"]),
        "selected_scaled_l2_displacement": float(surrogate_best["lower_scaled_l2_displacement"]),
        "selected_lower_qp_objective": float(surrogate_best["lower_qp_objective"]),
        "selected_invariant_linf": float(surrogate_best["lower_invariant_linf"]),
        "selected_minimum_component": float(surrogate_best["lower_minimum_component"]),
        "selected_stationarity_linf": float(surrogate_best["lower_stationarity_linf"]),
        "selected_complementarity_linf": float(surrogate_best["lower_complementarity_linf"]),
        "selected_active_component_count": int(surrogate_best["lower_active_component_count"]),
        "case_wall_seconds": float(time.perf_counter() - case_started),
        **{f"influent_{name}": float(x[j]) for j, name in enumerate(STATE_COLUMNS)},
        **flatten_composites("influent", COMPOSITION @ x),
        **flatten_state("predicted", predicted_state),
        **flatten_state("ASM_selected", actual_state),
        **{f"component_error_{name}": float(component_error[j]) for j, name in enumerate(STATE_COLUMNS)},
        **flatten_composites("predicted", predicted_composites),
        **flatten_composites("ASM_selected", actual_composites),
        **{f"composite_error_{name}": float(composite_error[j]) for j, name in enumerate(MEASURED_COLUMNS)},
        **{f"surrogate_{key}": value for key, value in surrogate_boundary.items()},
        **{f"mechanistic_{key}": value for key, value in mechanistic_boundary.items()},
    }
    summary = {key: json_scalar(value) for key, value in summary.items()}
    paths = save_case_search(
        case_dir, surrogate_evaluator, mechanistic_evaluator,
        surrogate_search, mechanistic_search, neighborhood,
    )
    complete_payload = {
        "contract_sha256": CONFIG_HASH,
        "case_id": case_id,
        "case_kind": case_kind,
        "influent_sha256": influent_sha256,
        "production_model_fingerprint": PRODUCTION_MODEL_FINGERPRINT,
        "completed_utc": utc_now(),
        "summary": summary,
        "artifacts": {name: {"path": path.name, "sha256": sha256_file(path)} for name, path in paths.items()},
    }
    atomic_json(complete_path, complete_payload)
    for artifact_name, artifact_path in paths.items():
        register_artifact(f"case_{case_id}_{artifact_name}", artifact_path)
    register_artifact(f"case_{case_id}_complete", complete_path)
    return summary

## Nominal case and mechanistic decision validation

In [ ]:
declared_nominal_composites = np.asarray([546.500, 47.795, 24.900, 325.480])
observed_nominal_composites = COMPOSITION @ NOMINAL_X
if not np.allclose(observed_nominal_composites, declared_nominal_composites, atol=5e-4, rtol=0.0):
    raise AssertionError("The nominal influent composites do not match the declared case study.")

NOMINAL_SUMMARY = run_validation_case("nominal", NOMINAL_X, case_kind="nominal")
nominal_summary_path = atomic_json(DIRS["validation"] / "nominal_summary.json", NOMINAL_SUMMARY)
register_artifact("nominal_summary", nominal_summary_path)
stage_complete(
    "nominal_case",
    HRT=NOMINAL_SUMMARY["surrogate_HRT"],
    Aeration=NOMINAL_SUMMARY["surrogate_Aeration"],
    predicted_objective=NOMINAL_SUMMARY["surrogate_predicted_objective"],
    ASM_objective=NOMINAL_SUMMARY["surrogate_ASM_objective"],
    regret=NOMINAL_SUMMARY["decision_regret"],
    surrogate_termination=NOMINAL_SUMMARY["surrogate_termination"],
)
display(pd.Series(NOMINAL_SUMMARY, name="value").to_frame().head(35))

## Independent 100-influent robustness study

The robustness influents are a second scrambled 20-dimensional Latin hypercube with seed 314159.
They are generated only after fitting, coefficient freezing, scale construction, and objective
declaration. Each case is checkpointed independently so the full article run can resume safely.

In [ ]:
robustness_count = int(ACTIVE["robustness"]["n_cases"])
robustness_seed = int(ACTIVE["robustness"]["seed"])
influent_lower = np.asarray(
    [ACTIVE["simulation"]["influent_state_ranges"][name][0] for name in STATE_COLUMNS], dtype=np.float64
)
influent_upper = np.asarray(
    [ACTIVE["simulation"]["influent_state_ranges"][name][1] for name in STATE_COLUMNS], dtype=np.float64
)
robustness_influents = qmc_scale(
    LatinHypercube(d=20, seed=robustness_seed).random(robustness_count),
    influent_lower, influent_upper,
)
robustness_input_frame = pd.DataFrame(robustness_influents, columns=STATE_COLUMNS)
robustness_input_frame.insert(0, "case_id", [f"robustness_{i:03d}" for i in range(robustness_count)])
robustness_input_path = atomic_csv(DIRS["validation"] / "robustness_influents.csv", robustness_input_frame)
register_artifact("robustness_influents", robustness_input_path)

robustness_summaries: list[dict[str, Any]] = []
robustness_checkpoint_path = DIRS["validation"] / "robustness_summary.partial.parquet"
for row in tqdm(robustness_input_frame.itertuples(index=False), total=robustness_count, desc="Robustness cases"):
    case_id = str(row.case_id)
    influent = np.asarray([getattr(row, name) for name in STATE_COLUMNS], dtype=np.float64)
    robustness_summaries.append(run_validation_case(case_id, influent, case_kind="robustness"))
    checkpoint = pd.DataFrame(robustness_summaries)
    atomic_parquet(robustness_checkpoint_path, checkpoint)

ROBUSTNESS_RESULTS = pd.DataFrame(robustness_summaries).sort_values("case_id").reset_index(drop=True)
if len(ROBUSTNESS_RESULTS) != robustness_count or ROBUSTNESS_RESULTS["case_id"].nunique() != robustness_count:
    raise AssertionError("The robustness result set is incomplete.")
expected_robustness_ids = [f"robustness_{index:03d}" for index in range(robustness_count)]
if ROBUSTNESS_RESULTS["case_id"].tolist() != expected_robustness_ids:
    raise AssertionError("The robustness case identifiers are missing or reordered.")
if set(ROBUSTNESS_RESULTS["case_kind"].astype(str)) != {"robustness"}:
    raise AssertionError("A non-robustness result entered the robustness cohort.")
numeric_robustness = ROBUSTNESS_RESULTS.select_dtypes(include=[np.number])
if numeric_robustness.empty or not np.all(np.isfinite(numeric_robustness.to_numpy(np.float64))):
    raise AssertionError("The robustness result set contains missing or non-finite numerical values.")
robustness_result_path = atomic_parquet(DIRS["validation"] / "robustness_results.parquet", ROBUSTNESS_RESULTS)
robustness_csv_path = atomic_csv(DIRS["validation"] / "robustness_results.csv", ROBUSTNESS_RESULTS)
for name, path in (("robustness_results", robustness_result_path), ("robustness_results_csv", robustness_csv_path)):
    register_artifact(name, path)
with contextlib.suppress(FileNotFoundError):
    robustness_checkpoint_path.unlink()
stage_complete(
    "robustness_cases",
    cases=robustness_count,
    mean_regret=float(ROBUSTNESS_RESULTS["decision_regret"].mean()),
    p95_regret=float(ROBUSTNESS_RESULTS["decision_regret"].quantile(0.95)),
    surrogate_resolution_qualified=int(ROBUSTNESS_RESULTS["surrogate_resolution_qualified"].sum()),
    mechanistic_resolution_qualified=int(ROBUSTNESS_RESULTS["mechanistic_resolution_qualified"].sum()),
)
display(
    ROBUSTNESS_RESULTS[
        ["case_id", "surrogate_HRT", "surrogate_Aeration", "surrogate_ASM_objective",
         "mechanistic_reference_objective", "decision_regret", "component_standardized_prediction_error"]
    ]
)

## Article source tables, figures, and complete computational report

Every displayed aggregate is written from row-level source data. Accuracy, coefficient, timing,
optimization, prediction-validation, decision-regret, boundary-activity, and solver diagnostics
remain available in physical units as well as normalized forms.

In [ ]:
def aggregate_numeric(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce").dropna()
        if len(values) != len(frame) or not np.all(np.isfinite(values.to_numpy(np.float64))):
            raise ValueError(f"Cannot aggregate incomplete or non-finite quantity {column!r}.")
        rows.append(
            {
                "quantity": column,
                "count": int(values.size),
                "mean": float(values.mean()),
                "standard_deviation": float(values.std(ddof=1)) if values.size > 1 else 0.0,
                "minimum": float(values.min()),
                "q25": float(values.quantile(0.25)),
                "median": float(values.median()),
                "q75": float(values.quantile(0.75)),
                "p95": float(values.quantile(0.95)),
                "maximum": float(values.max()),
            }
        )
    return pd.DataFrame(rows)


robustness_numeric_columns = [
    "surrogate_HRT", "surrogate_Aeration", "surrogate_predicted_objective",
    "surrogate_ASM_objective", "objective_prediction_error", "mechanistic_reference_HRT",
    "mechanistic_reference_Aeration", "mechanistic_reference_objective", "decision_regret",
    "normalized_decision_displacement", "component_standardized_prediction_error",
    "composite_standardized_prediction_error", "selected_scaled_l2_displacement",
    "selected_lower_qp_objective", "selected_invariant_linf", "selected_stationarity_linf",
    "selected_complementarity_linf", "surrogate_verified_evaluations",
    "mechanistic_verified_evaluations", "surrogate_search_seconds", "mechanistic_search_seconds",
    "case_wall_seconds", "mechanistic_neighborhood_points",
] + [f"component_error_{name}" for name in STATE_COLUMNS] + [
    f"composite_error_{name}" for name in MEASURED_COLUMNS
]
ROBUSTNESS_AGGREGATE = aggregate_numeric(ROBUSTNESS_RESULTS, robustness_numeric_columns)
bound_columns = [
    column for column in ROBUSTNESS_RESULTS.columns
    if column.endswith("_lower_active") or column.endswith("_upper_active")
]
BOUND_ACTIVITY = pd.DataFrame(
    {
        "bound": bound_columns,
        "active_count": [int(ROBUSTNESS_RESULTS[column].astype(bool).sum()) for column in bound_columns],
        "active_fraction": [float(ROBUSTNESS_RESULTS[column].astype(bool).mean()) for column in bound_columns],
    }
)

nominal_component_table = pd.DataFrame(
    {
        "component": STATE_COLUMNS,
        "unit": [ACTIVE["simulation"]["workbook"]["state_units"][name] for name in STATE_COLUMNS],
        "predicted": [NOMINAL_SUMMARY[f"predicted_{name}"] for name in STATE_COLUMNS],
        "ASM_selected": [NOMINAL_SUMMARY[f"ASM_selected_{name}"] for name in STATE_COLUMNS],
        "prediction_error": [NOMINAL_SUMMARY[f"component_error_{name}"] for name in STATE_COLUMNS],
        "standardized_error": [
            NOMINAL_SUMMARY[f"component_error_{name}"] / D_C_PRODUCTION[j]
            for j, name in enumerate(STATE_COLUMNS)
        ],
    }
)
nominal_composite_table = pd.DataFrame(
    {
        "composite": MEASURED_COLUMNS,
        "predicted": [NOMINAL_SUMMARY[f"predicted_{name}"] for name in MEASURED_COLUMNS],
        "ASM_selected": [NOMINAL_SUMMARY[f"ASM_selected_{name}"] for name in MEASURED_COLUMNS],
        "prediction_error": [NOMINAL_SUMMARY[f"composite_error_{name}"] for name in MEASURED_COLUMNS],
        "standardized_error": [
            NOMINAL_SUMMARY[f"composite_error_{name}"] / D_T_PRODUCTION[j]
            for j, name in enumerate(MEASURED_COLUMNS)
        ],
    }
)
nominal_operation_table = pd.DataFrame(
    [
        {
            "quantity": key,
            "value": NOMINAL_SUMMARY[key],
        }
        for key in (
            "surrogate_HRT", "surrogate_Aeration", "surrogate_dilution_rate_d-1",
            "surrogate_kLa_d-1", "surrogate_predicted_objective", "surrogate_ASM_objective",
            "mechanistic_reference_HRT", "mechanistic_reference_Aeration",
            "mechanistic_reference_objective", "decision_regret",
            "normalized_decision_displacement", "surrogate_search_seconds",
            "mechanistic_search_seconds", "surrogate_verified_evaluations",
            "mechanistic_verified_evaluations",
        )
    ]
)

target_row_space_diagnostics = []
row_projector = A_MATRIX.T @ A_MATRIX
for index, name in enumerate(MEASURED_COLUMNS):
    target_row = COMPOSITION[index]
    residual = target_row - target_row @ row_projector
    target_row_space_diagnostics.append(
        {
            "composite": name,
            "relative_distance_from_invariant_row_space": float(
                np.linalg.norm(residual) / np.linalg.norm(target_row)
            ),
        }
    )
TARGET_INVARIANT_RELATION = pd.DataFrame(target_row_space_diagnostics)

production_timing_indices = np.arange(min(int(ACTIVE["timing"]["batch_size"]), ID_TARGET))
production_timing = pd.concat(
    [
        pd.DataFrame(
            [{
                "operation": "production_training",
                "repetition": 0,
                "items": ID_TARGET,
                "seconds": PRODUCTION_MODEL["training_seconds"],
                "microseconds_per_item": PRODUCTION_MODEL["training_seconds"] * 1e6 / ID_TARGET,
            }]
        ),
        timed_call(
            "production_end_to_end_inference_batch",
            lambda: deploy_batch(
                PRODUCTION_MODEL, U_ALL[production_timing_indices], X_ALL[production_timing_indices],
                D_C_PRODUCTION,
            ),
            warmups=int(ACTIVE["timing"]["warmup_runs"]),
            repeats=int(ACTIVE["timing"]["measured_runs"]),
            items=len(production_timing_indices),
        ),
        timed_call(
            "production_end_to_end_inference_single",
            lambda: deploy_batch(PRODUCTION_MODEL, U_ALL[:1], X_ALL[:1], D_C_PRODUCTION),
            warmups=int(ACTIVE["timing"]["warmup_runs"]),
            repeats=int(ACTIVE["timing"]["single_prediction_repeats"]),
            items=1,
        ),
    ],
    ignore_index=True,
)
ALL_TIMING = pd.concat([assessment_timing, production_timing], ignore_index=True)
TIMING_SUMMARY = (
    ALL_TIMING.groupby("operation")
    .agg(
        repetitions=("repetition", "count"),
        items=("items", "max"),
        median_seconds=("seconds", "median"),
        q25_seconds=("seconds", lambda s: float(s.quantile(0.25))),
        q75_seconds=("seconds", lambda s: float(s.quantile(0.75))),
        median_microseconds_per_item=("microseconds_per_item", "median"),
    )
    .reset_index()
)

table_frames = {
    "dataset_validation": pd.DataFrame([dataset_validation]),
    "assessment_accuracy": assessment_metrics,
    "assessment_physical_diagnostics": assessment_physics,
    "production_scales": scale_frame,
    "coefficient_block_summary": coefficient_summary,
    "nominal_operation": nominal_operation_table,
    "nominal_components": nominal_component_table,
    "nominal_composites": nominal_composite_table,
    "robustness_aggregate": ROBUSTNESS_AGGREGATE,
    "robustness_bound_activity": BOUND_ACTIVITY,
    "target_invariant_relation": TARGET_INVARIANT_RELATION,
    "timing_summary": TIMING_SUMMARY,
}
for table_name, frame in table_frames.items():
    csv_path = atomic_csv(DIRS["tables"] / f"{table_name}.csv", frame)
    register_artifact(f"table_{table_name}", csv_path)
    if bool(ACTIVE["reporting"]["write_tex_tables"]):
        tex = frame.to_latex(index=False, float_format=lambda value: f"{value:.6g}", escape=True)
        tex_path = atomic_bytes(DIRS["tables"] / f"{table_name}.tex", tex.encode("utf-8"))
        register_artifact(f"table_{table_name}_tex", tex_path)

all_timing_path = atomic_csv(DIRS["timing"] / "all_timing_repetitions.csv", ALL_TIMING)
register_artifact("all_timing_repetitions", all_timing_path)


def save_figure(fig: mpl.figure.Figure, stem: str, source_data: pd.DataFrame | None = None) -> None:
    fig.tight_layout()
    for extension in ACTIVE["reporting"]["figure_formats"]:
        path = DIRS["figures"] / f"{stem}.{extension}"
        fd, temporary_name = tempfile.mkstemp(
            prefix=f".{stem}.", suffix=f".{extension}", dir=path.parent
        )
        os.close(fd)
        temporary_path = Path(temporary_name)
        try:
            fig.savefig(
                temporary_path,
                dpi=int(ACTIVE["reporting"]["figure_dpi"]),
                bbox_inches="tight",
            )
            replace_with_retry(temporary_path, path)
        finally:
            with contextlib.suppress(FileNotFoundError):
                temporary_path.unlink()
        register_artifact(f"figure_{stem}_{extension}", path)
    if source_data is not None:
        source_path = atomic_csv(DIRS["figures"] / f"{stem}.source.csv", source_data)
        register_artifact(f"figure_{stem}_source", source_path)
    plt.close(fig)


mpl.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.25})

training_plot_data = pd.concat(
    [
        ASSESSMENT_MODEL["history"].assign(fit="assessment"),
        PRODUCTION_MODEL["history"].assign(fit="production"),
    ],
    ignore_index=True,
)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for label, group in training_plot_data.groupby("fit"):
    ax.plot(group["sweep"], group["objective"], marker="o", ms=3, label=label)
ax.set_yscale("log"); ax.set_xlabel("Recursive sweep"); ax.set_ylabel("Training objective"); ax.legend()
save_figure(fig, "icsor_training_convergence", training_plot_data)

test_prediction_frame = assessment_predictions.query("partition == 'test'")
parity_source = test_prediction_frame[
    ["sample_id"] + [column for name in MEASURED_COLUMNS for column in (f"actual_{name}", f"deployed_{name}")]
].copy()
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.4))
for ax, name in zip(axes.ravel(), MEASURED_COLUMNS, strict=True):
    actual = parity_source[f"actual_{name}"]
    predicted = parity_source[f"deployed_{name}"]
    ax.scatter(actual, predicted, s=8, alpha=0.45)
    limits = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
    ax.plot(limits, limits, color="black", lw=1)
    ax.set_title(name); ax.set_xlabel("ASM2d-TSN"); ax.set_ylabel("ICSOR")
save_figure(fig, "heldout_composite_parity", parity_source)

nominal_archive = pd.read_parquet(DIRS["optimization"] / "nominal" / "surrogate_search.parquet")
grid_source = nominal_archive[nominal_archive["phase_visits"].str.contains("independent_grid", na=False)].copy()
fig, ax = plt.subplots(figsize=(6.3, 4.6))
contour = ax.tricontourf(grid_source["HRT"], grid_source["Aeration"], grid_source["objective"], levels=18)
ax.scatter([NOMINAL_SUMMARY["surrogate_HRT"]], [NOMINAL_SUMMARY["surrogate_Aeration"]],
           marker="*", s=130, color="red", edgecolor="white", label="Selected")
fig.colorbar(contour, ax=ax, label="Dimensionless objective")
ax.set_xlabel("HRT (h)"); ax.set_ylabel("Aeration setting"); ax.legend()
save_figure(fig, "nominal_objective_surface", grid_source[["HRT", "Aeration", "objective"]])

fig, ax = plt.subplots(figsize=(8.2, 4.5))
positions = np.arange(20)
width = 0.4
ax.bar(positions - width / 2, nominal_component_table["predicted"], width, label="ICSOR")
ax.bar(positions + width / 2, nominal_component_table["ASM_selected"], width, label="ASM2d-TSN")
ax.set_xticks(positions, STATE_COLUMNS, rotation=60, ha="right")
ax.set_ylabel("Component concentration (native units)"); ax.legend()
save_figure(fig, "nominal_component_comparison", nominal_component_table)

fig, ax = plt.subplots(figsize=(6.3, 4.6))
scatter = ax.scatter(
    ROBUSTNESS_RESULTS["surrogate_HRT"], ROBUSTNESS_RESULTS["surrogate_Aeration"],
    c=ROBUSTNESS_RESULTS["decision_regret"], cmap="viridis", s=35,
)
fig.colorbar(scatter, ax=ax, label="Decision regret")
ax.set_xlabel("Selected HRT (h)"); ax.set_ylabel("Selected aeration setting")
save_figure(
    fig, "robustness_decisions",
    ROBUSTNESS_RESULTS[["case_id", "surrogate_HRT", "surrogate_Aeration", "decision_regret"]],
)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4))
axes[0].hist(ROBUSTNESS_RESULTS["component_standardized_prediction_error"], bins="auto")
axes[0].set_xlabel("Standardized component prediction error"); axes[0].set_ylabel("Cases")
axes[1].hist(ROBUSTNESS_RESULTS["decision_regret"], bins="auto")
axes[1].set_xlabel("Decision regret"); axes[1].set_ylabel("Cases")
save_figure(
    fig, "robustness_error_and_regret",
    ROBUSTNESS_RESULTS[["case_id", "component_standardized_prediction_error", "decision_regret"]],
)

fig, ax = plt.subplots(figsize=(7.0, 4.0))
timing_plot = TIMING_SUMMARY.sort_values("median_seconds")
ax.barh(timing_plot["operation"], timing_plot["median_seconds"])
ax.set_xscale("log"); ax.set_xlabel("Median elapsed time (s)")
save_figure(fig, "runtime_summary", timing_plot)

stage_complete(
    "publication_reporting",
    tables=len(table_frames),
    figures=7,
    TP_relative_distance_from_invariant_row_space=float(
        TARGET_INVARIANT_RELATION.loc[
            TARGET_INVARIANT_RELATION["composite"] == "TP",
            "relative_distance_from_invariant_row_space",
        ].iloc[0]
    ),
)

## Terminal acceptance, artifact inventory, and immutable completion marker

In [ ]:
required_stages = {
    "workbook_and_matrices", "mechanistic_tests", "mechanistic_dataset", "assessment_split",
    "deployment_tests", "optimization_tests", "icsor_assessment", "production_icsor",
    "fixed_influent_quadratic_replay", "nominal_case", "robustness_cases",
    "publication_reporting",
}
missing_stages = required_stages - set(MANIFEST["stages"])
if missing_stages:
    raise AssertionError(f"Required stages are incomplete: {sorted(missing_stages)}")
unexpected_stages = set(MANIFEST["stages"]) - required_stages
if unexpected_stages:
    raise AssertionError(f"Unexpected pre-terminal stages are present: {sorted(unexpected_stages)}")
invalid_stage_status = [
    name for name in required_stages if MANIFEST["stages"].get(name, {}).get("status") != "complete"
]
if invalid_stage_status:
    raise AssertionError(f"Stages without complete status: {sorted(invalid_stage_status)}")
if len(ID_DATA) != ID_TARGET or len(ROBUSTNESS_RESULTS) != robustness_count:
    raise AssertionError("Terminal cardinality checks failed.")
if coefficient_frame.shape[0] != 20 * 467 or gamma_frame.shape[0] != 20 * 20:
    raise AssertionError("The frozen coefficient artifacts are incomplete.")
if assessment_test_physics.query("stage == 'deployed'")["maximum_invariant_linf"].iloc[0] > float(ACTIVE["deployment"]["invariant_tolerance"]):
    raise AssertionError("Held-out deployed predictions violate invariant tolerance.")
if assessment_test_physics.query("stage == 'deployed'")["minimum_component"].iloc[0] < -float(ACTIVE["deployment"]["nonnegativity_tolerance"]):
    raise AssertionError("Held-out deployed predictions violate non-negativity tolerance.")
if requested_profile == "full" and not ARTICLE_ELIGIBLE:
    raise AssertionError("The full profile must be article eligible.")
if requested_profile == "smoke" and ARTICLE_ELIGIBLE:
    raise AssertionError("Smoke output must never be article eligible.")

# Validate registered and stage-specific files before constructing the final inventory.
for artifact_name, metadata in MANIFEST["artifacts"].items():
    registered_path = RUN_ROOT / metadata["path"]
    if (
        not registered_path.is_file()
        or registered_path.stat().st_size != metadata["bytes"]
        or sha256_file(registered_path) != metadata["sha256"]
    ):
        raise AssertionError(f"Registered artifact validation failed for {artifact_name}.")
for fit_label, fitted_model in (("assessment", ASSESSMENT_MODEL), ("production", PRODUCTION_MODEL)):
    diagnostic_dir = DIRS["models"] / f"{fit_label}_training_qp"
    expected_sweeps = int(fitted_model["sweeps"])
    expected_gamma = {f"gamma_sweep_{sweep:03d}.parquet" for sweep in range(1, expected_sweeps + 1)}
    expected_chat = {f"chat_sweep_{sweep:03d}.parquet" for sweep in range(1, expected_sweeps + 1)}
    observed_gamma = {path.name for path in diagnostic_dir.glob("gamma_sweep_*.parquet")}
    observed_chat = {path.name for path in diagnostic_dir.glob("chat_sweep_*.parquet")}
    if observed_gamma != expected_gamma or observed_chat != expected_chat:
        raise AssertionError(f"The {fit_label} training-QP diagnostic filenames are incomplete or stale.")
    for sweep in range(1, expected_sweeps + 1):
        gamma_diagnostics = pd.read_parquet(diagnostic_dir / f"gamma_sweep_{sweep:03d}.parquet")
        chat_diagnostics = pd.read_parquet(diagnostic_dir / f"chat_sweep_{sweep:03d}.parquet")
        if len(gamma_diagnostics) != 20 or len(chat_diagnostics) != int(fitted_model["training_rows"]):
            raise AssertionError(f"The {fit_label} sweep {sweep} diagnostic cardinality is wrong.")
        for diagnostic_name, diagnostics in (("Gamma", gamma_diagnostics), ("C-hat", chat_diagnostics)):
            resolved_status = diagnostics["status"].astype(str).str.strip().str.lower()
            solved_status = resolved_status.isin(["solved", "solved inaccurate"])
            if not np.all(solved_status | diagnostics["fallback_used"].astype(bool)):
                raise AssertionError(
                    f"The {fit_label} {diagnostic_name} sweep {sweep} has a status without the "
                    "tested source fallback recorded."
                )
            if float(diagnostics["maximum_constraint_violation"].max()) > float(ICSOR_SETTINGS["constraint_tolerance"]):
                raise AssertionError(f"The {fit_label} {diagnostic_name} sweep {sweep} violates constraints.")
        if float(chat_diagnostics["minimum_component"].min()) < -float(ICSOR_SETTINGS["nonnegativity_tolerance"]):
            raise AssertionError(f"The {fit_label} C-hat sweep {sweep} violates non-negativity.")
required_case_files = {
    "case_complete.json", "surrogate_search.parquet", "mechanistic_search.parquet",
    "mechanistic_neighborhood.parquet", "surrogate_search_summary.json",
    "mechanistic_search_summary.json",
}
for case_id in ["nominal", *ROBUSTNESS_RESULTS["case_id"].tolist()]:
    case_dir = DIRS["optimization"] / case_id
    if {path.name for path in case_dir.iterdir() if path.is_file()} != required_case_files:
        raise AssertionError(f"The archived file set for {case_id} is incomplete or unexpected.")
unfinished_files = [
    path.relative_to(RUN_ROOT).as_posix()
    for path in RUN_ROOT.rglob("*")
    if path.is_file() and (".partial." in path.name or path.name.endswith(".tmp"))
]
if unfinished_files:
    raise AssertionError(f"Unfinished checkpoint files remain: {unfinished_files}")

inventory_rows = []
for path in sorted(item for item in RUN_ROOT.rglob("*") if item.is_file()):
    if path.name in {"manifest.json", "artifact_inventory.csv", "manifest.sha256", "COMPLETED.json"}:
        continue
    inventory_rows.append(
        {
            "path": path.relative_to(RUN_ROOT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )
ARTIFACT_INVENTORY = pd.DataFrame(inventory_rows)
inventory_path = atomic_csv(RUN_ROOT / "artifact_inventory.csv", ARTIFACT_INVENTORY)
for row in ARTIFACT_INVENTORY.itertuples(index=False):
    path = RUN_ROOT / row.path
    if path.stat().st_size != row.bytes or sha256_file(path) != row.sha256:
        raise AssertionError(f"Artifact inventory verification failed for {row.path}.")

MANIFEST["stages"]["terminal_acceptance"] = {
    "status": "complete",
    "completed_utc": utc_now(),
    "artifact_count": int(len(ARTIFACT_INVENTORY)),
    "inventory_sha256": sha256_file(inventory_path),
}
MANIFEST["artifact_inventory"] = {
    "path": inventory_path.relative_to(RUN_ROOT).as_posix(),
    "bytes": inventory_path.stat().st_size,
    "sha256": sha256_file(inventory_path),
}
MANIFEST["status"] = "complete"
MANIFEST["completed_utc"] = utc_now()
save_manifest()
manifest_sha256 = sha256_file(manifest_path)
manifest_digest_path = atomic_bytes(
    RUN_ROOT / "manifest.sha256", f"{manifest_sha256}  manifest.json\n".encode("ascii")
)
completion_path = atomic_json(
    RUN_ROOT / "COMPLETED.json",
    {
        "run_id": RUN_ID,
        "profile": requested_profile,
        "article_eligible": ARTICLE_ELIGIBLE,
        "contract_sha256": sha256_json(RUN_CONTRACT),
        "manifest_sha256": manifest_sha256,
        "artifact_inventory_sha256": sha256_file(inventory_path),
        "artifact_count": int(len(ARTIFACT_INVENTORY)),
        "completed_utc": utc_now(),
    },
)
if sha256_file(manifest_path) != manifest_sha256 or not manifest_digest_path.is_file() or not completion_path.is_file():
    raise AssertionError("Final manifest sealing failed.")

print(f"Completed {requested_profile!r} run {RUN_ID!r}.")
print(f"Article eligible: {ARTICLE_ELIGIBLE}")
print(f"Artifacts inventoried: {len(ARTIFACT_INVENTORY)}")
display(TIMING_SUMMARY)
display(ROBUSTNESS_AGGREGATE.head(20))